# SOLA 4237 Assignment 2 - Household Battery Energy Storage System (BESS) Optimisation

### Site: `23W0158` - `NSW Thredbo, Alpine` - `2022` / `2023`

### SOLA4237 Consumer Energy Resources (CER) Operations and Integration

<div style="font-size: 1rem; font-weight: normal; line-height: 1.4;">
Student: <b>Bilal Parkar</b> &nbsp;|&nbsp; zID: <b>z5476566</b><br>
Prepared for: VPP-SPREE single-site optimisation division (retailer client review)
</div>

---

### Purpose and Context
This notebook holds **all analysis and optimisation code** behind the Assignment 2 report. The report must show that VPP-SPREE's single-site optimisation can deliver real, quantified benefit to a household across **flat, time-of-use (ToU) and wholesale pass-through tariffs**, and that it still works when **PV and price forecasts are imperfect**.

| # | Question the report must answer | Evidence built in |
|---|---|---|
| 1 | What are the site, data quality and tariff context? | Parts 1-4 |
| 2 | What battery (size, power, efficiency, limits, degradation) suits this site? | Parts 4, 5, 8 |
| 3 | What is the benchmark (no battery, simple self-consumption)? | Part 6 |
| 4 | What is the best possible operation on each tariff (perfect foresight)? | Part 7 |
| 5 | How good are our PV and price forecasts? | Parts 9, 10 |
| 6 | How much value is lost to uncertainty, and is PV or price the bigger driver? | Parts 11, 12 |
| 7 | Annual bill, self-consumption, savings, payback, VPP fee vs value | Part 14 |

### How this notebook is organised
- **Part 0** setup and constants. **Parts 1-3** load, check and prepare data. **Part 4** characterises the site. **Part 5** is the battery model. **Part 6** benchmarks. **Part 7** perfect-foresight optimiser. **Part 8** sizing and payback. **Parts 9-10** forecasts. **Part 11** 48-hour rolling-horizon scheduler with 5-minute execution. **Part 12** perfect vs uncertain comparison. **Part 13** sensitivity and robustness. **Part 14** techno-economics. **Part 15** discussion and recommendations. **Part 16** report figures. **Appendix A** AI disclosure and verification log. **Appendix B** reproducibility.
- Each task follows one pattern: *markdown explanation -> function -> test -> apply to site data -> findings cell*.
- Functions are tested on tiny synthetic samples (`assert`) before they touch site data.
- **Findings cells** (blockquotes) record what the data showed. Anything marked `TODO` is not finished.
- Set `SOLA_DATA_PATH` (or edit `DATA_PATH` in Part 0) and run top to bottom. Set `QUICK_RUN_DAYS` in Part 0 for a fast dry run of the slow parts (11-12).

### Required files
- Assignment CSV (two years of 5-minute data). Column names are mapped in one place (`COL` in Part 0): **check these against your CSV first.**
- AEMO pre-dispatch price forecasts for the site's region (only if you choose them over your own price model; see Part 10).

### Contents and status

| Part | Title | Status | Note |
|---|---|---|---|
| 0 | Setup and configuration | Done | Site metadata, paths, wholesale tariff settings (sourced) |
| 1 | Data loading and integrity | Done | Findings 1.2-1.6 and Part 1 summary written |
| 2 | Data cleaning | Done | Functions written by me; all tests and contract checks pass; Figure 2.1 |
| 3 | Tariffs and prices | Code done | Wholesale price built and checked; Finding 3.2-3.5 to write; network windows pending tutor (one setting) |
| 4 | Site characterisation | Code done | Clock AEST confirmed; shiftable median 7.4 kWh/day; PV ~8.9 kWp; AC-coupled; Findings 4.1-4.4 to write |
| 5 | Battery model | Code done | Sigenergy 2 x 8 kWh; wear ~$0.39/kWh; (A) assumptions to confirm; Finding 5 to write |
| 6 | Benchmarks and billing | Code done | No battery $822 / $606 / $649; self-consumption saves $513-666/yr; Finding 6 to write |
| 7 | Perfect-foresight optimiser | Code done | LP checked by `check_schedule`; switch reasons and Finding 7 to write |
| 8 | Sizing and payback | Code done | Sweep cached; 1 module fastest payback; size decision and Finding 8 to write |
| 9 | PV forecasting | Code done | b/a = datasheet gamma; ~1% nRMSE vs commercial, ~5% vs measured; Finding 9 to write |
| 10 | Price forecasting | Code done | 7-day profile median chosen (MAE ~$41/MWh); Finding 10 to write |
| 11 | Rolling-horizon scheduler | Code done | Runs cached; wholesale saves ~$28/yr before the $300 fee; Finding 11 to write |
| 12 | Perfect vs uncertain; PV vs price | Code done | Price forecast loses ~$393 of $421; PV ~$0; Finding 12 to write |
| 13 | Sensitivity and robustness | 13.1-13.3 code done; 13.4 skeleton | 13.1 wear sweep; 13.2 top 1% of intervals earn ~84% of the optimiser saving; 13.3 2022: PV loss still ~$0 with measured PV; 13.4 **my 4 functions to write**; Findings 13.1-13.4 to write |
| 14 | Techno-economics and metrics | Code done (14.1, 14.2, 14.4, 14.5) | No mode has positive NPV at $15,600; 14.3 tornado to build; Finding 14 to write |
| 14 | Discussion and recommendations | Skeleton | Fill after results |
| 15 | Report figures | Skeleton | Planned figure list |
| A | AI disclosure and verification log | In progress | Running log in `AI_LOG.md`; disclosure to write from it |
| B | Reproducibility | Skeleton | Versions, seed, runtime |

#### Brief requirements tracker

| # | Requirement from the brief | Where | Status |
|---|---|---|---|
| 1 | Default self-consumption mode as benchmark | Part 6 | **Done** (Task 6.3) |
| 2 | AC vs DC bus assumption, losses | Parts 4, 5 | **Done**: AC-coupled retrofit (Task 4.4); one-way efficiency = sqrt(round trip) (Part 5) |
| 3 | BESS size, power, SOC limits, temperature limits, degradation cost | Parts 4, 5, 8 | **Done** (Sigenergy spec, cold derating, wear cost, sizing sweep); (A) assumptions to confirm |
| 4 | Optimal strategy for flat, ToU, wholesale | Part 7 | **Done** (Task 7.3, Figure 7.1) |
| 5 | Wholesale retail price (network, fees, certificates, hedging) | Part 3 | **Done** (Tasks 3.2-3.3); network windows pending tutor advice, one setting in Part 0 |
| 6 | $25/month VPP fee on wholesale pass-through | Parts 0, 7, 11 | **Applied** whenever VPP-SPREE optimises (Parts 7, 11); not on the benchmarks |
| 7 | Zero export (curtail) when prices are negative | Part 7 | **Built in** (curtailment variable; chosen only at negative export prices, Task 7.1 test 4) |
| 8 | PV forecast model trained on Year 1, compared with commercial forecast (MAE, RMSE, nRMSE) | Part 9 | **Done** (Tasks 9.1-9.2, Figure 9.1) |
| 9 | Price forecast (own model or AEMO pre-dispatch) | Part 10 | **Done** (own model, no look-ahead; Tasks 10.1-10.2) |
| 10 | 48 h rolling horizon, hourly replanning, 5-minute execution without knowing current-interval price/solar | Part 11 | **Built** (Task 11) |
| 11 | Settle on actual price, BESS flows, load and solar | Part 11 | **Built** (`settle`, Task 11.1) |
| 12 | Uncertainty vs perfect foresight; PV vs price influence | Part 12 | **Done** (ablation A-D, Figure 12.1) |
| 13 | Annual bill, self-consumption, savings, payback | Parts 6-8, 11, 13 | **Done** (master table, Part 14) |
| 14 | AI disclosure with evidence of checking AI output | Appendix A | In progress: running log in `AI_LOG.md` |
| 15 | Missing and spurious data identified and cleaned, with justification | Parts 1, 2 | **Done** (Tasks 1.2-1.6, 2.1-2.4, Figure 2.1) |
| 16 | Time handling: timestamps, alignment, power vs energy | Parts 1, 3, 4 | **Done** for prices (interval-end stamps, Task 3.1); AEST clock check in Task 4.1 |
| 17 | Site load and solar analysis | Parts 1, 4 | **Done** (Task 4.2, Figure 4.1) |
| 18 | Code shared via GitHub link | Appendix B | TODO: repo not yet on GitHub |


# Part 0 - Setup and Configuration

### Conventions used throughout (verify in Part 2)
- **Battery power** `batt_kW`: **positive = discharging**, **negative = charging** (same convention as Assignment 1).
- **Grid power** `grid_kW`: **positive = import**, **negative = export**.
- **Accounting identity:** `grid = load - (solar - curtailed) - battery`.
- **Power to energy:** `kWh = kW x dt_h` (`dt_h = 5/60` for raw data, `1` for hourly plans).
- **Prices** are in **$/kWh** inside the code. Wholesale data arrives in $/MWh, so divide by 1000.
- **Time:** timestamps are naive local time. State whether they are NEM time (fixed AEST, UTC+10) in Part 1 and align prices accordingly.

In [292]:
import os, io, sys, json, hashlib, tempfile, platform, datetime, warnings, dataclasses
from pathlib import Path
import requests
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
from scipy.optimize import linprog, brentq
import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
RNG_SEED = 42
np.random.seed(RNG_SEED)

In [293]:
# ---------- Paths ----------
DATA_PATH = os.environ.get(
    "SOLA_DATA_PATH",
    "/Users/bilalp/Documents/SOLA4237/CER_Assignments/Assignment_2/SOLA/Data_NSW_Thredbo - Alpine_2022-2023_partial.csv",
)
PRICE_DIR = Path(DATA_PATH).parent / "data"     # AEMO price cache (Part 3), created on first run
AEMO_PRICE_URL = "https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_{ym}_{region}.csv"
# ---------- Column map: the ONLY place raw CSV names appear ----------
COL = dict(
    time     = "timestamp",
    load     = "load_kW",
    solar    = "solar_output_kW",                        # measured PV (expected Year 1 only)
    solar_fc = "commercial_forecasted_solar_output_kW",  # commercial forecast (expected Year 2 only)
    irr      = "irradiance_W/m^2",
    temp     = "temperature_C",
    rrp      = "spot_price_AUD/kWh",                     # already $/kWh: do NOT divide by 1000
    flat_imp = "tariff_import_flat_AUD/kWh",
    flat_exp = "tariff_export_flat_AUD/kWh",
    tou_imp  = "tariff_import_tou_AUD/kWh",
    tou_exp  = "tariff_export_tou_AUD/kWh",
)

# ---------- Site metadata (from CER-data-generator screenshot) ----------
SITE_ID      = "NSW_Thredbo"
LOCATION     = "Thredbo, NSW (Alpine)"
NEM_REGION   = "NSW1"
LOAD_PROFILE = "23W0158 (scaled x29.1)"
PV_PR        = 0.85      # performance ratio
PV_GAMMA     = -0.004    # power temperature coefficient, per degC

# ---------- Time ----------
INTERVAL_MIN   = 5
DT_H           = INTERVAL_MIN / 60      # hours per interval: kWh = kW * DT_H
STEPS_PER_HOUR = 60 // INTERVAL_MIN

In [294]:
# ---------- Wholesale pass-through tariff (Part 3) ----------
# Network time windows: "assignment" = the site's own ToU windows (Finding 1.4, every day);
# "essential" = Essential Energy's network windows (weekdays only). Awaiting tutor advice: change here only.
NETWORK_WINDOWS = "assignment"
NETWORK_RATES = {   # Essential Energy BLNT3AL (LV Residential ToU, interval meter), c/kWh excl. GST
    "2021-22": dict(peak=14.5119, shoulder=11.2550, offpeak=4.4684),   # (A) indicative NUoS schedule 2020-24 (Apr 2020), $2020-21; Part 13.3 only
    "2022-23": dict(peak=16.1216, shoulder=12.0800, offpeak=4.8957),   # indicative NUoS schedule, Mar 2022
    "2023-24": dict(peak=17.6443, shoulder=12.6958, offpeak=5.0549),   # network price list, from 1 Jul 2023
}
LOSS_FACTOR = 1.034   # DLF x MLF, Essential NSLP (ACIL Allen for AER, DMO 2023-24, Table 4.33)
ENV_C_KWH   = 1.781   # LRET 7.27 + SRES 6.86 + NSW ESS 3.26 + PDRS 0.42 $/MWh (Tables 4.6-4.10)
FEES_C_KWH  = 0.157   # NEM management 1.15 + ancillary services 0.42 $/MWh (Tables 4.11-4.12)
HEDGE_C_KWH = 0.263   # prudential: AEMO 0.77 + hedge 1.86 $/MWh (Tables 4.16, 4.24)
GST = 0.10            # applied to wholesale imports so they compare with the GST-inclusive flat/ToU rates
VPP_FEE_PER_MONTH = 25.0   # $/month, wholesale pass-through only (applied in Part 13)

# Part 1 - Data Loading and Integrity

### Background: why check the timeline before anything else?
Every later calculation assumes one row per 5-minute interval. A missing or duplicated interval silently biases energy totals, and a clock shift (daylight saving) misaligns solar noon and price merges. "No nulls" is not the same as "complete". This dataset also mixes **two years with different columns populated** (measured solar in Year 1, commercial forecast in Year 2), so nulls in those columns are expected, not faults.

## Task 1.1 - Load the telemetry

In [295]:
def load_telemetry(path) -> pd.DataFrame:
    out = pd.read_csv(path, parse_dates=["timestamp"])
    return out.set_index("timestamp").sort_index()

In [296]:
"""Tests for Task 1.1"""
_csv = io.StringIO("timestamp,x\n2024-01-01 00:05:00,2\n2024-01-01 00:00:00,1\n")
_t = load_telemetry(_csv)
assert isinstance(_t.index, pd.DatetimeIndex)   # real datetimes, not strings
assert _t.index.is_monotonic_increasing          # sorted
assert _t["x"].tolist() == [1, 2]                # values travelled with their timestamps
print("Task 1.1 tests passed")

Task 1.1 tests passed


In [297]:
df = load_telemetry(DATA_PATH)
print(df.shape)
df.head()

(210240, 10)


,irradiance_W/m^2,temperature_C,solar_output_kW,commercial_forecasted_solar_output_kW,load_kW,spot_price_AUD/kWh,tariff_import_flat_AUD/kWh,tariff_export_flat_AUD/kWh,tariff_import_tou_AUD/kWh,tariff_export_tou_AUD/kWh
timestamp,,,,,,,,,,
2022-01-01 00:00:00,0.0,15.1,0.0,NaN,0.53,0.12477,0.3,0.05,0.15,0.05
2022-01-01 00:05:00,0.0,15.1,0.0,NaN,0.54,0.12486,0.3,0.05,0.15,0.05
2022-01-01 00:10:00,0.0,15.1,0.0,NaN,0.53,0.12602,0.3,0.05,0.15,0.05
2022-01-01 00:15:00,0.0,15.0,0.0,NaN,0.51,0.12493,0.3,0.05,0.15,0.05
2022-01-01 00:20:00,0.0,15.0,0.0,NaN,0.52,0.11354,0.3,0.05,0.15,0.05


## Task 1.2 - Timeline integrity and the two-year split
`check_timeline(df)` reports rows, expected rows, missing timestamps, duplicates. `split_years(df)` cuts the data at exactly one year after the first timestamp, so a financial-year dataset also splits correctly.

In [298]:
def check_timeline(df : pd.DataFrame, freq: str = "5min") -> dict:
    
    """Check the index is a complete, duplicate-free, regular time series.

    Compares the actual timestamps against an ideal grid from first to last
    timestamp at `freq`. Note: a complete timeline can still hold empty rows,
    so check nulls separately.
    """
    
    full = pd.date_range(df.index.min(), df.index.max(), freq = freq)

    return {
        "rows": len(df),
        "expected": len(full),
        "missing": len(full.difference(df.index)),
        "duplicates": int(df.index.duplicated().sum()),
        "monotonic": bool(df.index.is_monotonic_increasing),
        "start": df.index.min(),
        "end": df.index.max()
    }

In [299]:
"""Tests for Task 1.2a"""
# one missing interval (00:10)
_idx = pd.to_datetime(["2024-01-01 00:00", "2024-01-01 00:05", "2024-01-01 00:15"])
_r = check_timeline(pd.DataFrame({"a": [1, 2, 3]}, index=_idx))
assert _r["rows"] == 3 and _r["expected"] == 4 and _r["missing"] == 1 and _r["duplicates"] == 0

# the cancelling case: one gap AND one duplicate -> row count looks perfect but data isn't
_idx = pd.to_datetime(["2024-01-01 00:00", "2024-01-01 00:05", "2024-01-01 00:05", "2024-01-01 00:15"])
_r = check_timeline(pd.DataFrame({"a": [1, 2, 3, 4]}, index=_idx))
assert _r["rows"] == _r["expected"] == 4          # counts match...
assert _r["missing"] == 1 and _r["duplicates"] == 1   # ...but it's still broken
print("Task 1.2a tests passed")

Task 1.2a tests passed


In [300]:
def split_years(df:pd.DataFrame):

    """Split into (year1, year2) at exactly one year after the first timestamp.

    Uses DateOffset rather than a fixed 365 days so leap years split correctly.
    """
    
    cut = df.index.min() + pd.DateOffset(years=1)
    return df[df.index < cut], df[df.index >= cut]

In [301]:
"""Tests for Task 1.2b"""
_idx2 = pd.date_range("2023-01-01", "2024-12-31 23:55", freq="5min")
_y1, _y2 = split_years(pd.DataFrame({"a": 1}, index=_idx2))
assert _y1.index.max() < _y2.index.min()                      # no overlap
assert len(_y1) + len(_y2) == len(_idx2)                       # nothing lost
assert len(_y1) == 365 * 288 and len(_y2) == 366 * 288
print("Task 1.2b tests passed")

Task 1.2b tests passed


In [302]:
timeline = check_timeline(df)
print(pd.Series(timeline))
df1, df2 = split_years(df)
print("Year 1 rows:", len(df1), "| Year 2 rows:", len(df2))
pd.concat({"year1_nulls": df1.isna().sum(), "year2_nulls": df2.isna().sum()}, axis=1)

rows                       210240
expected                   210240
missing                         0
duplicates                      0
monotonic                    True
start         2022-01-01 00:00:00
end           2023-12-31 23:55:00
dtype: object
Year 1 rows: 105120 | Year 2 rows: 105120


,year1_nulls,year2_nulls
irradiance_W/m^2,184,866
temperature_C,184,866
solar_output_kW,184,105120
commercial_forecasted_solar_output_kW,105120,0
load_kW,184,866
spot_price_AUD/kWh,0,105120
tariff_import_flat_AUD/kWh,0,0
tariff_export_flat_AUD/kWh,0,0
tariff_import_tou_AUD/kWh,0,0
tariff_export_tou_AUD/kWh,0,0


In [303]:
sensor_cols = [COL["irr"], COL["temp"], COL["load"]]

# Are the sensor gaps the same rows? (True = yes)
blank = df[sensor_cols].isna()
print("Same rows across sensor columns:", (blank.all(axis=1) == blank.any(axis=1)).all())

# Group consecutive blank rows into gaps: start, end, length
is_gap = blank.any(axis=1)
gap_id = (is_gap != is_gap.shift()).cumsum()
gaps = (df[is_gap].groupby(gap_id[is_gap])
        .apply(lambda g: pd.Series({"start": g.index.min(), "end": g.index.max(), "intervals": len(g)})))
gaps["hours"] = gaps["intervals"] * DT_H
print(gaps.sort_values("intervals", ascending=False).head(15))
print("Number of gaps:", len(gaps))


Same rows across sensor columns: True
                 start                 end  intervals      hours
32 2023-07-01 22:20:00 2023-07-02 22:15:00        288  24.000000
12 2022-12-27 14:10:00 2022-12-27 19:50:00         69   5.750000
36 2023-08-12 07:10:00 2023-08-12 12:40:00         67   5.583333
30 2023-06-23 06:45:00 2023-06-23 12:15:00         67   5.583333
18 2023-03-15 13:50:00 2023-03-15 18:40:00         59   4.916667
42 2023-12-17 22:45:00 2023-12-18 03:25:00         57   4.750000
34 2023-07-23 22:15:00 2023-07-24 02:40:00         54   4.500000
28 2023-06-17 06:45:00 2023-06-17 10:45:00         49   4.083333
2  2022-03-13 10:50:00 2022-03-13 14:45:00         48   4.000000
38 2023-10-10 04:05:00 2023-10-10 08:00:00         48   4.000000
24 2023-04-29 07:10:00 2023-04-29 11:05:00         48   4.000000
22 2023-03-27 05:15:00 2023-03-27 07:10:00         24   2.000000
10 2022-09-13 15:55:00 2022-09-13 17:40:00         22   1.833333
44 2023-12-29 06:20:00 2023-12-29 08:05:00         2

> **Finding 1.2 - Completeness.** The timeline is complete: 210,240 of 210,240 expected 5-minute intervals, with no missing timestamps and no duplicates. Because there are no gaps or repeats around October and April, the timestamps are on a fixed offset with no daylight-saving jumps, consistent with NEM time (AEST); this is checked against solar noon in Part 4.
>
> Some columns are empty by design. Measured PV exists only in Year 1 and the commercial forecast only in Year 2, so the two are exact complements. There are no Year 2 spot prices, so actual 2023 NSW1 prices must be sourced from AEMO to settle the Year 2 wholesale bill.
>
> A complete timeline is not complete data. 22 data outages leave whole rows blank: the same rows in irradiance, temperature, load and PV (184 intervals in Year 1, 866 in Year 2, about 0.5% of the data). They range up to 24 hours (1–2 July 2023), and several cover the morning solar ramp. Handling is deferred to Part 2.

## Task 1.3 - Plausibility of each column

### Background: present is not the same as possible
Task 1.2 showed the data exists; this task checks it is physically possible. Each column has a hard physical bound: irradiance and load cannot be negative, PV is zero at night and capped by system size, and NEM spot prices sit between the market floor (-$1,000/MWh) and the market price cap. A value outside its bound means a unit or sensor problem that every later calculation would inherit. Year 1 and Year 2 are summarised separately so measured PV (Year 1) can be compared like-for-like with the commercial forecast (Year 2).

In [304]:
# Ranges per year (Year 2 PV is the forecast, so compare like with like)
desc = pd.concat({"Y1": df1.describe().T, "Y2": df2.describe().T})
desc[["min", "50%", "max", "mean"]].round(3)

min     50%      max     mean
Y1 irradiance_W/m^2                        0.00  13.000  1137.00  168.650
   temperature_C                          -8.20   6.400    24.30    6.816
   solar_output_kW                         0.00   0.100    10.31    1.263
   commercial_forecasted_solar_output_kW    NaN     NaN      NaN      NaN
   load_kW                                 0.17   0.570     5.87    1.212
   spot_price_AUD/kWh                     -1.00   0.134    15.50    0.183
   tariff_import_flat_AUD/kWh              0.30   0.300     0.30    0.300
   tariff_export_flat_AUD/kWh              0.05   0.050     0.05    0.050
   tariff_import_tou_AUD/kWh               0.15   0.250     0.40    0.242
   tariff_export_tou_AUD/kWh               0.05   0.050     0.05    0.050
Y2 irradiance_W/m^2                        0.00  16.000  1104.00  191.172
   temperature_C                         -10.10   7.400    26.20    7.805
   solar_output_kW                          NaN     NaN      NaN      NaN
   commercial_forecasted_solar_output_kW   0.00   0.120     8.25    1.458
   load_kW                                 0.17   0.570     5.87    1.215
   spot_price_AUD/kWh                       NaN     NaN      NaN      NaN
   tariff_import_flat_AUD/kWh              0.30   0.300     0.30    0.300
   tariff_export_flat_AUD/kWh              0.05   0.050     0.05    0.050
   tariff_import_tou_AUD/kWh               0.15   0.250     0.40    0.242
   tariff_export_tou_AUD/kWh               0.05   0.050     0.05    0.050

> **Finding 1.3 - Ranges.** Every column is physically plausible. Irradiance ranges from 0 to 1,137 W/m² (Year 1) and 0 to 1,104 W/m² (Year 2), consistent with clear-sky noon at about 1,000 W/m² plus brief cloud-edge enhancement and the site's high altitude. Temperature runs from −10.1 °C to 26.2 °C with a mean of about 7 °C, as expected for an alpine site, so cold-weather battery derating will matter (Part 5). Load is always positive (0.17–5.87 kW, mean 1.21 kW, about 29 kWh/day), so there is never zero standby load.
>
> The spot price reaches exactly −1.00 and 15.50 AUD/kWh, the NEM market floor (−1,000 AUD/MWh) and cap (15,500 AUD/MWh). This confirms the column is already in AUD/kWh and must not be divided by 1,000. Its mean of 183 AUD/MWh reflects the 2022 energy crisis.
>
> Measured PV peaks at 10.3 kW near 1,100 W/m², implying a system of roughly 10 kW (refined in Part 4). The Year 2 commercial forecast has a higher mean than Year 1 measured PV (1.46 vs 1.26 kW), in line with Year 2's sunnier irradiance, but a lower peak (8.25 vs 10.31 kW). Year 2 irradiance peaked only 3% lower than Year 1, so weather does not explain a 20% lower PV peak; this suggests the forecast is smoothed and misses short peaks, which is examined in Part 9.

## Task 1.4 - Tariff structure

### Background: read the tariff from the data, not from assumptions
The battery's value on a ToU tariff depends entirely on *when* each rate applies, so the windows must come from the data. `tou_schedule` builds an hour-by-hour table of the minimum and maximum rate, split by weekday and weekend. Taking min and max over the whole dataset tests three things at once: if they are equal for every hour, the boundaries fall on whole hours, the schedule does not change with season, and it applies identically to the days in each column.

In [305]:
def tou_schedule(rate: pd.Series) -> pd.DataFrame:
    """Hour-of-day x weekday/weekend table of the min and max rate in each hour.

    min == max in every cell means whole-hour boundaries and no seasonal change.
    min != max flags a mid-hour boundary or a rate that varies across the year.
    """
    t = pd.DataFrame({"hour": rate.index.hour,
                      "weekend": rate.index.dayofweek >= 5,
                      "rate": rate.values})
    return t.pivot_table(index="hour", columns="weekend", values="rate", aggfunc=["min", "max"])

In [306]:
"""Tests for Task 1.4"""
# Saturday 6 Jan and Monday 8 Jan 2024; peak 19:00-23:00, otherwise off-peak
_i = pd.date_range("2024-01-06", periods=288, freq="5min").append(pd.date_range("2024-01-08", periods=288, freq="5min"))
_r = pd.Series(np.where((_i.hour >= 19) & (_i.hour < 23), 0.40, 0.15), index=_i)
_s = tou_schedule(_r)
assert _s[("min", False)].loc[19] == 0.40 and _s[("min", True)].loc[19] == 0.40     # peak on both day types
assert _s[("max", False)].loc[3] == 0.15                                             # off-peak overnight
assert (_s["min"] == _s["max"]).all().all()                                          # whole-hour windows

# A boundary at 16:30 must show up as min != max in hour 16
_r2 = pd.Series(np.where(_i.hour + _i.minute / 60 >= 16.5, 0.40, 0.15), index=_i)
_s2 = tou_schedule(_r2)
assert _s2[("min", False)].loc[16] == 0.15 and _s2[("max", False)].loc[16] == 0.40
print("Task 1.4 tests passed")

Task 1.4 tests passed


In [307]:
for c in ["flat_imp", "flat_exp", "tou_imp", "tou_exp"]:
    print(f"{c:9s}", sorted(df[COL[c]].dropna().unique()))
tou_schedule(df[COL["tou_imp"]])

flat_imp  [np.float64(0.3)]
flat_exp  [np.float64(0.05)]
tou_imp   [np.float64(0.15), np.float64(0.25), np.float64(0.4)]
tou_exp   [np.float64(0.05)]


min         max      
weekend False True  False True 
hour                           
0        0.15  0.15  0.15  0.15
1        0.15  0.15  0.15  0.15
2        0.15  0.15  0.15  0.15
3        0.15  0.15  0.15  0.15
4        0.15  0.15  0.15  0.15
5        0.15  0.15  0.15  0.15
6        0.15  0.15  0.15  0.15
7        0.25  0.25  0.25  0.25
8        0.25  0.25  0.25  0.25
9        0.25  0.25  0.25  0.25
10       0.25  0.25  0.25  0.25
11       0.25  0.25  0.25  0.25
12       0.25  0.25  0.25  0.25
13       0.25  0.25  0.25  0.25
14       0.25  0.25  0.25  0.25
15       0.25  0.25  0.25  0.25
16       0.25  0.25  0.25  0.25
17       0.25  0.25  0.25  0.25
18       0.25  0.25  0.25  0.25
19       0.40  0.40  0.40  0.40
20       0.40  0.40  0.40  0.40
21       0.40  0.40  0.40  0.40
22       0.40  0.40  0.40  0.40
23       0.15  0.15  0.15  0.15

In [308]:
# What is one kWh moved by the battery worth on each tariff? (before losses)
h = df.index.hour
n_days = df.index.normalize().nunique()
peak_kwh_day = df.loc[(h >= 19) & (h < 23), COL["load"]].sum() * DT_H / n_days
print(f"Average load in the 19:00-23:00 peak window: {peak_kwh_day:.1f} kWh/day")

exp_rate = 0.05
pd.DataFrame({
    "store own solar, use later (AUD/kWh)":   {"Flat": 0.30 - exp_rate, "ToU into off-peak": 0.15 - exp_rate,
                                                "ToU into shoulder": 0.25 - exp_rate, "ToU into peak": 0.40 - exp_rate},
    "charge from grid off-peak, use later":  {"Flat": 0.0, "ToU into off-peak": 0.0,
                                                "ToU into shoulder": 0.25 - 0.15, "ToU into peak": 0.40 - 0.15},
}).round(2)

Average load in the 19:00-23:00 peak window: 2.6 kWh/day


,"store own solar, use later (AUD/kWh)","charge from grid off-peak, use later"
Flat,0.25,0.00
ToU into off-peak,0.10,0.00
ToU into shoulder,0.20,0.10
ToU into peak,0.35,0.25


> **Finding 1.4 - Tariffs.** The tariff columns match the supplied metadata (flat 0.30/0.05 AUD/kWh; ToU 0.15/0.25/0.40 AUD/kWh import, 0.05 export). The metadata gives the rates but not their timing; the data shows:
>
> | Window | Hours | Import (AUD/kWh) |
> |---|---|---|
> | Off-peak | 23:00 - 07:00| 0.15 |
> | Shoulder | 07:00 - 19:00 | 0.25 |
> | Peak | 19:00 - 23:00 | 0.40 |
>
> The schedule is identical on weekdays and weekends with no seasonal change. Export earns 0.05 AUD/kWh under every tariff.
>
> **So what for the battery.** Export is worth less than every import rate, so on every tariff a stored kWh of solar beats an exported one. On the **flat** tariff that is the battery's only value (0.25 AUD/kWh shifted, before losses); there is no price spread to arbitrage. On **ToU** the value depends on *when* the stored energy is used: 0.35 AUD/kWh into the peak, 0.20 into the shoulder, but only 0.10 overnight. ToU also opens **grid arbitrage**: charging off-peak and discharging in the peak is worth 0.25 AUD/kWh before losses, which matters in winter when there is little surplus solar. The peak window is only 4 hours and carries about 2.6 kWh/day of load, so a battery sized for the peak alone would be small; the sizing question (Part 4) is how much of the evening and overnight load is worth covering.


## Task 1.5 - Is the load independent across years?

### Background: check whether two years are really two years
The Year 1 and Year 2 load summaries in Task 1.3 were almost identical. `compare_years` lines the two years up interval by interval (by calendar position) and measures how often the values are identical, how correlated they are, and how many gaps in each year could be filled from the other. Irradiance is run through the same check as a control: real weather should not repeat.

In [309]:
def compare_years(y1: pd.Series, y2: pd.Series) -> dict:
    """Compare two equal-length years aligned by calendar position.

    Positional alignment needs equal-length years (true for 2022/2023, both 365 days);
    a leap year would shift every interval after 29 Feb, so the function refuses.
    """
    a, b = np.asarray(y1, float), np.asarray(y2, float)
    if len(a) != len(b):
        raise ValueError(f"Years differ in length ({len(a)} vs {len(b)}): positional alignment invalid")
    both = ~np.isnan(a) & ~np.isnan(b)
    return dict(
        identical_frac=float(np.isclose(a[both], b[both]).mean()),
        corr=float(np.corrcoef(a[both], b[both])[0, 1]),
        y1_gaps=int(np.isnan(a).sum()), y1_fillable_from_y2=int((np.isnan(a) & ~np.isnan(b)).sum()),
        y2_gaps=int(np.isnan(b).sum()), y2_fillable_from_y1=int((np.isnan(b) & ~np.isnan(a)).sum()),
        missing_both=int((np.isnan(a) & np.isnan(b)).sum()),
    )

In [310]:
"""Tests for Task 1.5"""
_c = compare_years([1, 2, np.nan, 4], [1, np.nan, 3, 4])
assert _c["identical_frac"] == 1.0                                   # compared only where both exist
assert _c["y1_fillable_from_y2"] == 1 and _c["y2_fillable_from_y1"] == 1 and _c["missing_both"] == 0

_c = compare_years([1, 2, 3, 4], [1, 2, 3, 9])
assert _c["identical_frac"] == 0.75

try:
    compare_years([1, 2, 3], [1, 2])
    raise AssertionError("should have refused unequal lengths")
except ValueError:
    pass
print("Task 1.5 tests passed")

Task 1.5 tests passed


In [311]:
pd.DataFrame({
    "load": compare_years(df1[COL["load"]], df2[COL["load"]]),
    "irradiance (control)": compare_years(df1[COL["irr"]], df2[COL["irr"]]),
}).round(4)

,load,irradiance (control)
identical_frac,1.0,0.4570
corr,1.0,0.8482
y1_gaps,184.0,184.0000
y1_fillable_from_y2,184.0,184.0000
y2_gaps,866.0,866.0000
y2_fillable_from_y1,866.0,866.0000
missing_both,0.0,0.0000


In [312]:
# Weekday labels: 1 Jan 2022 was a Saturday, 1 Jan 2023 a Sunday, so a copied year shifts weekdays by one day
print(df1.index[0].day_name(), "|", df2.index[0].day_name())
(df1[COL["load"]].groupby(df1.index.dayofweek).mean() * 24).round(1).rename("Y1 load kWh/day by weekday (0 = Mon)")

Saturday | Sunday


timestamp
0    28.8
1    28.8
2    28.2
3    29.1
4    29.6
5    29.8
6    29.3
Name: Y1 load kWh/day by weekday (0 = Mon), dtype: float64

> **Finding 1.5 - Load is repeated.** Year 2 load equals Year 1 load at every interval where both exist (identical fraction and correlation both 1.0), while irradiance does not (46% identical, r = 0.85). So this is a property of the load data, not a bug in the check. Implications:
> 1. Year 2 is not independent load data. Any difference in battery value between the years comes only from PV, weather and prices, which makes the years a clean comparison of those drivers.
> 2. The copy is by calendar date, so Year 2 weekday labels are shifted one day against the load behaviour. Year 1 load barely varies by weekday (28.2-29.8 kWh/day), so this has no practical effect, but day-of-week should not be used as a feature.
> 3. No interval is missing in both years, so all 1,050 load gaps can be filled exactly from the other year (Part 2).

## Task 1.6 - Spurious data

### Background: in range is not the same as consistent
Task 1.3 checked that every value is *possible* on its own. A value can still be wrong if it contradicts how the sensor behaves over time, or what another sensor says at the same moment. `quality_report` runs three kinds of check:

| Kind | Question | Checks |
|---|---|---|
| Range | Is each value possible on its own? | Negative or zero load, negative irradiance or PV, prices outside the NEM floor and cap. These repeat Task 1.3 as a formal record |
| Behaviour over time | Does the sensor behave like a real sensor? | Longest constant run (stuck meter or sensor), largest 5-minute temperature jump |
| Cross-sensor | Do two sensors measuring the same sunlight agree? | PV when irradiance is zero, no PV under strong sun (outage, snow or inverter trip, plausible at an alpine site), PV above system capacity, daily PV-to-irradiance ratio |

Cross-sensor checks are the strongest evidence, because one sensor checks the other. Each year is checked against its own PV column: measured PV in Year 1, the commercial forecast in Year 2. `quality_report` is adapted from my Assignment 1 data-quality check, with the PV column and system capacity as arguments.

In [313]:
def longest_constant_run(s: pd.Series) -> int:
    """Length (in intervals) of the longest run of identical consecutive values."""
    s = s.dropna()
    return int(s.groupby((s != s.shift()).cumsum()).size().max()) if len(s) else 0


def quality_report(df: pd.DataFrame, pv_col: str, cap_kw: float, col: dict = COL) -> pd.DataFrame:
    """Physical-plausibility checks in one table. Columns: check, value, note.

    pv_col: which PV column to check (measured in Year 1, commercial forecast in Year 2).
    cap_kw: estimated PV system size, used for the above-capacity check.
    """
    irr, load = df[col["irr"]], df[col["load"]]
    rows = [
        ("Negative load intervals", int((load < 0).sum()), "Should be 0 (no export-side meter in load)"),
        ("Zero-load intervals", int((load == 0).sum()), "A real home always has standby load"),
        ("Longest constant load run (intervals)", longest_constant_run(load), "Stuck meter if very long"),
        ("Longest constant temperature run (intervals)", longest_constant_run(df[col["temp"]]), "Stuck sensor if very long"),
        ("Max 5-min temperature jump (degC)", float(df[col["temp"]].diff().abs().max()), "Implausible if large"),
        ("Negative irradiance intervals", int((irr < 0).sum()), "Should be 0"),
    ]
    s = df[pv_col].dropna()
    if len(s):
        i = irr.reindex(s.index)
        rows += [
            ("PV rows checked", len(s), "Proves the check actually ran"),
            ("Solar > 0.01 kW at zero irradiance", int(((i == 0) & (s > 0.01)).sum()), "Should be 0"),
            ("Negative solar intervals", int((s < 0).sum()), "Should be 0"),
            ("Daytime (irr>100) intervals with solar <= 0.01 kW",
             int(((i > 100) & (s <= 0.01)).sum()), "Outage, snow or fault"),
            (f"Solar above 1.1 x {cap_kw} kW", int((s > 1.1 * cap_kw).sum()), "Above system capacity"),
        ]
    p = df[col["rrp"]].dropna()
    if len(p):
        rows += [("Prices outside [-1.00, 15.50] $/kWh", int(((p < -1.00) | (p > 15.50)).sum()),
                  "Outside 2022 NEM floor/cap: check units"),
                 ("Negative-price intervals", int((p < 0).sum()), "Drives the curtailment logic")]
    return pd.DataFrame(rows, columns=["check", "value", "note"])

In [314]:
"""Tests for Task 1.6"""
assert longest_constant_run(pd.Series([1, 1, 1, 2, 2])) == 3
assert longest_constant_run(pd.Series([np.nan, np.nan])) == 0            # empty column must not crash

_idx = pd.date_range("2022-01-01 12:00", periods=4, freq="5min")
_q = pd.DataFrame({
    COL["irr"]:      [0, 0, 500, 500],
    COL["load"]:     [0.0, 0.3, 0.3, 0.3],
    COL["temp"]:     [5, 5, 5, 6],
    COL["solar"]:    [0.0, 0.5, 0.0, 12.0],        # PV in the dark, no PV in sun, above 1.1 x 10 kW
    COL["solar_fc"]: np.nan,
    COL["rrp"]:      [0.10, -0.05, 16.0, 0.10],    # one negative price, one above the cap
}, index=_idx)
_r = quality_report(_q, COL["solar"], cap_kw=10).set_index("check")["value"]
assert _r["Zero-load intervals"] == 1
assert _r["Solar > 0.01 kW at zero irradiance"] == 1
assert _r["Daytime (irr>100) intervals with solar <= 0.01 kW"] == 1
assert _r["Solar above 1.1 x 10 kW"] == 1
assert _r["Prices outside [-1.00, 15.50] $/kWh"] == 1 and _r["Negative-price intervals"] == 1

# Year 2 shape: measured PV empty, forecast present -> the forecast is checked, not silently skipped
_y2 = _q.assign(**{COL["solar"]: np.nan, COL["solar_fc"]: [0.0, 0.5, 0.0, 12.0]})
_r2 = quality_report(_y2, COL["solar_fc"], cap_kw=10).set_index("check")["value"]
assert _r2["PV rows checked"] == 4 and _r2["Solar above 1.1 x 10 kW"] == 1
# an empty PV column skips the PV checks instead of reporting false zeros
assert "PV rows checked" not in quality_report(_y2, COL["solar"], cap_kw=10)["check"].values
print("Task 1.6 tests passed")

Task 1.6 tests passed


In [315]:
pd.concat({"Y1": quality_report(df1, COL["solar"], 10).set_index("check")["value"],
           "Y2": quality_report(df2, COL["solar_fc"], 10).set_index("check")["value"]}, axis=1)

,Y1,Y2
check,,
Negative load intervals,0.0,0.0
Zero-load intervals,0.0,0.0
Longest constant load run (intervals),14.0,14.0
Longest constant temperature run (intervals),73.0,81.0
Max 5-min temperature jump (degC),0.8,0.5
Negative irradiance intervals,0.0,0.0
PV rows checked,104936.0,105120.0
Solar > 0.01 kW at zero irradiance,0.0,0.0
Negative solar intervals,0.0,0.0


In [316]:
def daily_pv_irr_ratio(pv: pd.Series, irr: pd.Series) -> pd.Series:
    """Daily PV energy divided by daily irradiation (kWh per kWh/m^2, i.e. the array's effective kW).

    Uses complete days only: an irradiance outage with no matching PV gap divides
    by too little and inflates the ratio (to infinity if only night-time irradiance is left).
    """
    complete = pd.concat([pv, irr], axis=1).notna().all(axis=1).groupby(pv.index.normalize()).all()
    irr_day = irr.resample("D").sum() / 1000
    ratio = pv.resample("D").sum() / irr_day
    return ratio[complete & (irr_day > 0)]

In [317]:
"""Tests for Task 1.6 (ratio)"""
_i = pd.date_range("2024-01-01", periods=2 * 288, freq="5min")
_irr = pd.Series(np.where((_i.hour >= 6) & (_i.hour < 18), 800.0, 0.0), index=_i)
_pv = _irr / 1000 * 7.5                                     # a 7.5 kW-effective array
_irr_gap = _irr.copy(); _irr_gap.loc["2024-01-02 08:00":"2024-01-02 12:00"] = np.nan
_r = daily_pv_irr_ratio(_pv, _irr_gap)
assert np.isclose(_r.loc["2024-01-01"], 7.5)                # complete day: exact
assert pd.Timestamp("2024-01-02") not in _r.index           # day with a gap: excluded, not inflated
print("Task 1.6 ratio tests passed")

Task 1.6 ratio tests passed


In [318]:
ratio = pd.DataFrame({
    "Y1 measured":   daily_pv_irr_ratio(df1[COL["solar"]],    df1[COL["irr"]]),
    "Y2 commercial": daily_pv_irr_ratio(df2[COL["solar_fc"]], df2[COL["irr"]]),
})
summary = ratio.describe().loc[["count", "min", "50%", "max"]].round(2)
summary.loc["min day"] = ratio.idxmin().dt.date
summary.loc["max day"] = ratio.idxmax().dt.date
summary

,Y1 measured,Y2 commercial
count,359.0,346.0
min,6.95,7.29
50%,7.57,7.72
max,8.09,8.12
min day,2022-01-23,2023-12-12
max day,2022-06-29,2023-06-20


In [319]:
def longest_run_details(s: pd.Series) -> dict:
    """Where the longest constant run is: start, end, length and value (to judge stuck sensor vs calm night)."""
    s = s.dropna()
    groups = (s != s.shift()).cumsum()
    run = s[groups == s.groupby(groups).size().idxmax()]
    return dict(intervals=len(run), hours=round(len(run) * DT_H, 2),
                start=run.index[0], end=run.index[-1], value_C=run.iloc[0])

pd.DataFrame({"Y1": longest_run_details(df1[COL["temp"]]),
              "Y2": longest_run_details(df2[COL["temp"]])})

,Y1,Y2
intervals,73,81
hours,6.08,6.75
start,2022-08-24 18:30:00,2023-07-05 02:30:00
end,2022-08-25 00:30:00,2023-07-05 09:10:00
value_C,-0.8,4.2


In [320]:
# Is 5-minute irradiance real 5-minute data? If it was interpolated from hourly values,
# the 5-minute steps inside each hour are all the same size (+/-1 W/m^2 for rounding).
irr1 = df1[COL["irr"]]
step = irr1.diff()
inside_hour = (irr1.index.minute != 0) & step.notna()
spread = step[inside_hour].groupby(irr1.index[inside_hour].floor("h")).agg(lambda x: x.max() - x.min())
daylight = irr1.resample("h").mean().reindex(spread.index) > 50
print(f"Daylight hours whose 5-min irradiance steps are all equal: {(spread[daylight] <= 1).mean():.1%}")

Daylight hours whose 5-min irradiance steps are all equal: 100.0%


> **Finding 1.6 - No spurious data.** Every check passes in both years (104,936 measured PV rows in Year 1 and 105,120 commercial-forecast rows in Year 2): no PV in the dark, no PV dropout under strong sun, nothing above the ~10 kW system capacity, no zero or negative load, and every price within the NEM floor and cap. The 3,114 negative-price intervals in Year 1 (about 3% of the year) are real market events, not errors; they are when PV can be curtailed (Part 7).
>
> **Temperature.** The longest constant runs (73 intervals, about 6 h, in Year 1; 81, about 6.75 h, in Year 2) are both overnight in winter: from the evening of 24 Aug 2022 at -0.8 °C, and the early morning of 5 Jul 2023 at 4.2 °C. Temperature is recorded to 0.1 °C and never moves more than 0.8 °C in 5 minutes, so a flat reading through a calm alpine night is expected. This is not a stuck sensor, so the data is kept.
>
> **PV vs irradiance.** On complete days the daily PV-to-irradiance ratio is very stable: 6.95 to 8.09 (median 7.57) in Year 1 and 7.29 to 8.12 (median 7.72) in Year 2. There are no snow-covered, soiled or faulted days to remove. The ratio is lowest in summer and highest in winter, as expected from the negative temperature coefficient (cold panels are more efficient). It must be computed on complete days only: on days with an irradiance outage it is inflated, because the PV columns have no matching gap. The ratio (~7.6 kW per kW/m²) is also a first estimate of effective array size for Part 4.
>
> **Irradiance resolution.** In every daylight hour, the 5-minute irradiance rises in equal steps, so it is **hourly data interpolated to 5 minutes**, while measured PV is true 5-minute data. Cloud transients therefore appear in PV but not in irradiance. A weather-based PV model (Part 9) cannot predict sub-hourly dips, which sets a floor under its 5-minute error.

## Part 1 summary - what we are working with

Part 1 diagnosed the data without changing it. Part 2 fixes only what this list says needs fixing.

| # | Issue | Found in | What happens next |
|---|---|---|---|
| 1 | 22 sensor outages blank irradiance, temperature, load and measured PV together (184 rows in Y1, 866 in Y2; up to 24 h) | 1.2 | Part 2: fill or flag, column by column |
| 2 | Measured PV only in Y1, commercial forecast only in Y2 | 1.2 | By design. Y2 "actual" PV = commercial forecast (per the brief) |
| 3 | No Y2 spot prices | 1.2 | Part 3: source 2023 NSW1 prices from AEMO |
| 4 | Y2 load is an exact copy of Y1, with weekdays shifted one day | 1.5 | Part 2: fill load gaps from the other year; no day-of-week features |
| 5 | Prices already in $/kWh; hit the floor and cap; 3,114 negative intervals | 1.3, 1.6 | Keep as-is; curtail PV at negative prices (Part 7) |
| 6 | Irradiance is hourly data interpolated to 5 min | 1.6 | Caveat for the PV model (Part 9) |
| 7 | No spurious data; temperature runs are calm winter nights | 1.6 | Nothing to remove |
| 8 | No daylight-saving jumps (likely AEST) | 1.2 | Confirm with solar noon (Part 4) |

# Part 2 - Data Cleaning

Part 1 diagnosed the data; this Part fixes only what the Part 1 summary lists. Filling a gap invents data, so each column is filled only if a later step needs it complete, using the simplest method that is defensible for that use.

## Task 2.1 - Cleaning decisions

| Column | Used later for | Gaps (Part 1) | Decision | Why |
|---|---|---|---|---|
| Load | every bill and optimisation, both years | 184 (Y1), 866 (Y2) | **Fill exactly** from the other year (Task 2.2) | The years are copies (Finding 1.5) and no interval is missing in both, so this recovers the true value |
| "Actual" PV in Y2 (= commercial forecast) | perfect-foresight optimisation and settlement | 0 | Nothing to fill | Complete by design |
| Measured PV (Y1) | training the PV model (Part 9) | 184 | **Do not fill**; drop those rows from training | Filling a training target would teach the model invented data |
| Irradiance, temperature | PV model inputs; battery temperature limits | 184 (Y1), 866 (Y2) | **Fill and flag** (Task 2.3) | Inputs must be complete for a forecast at every hour; flagged rows are left out when forecasts are scored |
| Spot price (Y1) | price forecast training | 0 | Nothing to fill | Y2 prices are sourced in Part 3 |
| Everything else | | | Keep as-is | No spurious data (Finding 1.6) |

## Task 2.2 - Fill load from the other year

In [321]:
def fill_from_other_year(s: pd.Series, offset=pd.DateOffset(years=1)):
    """Fill gaps with the value at the same calendar position one year earlier or later.

    Only valid because Finding 1.5 showed the two years of load are copies.
    Returns (filled series, boolean mask of rows that were filled).
    """
    prev = s.reindex(s.index - offset).set_axis(s.index)
    nxt = s.reindex(s.index + offset).set_axis(s.index)

    filled = s.fillna(prev).fillna(nxt)

    return filled, s.isna() & filled.notna()

In [322]:
"""Tests for Task 2.2"""
_i = pd.date_range("2022-01-01", periods=3, freq="D").append(pd.date_range("2023-01-01", periods=3, freq="D"))
_f, _m = fill_from_other_year(pd.Series([1, np.nan, 3, 1, 2, np.nan], index=_i))
assert _f.tolist() == [1, 2, 3, 1, 2, 3]                                  # filled in both directions
assert _m.tolist() == [False, True, False, False, False, True]            # only the gaps are flagged

# missing in BOTH years must stay missing, not be invented
_i2 = pd.to_datetime(["2022-01-01", "2022-01-02", "2023-01-01", "2023-01-02"])
_f2, _ = fill_from_other_year(pd.Series([np.nan, 1, np.nan, 1], index=_i2))
assert np.isnan(_f2.iloc[0]) and np.isnan(_f2.iloc[2])
print("Task 2.2 tests passed")

Task 2.2 tests passed


In [323]:
load_clean, load_filled = fill_from_other_year(df[COL["load"]])
print("Load gaps filled:", int(load_filled.sum()), "| still missing:", int(load_clean.isna().sum()))

Load gaps filled: 1050 | still missing: 0


## Task 2.3 - Fill irradiance and temperature

### Background: why not straight-line interpolation?
Weather is not copied between years (the irradiance control in Task 1.5), so it cannot be filled exactly. Straight-line interpolation is fine for a few minutes but wrong for these outages: the shortest is 1 hour and the longest is 24 hours (1-2 July 2023), and a line across a day erases the daily solar shape. Instead each missing value is the **mean of the same time of day on the 3 days either side**, which keeps night at zero and noon bright. The method is then tested on real data by blanking complete days and refilling them.

In [324]:
def fill_same_time_nearby_days(s: pd.Series, days: int = 3):
    """Fill each gap with the mean of the same time of day on the `days` days either side.

    Neighbouring days that are themselves missing are skipped by the mean.
    Returns (filled series, boolean mask of rows that were filled).
    """
    surrounding = [s.shift(freq=pd.Timedelta(days=k)).reindex(s.index) 
                   for k in range (-days,days+1) if k!=0]
    
    filled = s.fillna(pd.concat(surrounding, axis=1).mean(axis=1))

    return filled, s.isna() & filled.notna()



In [325]:
"""Tests for Task 2.3"""
_i = pd.date_range("2024-01-01", periods=5 * 24, freq="h")
_true = pd.Series(np.tile(np.r_[np.zeros(6), np.full(12, 500.0), np.zeros(6)], 5), index=_i)
_gap = _true.copy(); _gap.loc["2024-01-03"] = np.nan                 # one whole day missing
_f, _m = fill_same_time_nearby_days(_gap, days=2)
assert np.allclose(_f.values, _true.values)                          # daily shape restored, night stays zero
assert _m.sum() == 24
assert not np.allclose(_gap.interpolate().values, _true.values)      # a straight line would lose the noon peak
print("Task 2.3 tests passed")

Task 2.3 tests passed


In [326]:

rng = np.random.default_rng(RNG_SEED)
irr = df[COL["irr"]]
complete_days = irr.notna().groupby(irr.index.normalize()).all()
test_days = rng.choice(complete_days[complete_days].index[3:-3], size=50, replace=False)
blanked = irr.copy()
for d in test_days:
    blanked.loc[str(pd.Timestamp(d).date())] = np.nan
daylight = blanked.isna() & irr.notna() & (irr > 0)
err = pd.DataFrame({
    "nearby-days mean": (fill_same_time_nearby_days(blanked)[0] - irr)[daylight].abs(),
    "straight line":    (blanked.interpolate(method="time") - irr)[daylight].abs(),
})
print("Daylight MAE when refilling 50 whole blanked days (W/m^2):")
err.mean().round(1)

Daylight MAE when refilling 50 whole blanked days (W/m^2):


nearby-days mean     80.5
straight line       332.5
dtype: float64

In [327]:
weather, flags = {}, pd.DataFrame(index=df.index)
for c in ["irr", "temp"]:
    weather[c], flags[f"{c}_filled"] = fill_same_time_nearby_days(df[COL[c]])
    print(f"{c:5s} filled {int(flags[f'{c}_filled'].sum()):4d} | still missing {int(weather[c].isna().sum())}")
flags["load_filled"] = load_filled
flags["pv_meas_missing"] = df[COL["solar"]].isna() & (df.index < df2.index.min())   # Y1 rows to drop from PV training

irr   filled 1050 | still missing 0
temp  filled 1050 | still missing 0


In [328]:
# Figure 2.1 - Before/after: the 24-hour outage (1-2 July 2023), refilled two ways
win = slice("2023-06-30", "2023-07-03")
raw = df[COL["irr"]].loc[win]
gap = raw.isna()
straight = df[COL["irr"]].interpolate(method="time").loc[win]

fig = go.Figure()
fig.add_trace(go.Scatter(x=raw.index, y=raw, name="Measured",
                         line=dict(color="black", width=1.5)))
fig.add_trace(go.Scatter(x=raw.index, y=weather["irr"].loc[win].where(gap), name="Filled: same time on nearby days (used)",
                         line=dict(color="#1f77b4", width=2.5)))
fig.add_trace(go.Scatter(x=raw.index, y=straight.where(gap), name="Straight-line interpolation (rejected)",
                         line=dict(color="#d62728", width=2, dash="dash")))
fig.add_vrect(x0=raw.index[gap].min(), x1=raw.index[gap].max(), fillcolor="grey", opacity=0.15, line_width=0,
              annotation_text="24 h sensor outage", annotation_position="top left")
fig.update_layout(
    title="Filling the longest outage: the nearby-days fill keeps the daily solar shape; a straight line erases the day",
    xaxis_title="Time", yaxis_title="Irradiance (W/m²)", template="plotly_white", height=420,
    legend=dict(orientation="h", yanchor="top", y=-0.18, x=0))
fig.show()

*Figure 2.1 - The 24-hour outage of 1-2 July 2023. A straight line joins midnight to midnight (both zero), so it fills the whole day as darkness (peak 0 W/m²). The nearby-days fill gives a peak of about 280 W/m², in line with the measured days either side (180-490 W/m²).*

> **Finding 2.2-2.3 - Gaps.** All 1,050 load gaps were filled exactly from the other year. All 1,050 irradiance and temperature gaps were filled from the same time of day on the 3 days either side. Tested on 50 real days blanked on purpose, this gives a daylight error of about 80 W/m², against 333 W/m² for straight-line interpolation. The error is not small, because the weather on a given day cannot be known. That is acceptable: filled weather rows are only inputs (to the Year 2 PV forecast and the battery temperature limits), cover under 1% of each year, and are flagged so no forecast is scored on them. Measured Year 1 PV was not filled, so the PV model never trains on invented targets.

## Task 2.4 - Assemble the clean dataset
One analysis-ready frame `dfc` with short column names, the brief's definition of "actual" PV (measured in Year 1, commercial forecast in Year 2), and the fill flags. **Everything from Part 3 onwards uses `dfc`, never the raw `df`.** The contract checks below prove the frame matches the decisions in Task 2.1.

In [329]:
def build_clean(df: pd.DataFrame, load: pd.Series, weather: dict, flags: pd.DataFrame, cut) -> pd.DataFrame:
    """Analysis-ready frame. 'pv_kW' follows the brief: measured PV in Year 1, commercial forecast in Year 2.

    Columns: year, load_kW, pv_kW, pv_meas_kW, pv_comm_kW, irr_Wm2, temp_C, rrp_kWh,
    flat_imp, flat_exp, tou_imp, tou_exp, then the flag columns joined on.
    """
    y1 = df.index < cut
    out = pd.DataFrame({
        "year":         np.where(y1,1,2),
        "load_kW":      load,
        "pv_meas_kW":   df[COL["solar"]],
        "pv_comm_kW":   df[COL["solar_fc"]],
        "pv_kW":        df[COL["solar"]].where(y1, df[COL["solar_fc"]]),
        "irr_Wm2":      weather["irr"],
        "temp_C":       weather["temp"],
        "rrp_kWh":      df[COL["rrp"]],

        **{c: df[COL[c]] for c in ["flat_imp", "flat_exp", "tou_imp", "tou_exp"]},

    }, index =df.index)

    return out.join(flags)

In [330]:
dfc = build_clean(df, load_clean, weather, flags, cut=df2.index.min())

# Contract checks: what every later Part is allowed to assume (matches Task 2.1)
must_be_complete = ["load_kW", "irr_Wm2", "temp_C", "flat_imp", "flat_exp", "tou_imp", "tou_exp"]
assert dfc[must_be_complete].notna().all().all(),                         "gaps left in a column that must be complete"
assert dfc.loc[dfc.year == 2, "pv_kW"].notna().all(),                      "Year 2 'actual' PV must be complete"
assert dfc["pv_kW"].isna().sum() == dfc["pv_meas_missing"].sum(),          "only the unfilled Y1 PV rows may be blank"
assert dfc.loc[dfc.year == 2, "pv_kW"].equals(dfc.loc[dfc.year == 2, "pv_comm_kW"])
assert dfc.loc[dfc.year == 2, "rrp_kWh"].isna().all()                      # Y2 prices still to be sourced (Part 3)
print("Clean dataset:", dfc.shape, "| all contract checks passed")
dfc.isna().sum()[lambda s: s > 0]

Clean dataset: (210240, 16) | all contract checks passed


pv_meas_kW    105304
pv_comm_kW    105120
pv_kW            184
rrp_kWh       105120
dtype: int64

> **Finding 2.4 - Clean dataset.** `dfc` holds 210,240 rows and 16 columns. Load, weather and tariffs are complete; "actual" PV is complete in Year 2 and blank only for the 184 Year 1 rows deliberately left unfilled; Year 2 spot prices remain blank until Part 3. Every filled value is flagged (`load_filled`, `irr_filled`, `temp_filled`), so any later result can be checked with and without filled data.

# Part 3 - Tariffs and Prices

Part 3 builds the price every later Part uses: the import and export price for each of the three tariffs at every 5-minute interval. Flat and ToU come straight from the site data (Task 1.4). Wholesale pass-through needs the spot price for both years. The site data only has Year 1 prices, so Year 2 is sourced from AEMO first.

## Task 3.1 - Source the 2023 spot prices from AEMO

### Background: download once, align carefully
`load_spot_prices` is adapted from my thesis downloader. It is cached twice, so a full top-to-bottom run never downloads again: one raw CSV per month, and one combined CSV for the whole range, both in `PRICE_DIR`. Two changes from the thesis version: a month that fails to download now **raises an error** instead of being skipped, because a silently missing month would corrupt the annual bill; and it returns $/MWh exactly as published, with conversion to $/kWh done in one visible place.

AEMO stamps each 5-minute interval at its **end** (`SETTLEMENTDATE` 00:05 covers 00:00-00:05). The site data could use either convention, so rather than assume, January 2022 is downloaded as well and compared against the Year 1 prices already in the site data. The convention that matches exactly is the one used for 2023.

In [331]:
def months_between(start: str, end: str) -> list:
    """'YYYYMM' strings for every month from start to end inclusive, e.g. ('2022-11', '2023-02')."""
    return [p.strftime("%Y%m") for p in pd.period_range(start, end, freq="M")]


def load_spot_prices(region: str, start: str, end: str, cache_dir) -> pd.Series:
    """AEMO 5-minute regional reference price ($/MWh) for whole months `start`..`end`.

    Index is AEMO's SETTLEMENTDATE (interval END, NEM time). Cached in `cache_dir`:
      1. a combined CSV for this region and range: read immediately if it exists;
      2. one raw CSV per month: only missing months are downloaded.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    combined = cache_dir / f"spot_{region}_{start}_{end}.csv"
    if combined.exists():
        return pd.read_csv(combined, index_col=0, parse_dates=True)["rrp_MWh"]

    frames = []
    for ym in months_between(start, end):
        month_file = cache_dir / f"PRICE_AND_DEMAND_{ym}_{region}.csv"
        if not month_file.exists():
            url = AEMO_PRICE_URL.format(ym=ym, region=region)
            print(f"Downloading {url}")
            resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()                      # fail loudly: a missing month corrupts the year
            if not resp.content.lstrip().startswith(b"REGION"):  # AEMO can return an HTML page with status 200
                raise ValueError(f"{url} did not return a price CSV; download it manually into {cache_dir}")
            month_file.write_bytes(resp.content)         # only a validated file is cached
        frames.append(pd.read_csv(month_file))

    raw = pd.concat(frames, ignore_index=True)
    s = (raw.assign(settlement_end=pd.to_datetime(raw["SETTLEMENTDATE"]))
            .set_index("settlement_end")["RRP"].sort_index().rename("rrp_MWh"))
    s = s[~s.index.duplicated()]
    s.to_csv(combined)
    return s


def to_period_start(s: pd.Series, minutes: int = INTERVAL_MIN) -> pd.Series:
    """Re-stamp interval-END timestamps (AEMO) as interval-START timestamps (site data)."""
    out = s.copy()
    out.index = out.index - pd.Timedelta(minutes=minutes)
    out.index.name = "timestamp"
    return out

In [332]:
"""Tests for Task 3.1 (offline: uses a fake cached month, never the network)"""

assert months_between("2022-11", "2023-02") == ["202211", "202212", "202301", "202302"]

with tempfile.TemporaryDirectory() as _d:
    _d = Path(_d)
    pd.DataFrame({"REGION": "NSW1", "SETTLEMENTDATE": ["2022/01/01 00:05:00", "2022/01/01 00:10:00"],
                  "TOTALDEMAND": 1.0, "RRP": [100.0, -50.0], "PERIODTYPE": "TRADE"}
                ).to_csv(_d / "PRICE_AND_DEMAND_202201_NSW1.csv", index=False)
    _s = load_spot_prices("NSW1", "2022-01", "2022-01", _d)            # reads the cached month, no download
    assert _s.tolist() == [100.0, -50.0]
    assert (_d / "spot_NSW1_2022-01_2022-01.csv").exists()              # combined cache written
    assert load_spot_prices("NSW1", "2022-01", "2022-01", _d).equals(_s)  # second call: combined cache, same data

assert to_period_start(_s).index[0] == pd.Timestamp("2022-01-01 00:00")  # interval ending 00:05 starts at 00:00
print("Task 3.1 tests passed")

Task 3.1 tests passed


In [333]:
# Alignment check: which timestamp convention reproduces the Year 1 prices in the site data?
jan22 = load_spot_prices("NSW1", "2022-01", "2022-01", PRICE_DIR) / 1000          # $/MWh -> $/kWh
site_jan22 = df.loc["2022-01", COL["rrp"]]
conventions = {"AEMO stamp = interval END (as published)": jan22,
               "AEMO stamp shifted to interval START":    to_period_start(jan22)}
pd.DataFrame({
    name: {"intervals compared": int(site_jan22.index.isin(s.index).sum()),
           "exact matches (to $0.00001/kWh)": f"{np.isclose(site_jan22, s.reindex(site_jan22.index), atol=1e-5).mean():.1%}",
           "max abs difference ($/kWh)": float((site_jan22 - s.reindex(site_jan22.index)).abs().max())}
    for name, s in conventions.items()
}).T

,intervals compared,exact matches (to $0.00001/kWh),max abs difference ($/kWh)
AEMO stamp = interval END (as published),8927,100.0%,0.0
AEMO stamp shifted to interval START,8928,22.7%,0.37464


In [334]:
# Year 2 spot price ($/kWh). The alignment check showed the site data uses AEMO's interval-END stamps
# as published, so no shift. The first Year 2 interval (2023-01-01 00:00) ends the last AEMO interval
# of December 2022, so the download starts in December 2022.
spot_y2 = load_spot_prices("NSW1", "2022-12", "2023-12", PRICE_DIR) / 1000
spot_y2 = spot_y2.reindex(dfc.index[dfc.year == 2])
assert spot_y2.notna().all(), f"{spot_y2.isna().sum()} Year 2 intervals have no AEMO price"

dfc.loc[dfc.year == 2, "rrp_kWh"] = spot_y2.values
assert dfc["rrp_kWh"].notna().all()                               # both years now have a spot price
print("Year 2 spot price added:", len(spot_y2), "intervals")
dfc.groupby("year")["rrp_kWh"].describe()[["min", "50%", "mean", "max"]].round(3)

Year 2 spot price added: 105120 intervals


,min,50%,mean,max
year,,,,
1,-1.0,0.134,0.183,15.5
2,-1.0,0.086,0.096,16.6


> **Finding 3.1 - Spot prices.** AEMO's timestamps *as published* (interval end) reproduce the site's January 2022 prices exactly (100% of 8,927 intervals, maximum difference 0), while shifting them to interval start matches only 22.7%. So the site's Year 1 prices are AEMO NSW1 prices on interval-end stamps, and the 2023 prices are aligned the same way. December 2022 is downloaded as well because the first Year 2 interval (00:00 on 1 January 2023) is the last AEMO interval of December 2022.
>
> 2023 prices were about half of 2022's: the mean fell from 183 to 96 AUD/MWh and the median from 134 to 86 AUD/MWh. 2022 was an energy-crisis year, with gas price spikes and coal outages culminating in AEMO suspending the market in June 2022. Negative prices more than doubled, from 3.0% of intervals in 2022 to 6.3% in 2023. The 2023 maximum (16,600 AUD/MWh on 4 September 2023) is a genuine price: the market price cap rose from 15,500 to 16,600 AUD/MWh on 1 July 2023.
>
> For the battery, this means a wholesale pass-through customer in 2023 earns less from price spikes than in 2022, but gets twice as many chances to charge cheaply (or be paid to charge) and to curtail exports when prices are negative.

## Task 3.2 - Wholesale pass-through price: decisions

The site metadata says the wholesale tariff is "taken directly from market spot price data". The brief adds that a pass-through customer also pays time-of-day network costs, market fees, environmental certificates and hedging costs. This matches how Amber Electric works: all of these are passed through at cost, and a monthly subscription ($25, as in the brief) is the only margin. The flat and ToU tariffs are used exactly as given.

**Import ($/kWh)** = [spot x loss factor + network(period, financial year) + environmental + market fees + hedging] x (1 + GST)
**Export ($/kWh)** = spot (negative prices included, so exporting at a negative price costs money and PV should curtail)

| Component | Value | Source / reason |
|---|---|---|
| Loss factor | x 1.034 (DLF 6.09%, MLF -2.52%) | ACIL Allen for the AER, DMO 2023-24, Essential NSLP (Table 4.33) |
| Network, 1 Jan-30 Jun 2023 | peak 16.12, shoulder 12.08, off-peak 4.90 c/kWh | Essential Energy BLNT3AL, indicative 2022-23 schedule (Mar 2022). *Limitation: indicative, not the final price list* |
| Network, 1 Jul-31 Dec 2023 | peak 17.64, shoulder 12.70, off-peak 5.05 c/kWh | Essential Energy BLNT3AL, 2023-24 network price list |
| Network time windows | the assignment's ToU windows (setting `NETWORK_WINDOWS`) | Keeps one definition of peak/shoulder/off-peak across all three tariffs. Essential's own windows differ (peak 5-8 pm weekdays); **tutor's advice requested**, switchable in Part 0 |
| Environmental schemes | 1.78 c/kWh | LRET 7.27 + SRES 6.86 + NSW ESS 3.26 + PDRS 0.42 $/MWh (ACIL Allen Tables 4.6-4.10) |
| Market fees | 0.16 c/kWh | AEMO fees 1.15 + ancillary services 0.42 $/MWh (Tables 4.11-4.12) |
| Hedging | 0.26 c/kWh | Prudential costs only (AEMO 0.77 + hedge 1.86 $/MWh). A pass-through customer is exposed to the spot price, so full hedge premiums do not apply |
| Daily charges | excluded from all three tariffs | The given flat and ToU tariffs have none; excluding them keeps the comparison like-for-like |
| GST | +10% on imports | The given flat and ToU rates are taken as GST-inclusive |

Only Year 2 (2023) is priced: it is the year the optimisation is run on.

In [335]:
def network_period(index: pd.DatetimeIndex, windows: str = NETWORK_WINDOWS) -> np.ndarray:
    """'peak' / 'shoulder' / 'offpeak' for each timestamp, by the hour of its stamp (as the site's ToU tariff does)."""
    h = index.hour
    if windows == "assignment":        # Finding 1.4: identical every day
        return np.select([(h >= 19) & (h < 23), (h >= 7) & (h < 19)], ["peak", "shoulder"], "offpeak")
    if windows == "essential":         # weekdays only; weekends all off-peak; applied on the data's AEST clock
        wk = index.dayofweek < 5
        return np.select([wk & (h >= 17) & (h < 20),
                          wk & (((h >= 7) & (h < 17)) | ((h >= 20) & (h < 22)))], ["peak", "shoulder"], "offpeak")
    raise ValueError(f"unknown NETWORK_WINDOWS: {windows!r}")


def financial_year(index: pd.DatetimeIndex) -> np.ndarray:
    """Australian financial year label for each timestamp, e.g. '2022-23' (the year starts 1 July)."""
    start = np.where(index.month >= 7, index.year, index.year - 1)
    return np.array([f"{y}-{str(y + 1)[-2:]}" for y in start])

In [336]:
"""Tests for Task 3.3 helpers"""
_t = pd.to_datetime(["2023-01-02 06:55", "2023-01-02 07:00", "2023-01-02 19:00", "2023-01-07 18:00"])  # Mon x3, Sat
assert network_period(_t, "assignment").tolist() == ["offpeak", "shoulder", "peak", "shoulder"]
assert network_period(_t, "essential").tolist()  == ["offpeak", "shoulder", "peak", "offpeak"]       # 19:00 Mon in 5-8 pm peak; weekend off-peak
assert financial_year(pd.to_datetime(["2023-06-30 23:55", "2023-07-01 00:05"])).tolist() == ["2022-23", "2023-24"]
print("Task 3.3 helper tests passed")

Task 3.3 helper tests passed


## Task 3.3 - Build the wholesale pass-through price
`wholesale_retail_price` turns the spot price into the import and export price a pass-through customer pays, using the Part 0 settings and the formula in Task 3.2. The tests below fix the expected answer for a few hand-calculated intervals, so they catch unit slips (c/kWh vs $/kWh) as well as logic errors.

In [337]:
def wholesale_retail_price(spot: pd.Series, windows: str = NETWORK_WINDOWS) -> pd.DataFrame:
    """Wholesale pass-through prices for each interval of `spot` ($/kWh, interval-end stamps).

    import = (spot x LOSS_FACTOR + network + environmental + fees + hedging) x (1 + GST)
    export = spot   (negative included)
    network comes from NETWORK_RATES[financial year][period]; all adders are in c/kWh, so divide by 100.
    Returns a DataFrame on spot's index with columns: period, network_kWh, import_kWh, export_kWh.
    """
    # period for each interval (network_period) and financial year (financial_year)
    period = network_period(spot.index, windows)
    fy = financial_year(spot.index)

    network = np.array([NETWORK_RATES[f][p] for f, p in zip(fy, period)]) / 100   # c/kWh -> $/kWh

    import_kwh = (spot * LOSS_FACTOR + network + (ENV_C_KWH + FEES_C_KWH + HEDGE_C_KWH) / 100) * (1 + GST)

    out = pd.DataFrame({"period" : period, "network_kWh": network,"import_kWh": import_kwh, "export_kWh": spot }, index=spot.index)
    return out

In [338]:
"""Tests for Task 3.3 (hand-calculated)"""
_idx = pd.to_datetime(["2023-01-10 02:00",    # off-peak, FY 2022-23
                       "2023-08-15 19:30",    # peak, FY 2023-24 (assignment windows)
                       "2023-08-15 12:00"])   # shoulder, FY 2023-24, negative spot
_w = wholesale_retail_price(pd.Series([0.05, 0.10, -0.04], index=_idx), windows="assignment")
_adders = (1.781 + 0.157 + 0.263) / 100                                        # 0.02201 $/kWh
assert _w["period"].tolist() == ["offpeak", "peak", "shoulder"]
assert np.allclose(_w["network_kWh"], [0.048957, 0.176443, 0.126958])
assert np.isclose(_w["import_kWh"].iloc[0], (0.05 * 1.034 + 0.048957 + _adders) * 1.1)   # = 0.13493
assert np.isclose(_w["import_kWh"].iloc[1], (0.10 * 1.034 + 0.176443 + _adders) * 1.1)   # = 0.33204
assert np.isclose(_w["import_kWh"].iloc[2], (-0.04 * 1.034 + 0.126958 + _adders) * 1.1)  # negative spot lowers the import price
assert _w["export_kWh"].tolist() == [0.05, 0.10, -0.04]                                  # export = spot, negatives kept
assert _w.index.equals(_idx)
print("Task 3.3 tests passed")

Task 3.3 tests passed


In [339]:
wholesale = wholesale_retail_price(dfc.loc[dfc.year == 2, "rrp_kWh"])
print("Network windows used:", NETWORK_WINDOWS)
wholesale.groupby("period")[["network_kWh", "import_kWh", "export_kWh"]].mean().round(3)

Network windows used: assignment


,network_kWh,import_kWh,export_kWh
period,,,
offpeak,0.050,0.183,0.091
peak,0.169,0.343,0.117
shoulder,0.124,0.265,0.092


## Task 3.4 - One price table for all three tariffs
`prices` holds the import and export price ($/kWh) for flat, ToU and wholesale at every Year 2 interval. From Part 6 onwards every bill and every optimisation reads from this one table, so a tariff is just a pair of columns: the same code runs all three.

In [340]:
TARIFFS = ["flat", "tou", "wholesale"]
y2 = dfc.year == 2
prices = pd.DataFrame({
    "flat_imp":      dfc.loc[y2, "flat_imp"],
    "flat_exp":      dfc.loc[y2, "flat_exp"],
    "tou_imp":       dfc.loc[y2, "tou_imp"],
    "tou_exp":       dfc.loc[y2, "tou_exp"],
    "wholesale_imp": wholesale["import_kWh"],
    "wholesale_exp": wholesale["export_kWh"],
})

# Contract checks: complete, Year 2 only, one row per interval
assert prices.notna().all().all() and len(prices) == y2.sum()
assert prices.index.equals(dfc.index[y2])
assert (prices[["flat_imp", "tou_imp"]] > prices[["flat_exp", "tou_exp"]].values).all().all()   # fixed tariffs: import > export
print("prices:", prices.shape)
prices.describe().loc[["min", "50%", "mean", "max"]].round(3)

prices: (105120, 6)


,flat_imp,flat_exp,tou_imp,tou_exp,wholesale_imp,wholesale_exp
min,0.3,0.05,0.150,0.05,-0.980,-1.000
50%,0.3,0.05,0.250,0.05,0.228,0.086
mean,0.3,0.05,0.242,0.05,0.251,0.096
max,0.3,0.05,0.400,0.05,19.045,16.600


## Task 3.5 - How the three tariffs compare through the day

In [341]:
# Figure 3.1 - Average price by time of day, Year 2 (import solid, export dashed)
by_hour = prices.groupby(prices.index.hour + prices.index.minute / 60).mean()
colours = {"flat": "#7f7f7f", "tou": "#1f77b4", "wholesale": "#d62728"}
names = {"flat": "Flat", "tou": "Time of use", "wholesale": "Wholesale pass-through"}

fig = go.Figure()
for t in TARIFFS:
    fig.add_trace(go.Scatter(x=by_hour.index, y=by_hour[f"{t}_imp"] * 100, name=f"{names[t]} - import",
                             line=dict(color=colours[t], width=2.5, shape="hv" if t != "wholesale" else "linear")))
    fig.add_trace(go.Scatter(x=by_hour.index, y=by_hour[f"{t}_exp"] * 100, name=f"{names[t]} - export",
                             line=dict(color=colours[t], width=1.5, dash="dash", shape="hv" if t != "wholesale" else "linear")))
fig.update_layout(
    title="Average price by time of day (2023): wholesale is cheapest at midday but dearest 4-7 pm; ToU is dearest 7-11 pm",
    xaxis=dict(title="Hour of day", tickmode="array", tickvals=list(range(0, 25, 3))),
    yaxis_title="Price (c/kWh)", template="plotly_white", height=460,
    legend=dict(orientation="h", yanchor="top", y=-0.18, x=0))
fig.show()

# Numbers behind the figure: average import price in four parts of the day
blocks = pd.cut(prices.index.hour, [0, 7, 10, 16, 19, 23, 24], right=False,
                labels=["00-07", "07-10", "10-16 (solar)", "16-19", "19-23 (peak)", "23-24"])
(prices.groupby(blocks, observed=True)[[f"{t}_imp" for t in TARIFFS] + ["wholesale_exp"]].mean() * 100).round(1)

,flat_imp,tou_imp,wholesale_imp,wholesale_exp
00-07,30.0,15.0,18.1,9.0
07-10,30.0,25.0,23.4,6.4
10-16 (solar),30.0,25.0,21.8,5.1
16-19,30.0,25.0,39.2,20.3
19-23 (peak),30.0,40.0,34.3,11.7
23-24,30.0,15.0,19.3,10.1


> **Finding 3.2-3.5 - Tariffs.** TODO in my words: (1) the wholesale price formula and its sources (Task 3.2), and that the network windows follow the assignment's ToU windows pending tutor advice; (2) how wholesale compares with ToU by time of day (table above): cheaper or dearer in each block, and where the export price beats 5c; (3) what that means for the battery on each tariff (e.g. flat rewards only self-consumption; ToU rewards shifting into 19-23; wholesale rewards charging at midday and avoiding/selling into evening spikes).

# Part 4 - Site Characterisation

Part 4 answers the questions the battery design depends on: is the clock what we think it is, how much energy could a battery actually move each day, how big is the PV system, and how is the battery connected? Everything uses Year 2 (2023), the year the optimisation is run on, unless stated.

## Task 4.1 - Clock check: where is solar noon?

### Background
Finding 1.2 showed no daylight-saving jumps, which suggests the data runs on AEST all year. Solar noon gives an independent test. Thredbo is at about 148.3°E, 1.7° west of the AEST reference meridian (150°E), so mean solar noon is about 12:07 AEST. The equation of time moves it by up to about ±16 minutes across the year. If the data were on local daylight-saving time, the centre of PV generation would jump by an hour, to around 13:00, between October and April.

`pv_centroid_hour` finds the "centre of mass" of daily generation for each month: the PV-weighted mean hour of day.

In [342]:
def pv_centroid_hour(pv: pd.Series) -> pd.Series:
    """PV-weighted mean hour of day for each month (e.g. 12.5 = 12:30): the centre of daily generation.

    centroid = sum(hour x pv) / sum(pv), grouped by calendar month, using fractional hours (hour + minute / 60).
    """
    hour = pv.index.hour + pv.index.minute / 60.0
    centroid = (pv * hour).groupby(pv.index.month).sum()/ pv.groupby(pv.index.month).sum()

    return centroid


In [343]:
"""Tests for Task 4.1"""
_i = pd.date_range("2023-01-01", "2023-02-28 23:55", freq="5min")
_h = np.asarray(_i.hour + _i.minute / 60)
_pv = pd.Series(np.clip(1 - np.abs(_h - 12) / 6, 0, None), index=_i)                    # symmetric about 12:00
_pv[_i.month == 2] = np.clip(1 - np.abs(_h[_i.month == 2] - 13) / 6, 0, None)          # Feb shifted 1 h (like daylight saving)
_c = pv_centroid_hour(_pv)
assert np.isclose(_c.loc[1], 12.0) and np.isclose(_c.loc[2], 13.0)
print("Task 4.1 tests passed")

Task 4.1 tests passed


In [344]:
centroid = pd.DataFrame({
    "Y1 measured PV":   pv_centroid_hour(dfc.loc[dfc.year == 1, "pv_meas_kW"].dropna()),
    "Y2 commercial PV": pv_centroid_hour(dfc.loc[dfc.year == 2, "pv_comm_kW"]),
})
print("Centre of PV generation, minutes after 12:00 (AEST if the clock is standard time):")
((centroid - 12) * 60).round(0).T

Centre of PV generation, minutes after 12:00 (AEST if the clock is standard time):


timestamp,1,2,3,4,5,6,7,8,9,10,11,12
Y1 measured PV,37.0,40.0,37.0,33.0,34.0,36.0,44.0,43.0,29.0,17.0,14.0,20.0
Y2 commercial PV,26.0,44.0,43.0,31.0,33.0,40.0,43.0,38.0,27.0,17.0,15.0,24.0


> **Finding 4.1 - Clock.** TODO in my words: the centre of generation each month (minutes after 12:00); whether there is a one-hour jump in Oct-Apr (daylight saving) or not; so is the data AEST all year? (Small offsets from 12:07 can come from the equation of time, panel orientation and the interval-end timestamps.)

## Task 4.2 - Daily energy balance: how much could a battery shift?

### Background
Without a battery, each 5-minute interval splits into three flows (kW):
- **self-consumed** = min(load, PV): solar used directly;
- **export** = PV − load when PV exceeds load: surplus solar sent to the grid;
- **import** = load − PV when load exceeds PV: demand met from the grid.

Summed over a day (kW × `dt_h` = kWh), **shiftable = min(export, import)** is the most energy a perfect battery could move from surplus to deficit that day. It is an upper bound for sizing: it ignores losses, power limits and carrying charge to the next day.

In [345]:
def daily_energy_balance(load: pd.Series, pv: pd.Series, dt_h: float = DT_H) -> pd.DataFrame:
    """Daily energy with no battery (kWh): load, pv, self_consumed, export, import, shiftable.

    Per interval (kW): self_consumed = min(load, pv); export = surplus PV; import = unmet load.
    Convert to kWh with dt_h, sum each day, then shiftable = min(export, import) for each day.
    """
    self_consumed = np.minimum(load, pv)
    export = (pv - load).clip(lower = 0)
    imp = (load - pv).clip(lower = 0)

    flows = pd.DataFrame({
        "load": load,
        "pv": pv,
        "self_consumed": self_consumed,
        "export": export,
        "import": imp
    }) * dt_h

    # Resample and sum to get daily totals
    daily = flows.resample("D").sum()
    
    # Shiftable energy is the minimum of daily export and daily import
    daily["shiftable"] = np.minimum(daily["export"], daily["import"])

    return daily

In [346]:
"""Tests for Task 4.2"""
_i = pd.to_datetime(["2023-01-01 10:00", "2023-01-01 11:00", "2023-01-01 12:00", "2023-01-01 13:00",
                     "2023-01-02 10:00", "2023-01-02 11:00", "2023-01-02 12:00", "2023-01-02 13:00"])
_load = pd.Series([1, 1, 3, 2,  2, 2, 2, 2], index=_i, dtype=float)
_pv   = pd.Series([0, 4, 1, 0,  0, 0, 0, 0], index=_i, dtype=float)
_b = daily_energy_balance(_load, _pv, dt_h=1.0)
assert _b.loc["2023-01-01"].to_dict() == dict(load=7, pv=5, self_consumed=2, export=3, **{"import": 5}, shiftable=3)
assert _b.loc["2023-01-02", "shiftable"] == 0                                   # no surplus solar -> nothing to shift
assert np.allclose(_b["load"], _b["self_consumed"] + _b["import"])              # energy balance: load side
assert np.allclose(_b["pv"],   _b["self_consumed"] + _b["export"])              # energy balance: PV side
print("Task 4.2 tests passed")

Task 4.2 tests passed


In [347]:
bal = daily_energy_balance(dfc.loc[dfc.year == 2, "load_kW"], dfc.loc[dfc.year == 2, "pv_kW"])
tot = bal.sum()
print(f"No battery, 2023: self-consumption {tot.self_consumed / tot.pv:.0%} of PV is used on site; "
      f"self-sufficiency {tot.self_consumed / tot.load:.0%} of load is met by PV")
print(f"Shiftable energy per day: median {bal.shiftable.median():.1f} kWh, 90th percentile {bal.shiftable.quantile(0.9):.1f} kWh, max {bal.shiftable.max():.1f} kWh")
monthly = bal.groupby(bal.index.month).mean()
monthly.round(1)

No battery, 2023: self-consumption 54% of PV is used on site; self-sufficiency 65% of load is met by PV
Shiftable energy per day: median 7.4 kWh, 90th percentile 10.7 kWh, max 14.2 kWh


,load,pv,self_consumed,export,import,shiftable
timestamp,,,,,,
1,32.3,48.5,24.0,24.4,8.3,7.7
2,35.1,49.0,26.7,22.2,8.3,7.4
3,28.9,36.4,19.6,16.8,9.3,8.1
4,22.8,26.0,13.2,12.8,9.6,7.3
5,24.2,19.4,10.8,8.6,13.4,7.4
6,22.1,12.6,7.2,5.4,14.9,5.2
7,23.8,18.5,9.8,8.7,14.0,7.4
8,23.9,24.1,13.3,10.8,10.6,8.2
9,34.9,40.5,26.8,13.7,8.1,6.9


In [348]:
# Figure 4.1 - Monthly average daily energy (2023) and what a battery could shift
months = pd.to_datetime(monthly.index.astype(str), format="%m").strftime("%b")
fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.1,
                    subplot_titles=("Average day in each month (kWh/day)", "Shiftable energy per day (all 365 days)"))
for col, name, colour in [("load", "Load", "#444444"), ("pv", "PV generation", "#f2a900"),
                          ("export", "Export (surplus PV)", "#1f77b4"), ("import", "Import (unmet load)", "#d62728")]:
    fig.add_trace(go.Bar(x=months, y=monthly[col], name=name, marker_color=colour), row=1, col=1)
fig.add_trace(go.Scatter(x=months, y=monthly["shiftable"], name="Shiftable (battery upper bound)",
                         mode="lines+markers", line=dict(color="#2ca02c", width=3)), row=1, col=1)
fig.add_trace(go.Histogram(x=bal["shiftable"], nbinsx=30, marker_color="#2ca02c", opacity=0.8, showlegend=False), row=1, col=2)
for q, dash, label, side in [(0.5, "dash", "Median", "top left"), (0.9, "dot", "90th pct", "top right")]:
    v = bal["shiftable"].quantile(q)
    fig.add_vline(x=v, line_dash=dash, line_color="black", row=1, col=2,
                  annotation_text=f"{label}<br>{v:.1f} kWh", annotation_position=side)
fig.update_xaxes(title_text="kWh/day", row=1, col=2)
fig.update_yaxes(title_text="kWh/day", row=1, col=1)
fig.update_yaxes(title_text="Number of days", row=1, col=2)
fig.update_layout(title="Load, PV and shiftable energy, 2023", barmode="group", template="plotly_white", height=460,
                  legend=dict(orientation="h", yanchor="top", y=-0.15, x=0))
fig.show()

> **Finding 4.2 - Energy balance.** TODO in my words: no-battery self-consumption and self-sufficiency; how PV and load swap seasonally (surplus in summer, shortfall in winter); and the key sizing number: shiftable energy per day (median, 90th percentile) and how steady it is across months. What battery size (usable kWh) does that point to for self-consumption, and why might ToU/wholesale arbitrage justify more (tested in Part 8)?

## Task 4.3 - PV system size

The metadata gives the performance ratio (85%) but not the array size. It can be estimated from data already in hand: the median daily PV-to-irradiance ratio (Task 1.6) is the array's effective output in kW per kW/m² of sunlight, which equals size (kWp) × performance ratio. Compare the estimate with the measured peak output (10.31 kW, Finding 1.3).

In [349]:
# Task 4.3 - Estimate PV array size (kWp): median Year 1 daily PV/irradiance ratio (Task 1.6) / performance ratio (Part 0)
PV_KWP_EST = ratio["Y1 measured"].median() / PV_PR
print(f"Estimated PV array size: {PV_KWP_EST:.3f} kWp")

Estimated PV array size: 8.904 kWp


## Task 4.4 - Battery topology: AC- or DC-coupled?

The brief asks for an assumption on how the battery connects, because it decides which conversion losses apply.

| | **AC-coupled** | **DC-coupled (hybrid inverter)** |
|---|---|---|
| How it connects | Battery has its own inverter on the house AC side, next to the existing PV inverter | PV and battery share one hybrid inverter on a DC bus |
| Solar to battery | PV DC -> AC (PV inverter) -> DC (battery inverter): **two** conversions | PV DC -> battery DC: **no** inverter conversion |
| Battery to house | DC -> AC: one conversion | DC -> AC: one conversion |
| Grid to battery (ToU / wholesale arbitrage) | AC -> DC: one conversion | AC -> DC: one conversion |
| Typical use | Retrofit to an existing PV system | New PV + battery installed together |
| Product examples | Separate AC battery systems | Hybrid-inverter battery systems |

Efficiency values for the chosen topology and product are set and sourced in Part 5 (battery model).

**Decision: AC-coupled.** The site already has about 8.9 kWp of PV (Task 4.3), so this is a retrofit. AC coupling leaves the existing inverter and DC wiring untouched, so the installer does not take on responsibility for someone else's work; DC coupling would mean replacing the inverter with a hybrid and rewiring the strings. The cost is one extra conversion on solar charging (DC -> AC -> DC), covered by the round-trip efficiency assumption in Part 5.

In [350]:
TOPOLOGY = "AC"   # Retrofit to an existing ~8.9 kWp PV system: AC coupling leaves the existing inverter and DC wiring
                  # untouched, so the installer does not take on responsibility for someone else's work (DC coupling
                  # would mean replacing the inverter with a hybrid and rewiring the strings). Cost: PV that is stored
                  # is converted DC->AC->DC->AC, covered by the 0.90 round-trip assumption (Part 5).

> **Finding 4.3-4.4 - System.** TODO in my words: estimated PV size and how it compares with the measured peak; the chosen topology and the reason.

# Part 5 - Battery Model

Part 5 turns a real product's datasheet into the battery model every later Part uses: how much energy it holds, how fast it charges and discharges, what is lost on the way in and out, what limits keep it healthy, and what each kWh of cycling costs in wear. The topology is **AC-coupled** (Task 4.4): the battery is retrofitted next to the existing PV inverter, so efficiencies are AC-to-AC.

## Task 5.1 - Choose the product

Sizing evidence from Task 4.2: shiftable energy is about **7.4 kWh/day (median)** and **10.7 kWh/day (90th percentile)**, fairly steady across the year. ToU and wholesale arbitrage may justify more (tested in Part 8).

Two current products that can be AC-coupled to an existing PV system were shortlisted. (A standard Powerwall 3 was ruled out: it can only AC-couple to existing PV inverters of 5 kW or less per unit, and this site's array is about 9 kWp.)

| | **Tesla Powerwall 3AC** | **Sigenergy SigenStor** (AC-coupled, single phase) |
|---|---|---|
| Usable energy | 13.5 kWh | 7.8 kWh per 8 kWh module (2 modules = 15.6 kWh) |
| Continuous power | 5 kW charge; 5 kW output | 4 kW per module; system limited by the 5 kW energy controller (SP 16: 5 kW) |
| Efficiency | 89% battery-to-home/grid (25 °C, beginning of life, 3.3 kW) | Not published AC-to-AC. Energy controller max 97.6%; AC coupling adds one conversion. **Assumption needed** |
| Chemistry | not stated on the 3AC datasheet | LFP, cycle life up to 10,000 |
| Operating temperature | -20 to 50 °C (may derate above 40 °C) | -20 to 55 °C (battery) |
| Warranty | 10 years (retention and throughput not stated) | 10 years / 70%; minimum throughput 23.77 MWh per 8 kWh module |
| Indicative installed price (before rebate) | about $13,360 for 13.5 kWh (2026 average) | about $1,000 per usable kWh (16 kWh about $14,000-17,000) |

**Points to weigh:** 13.5 kWh (Tesla) or 15.6 kWh (two Sigen modules) both exceed the 10.7 kWh 90th percentile, so the extra room is only useful for ToU/wholesale arbitrage; one Sigen module (7.8 kWh) matches the median. Sigenergy's throughput warranty makes the degradation cost in Task 5.3 directly calculable (23.77 MWh / 7.8 kWh = about 3,050 equivalent full cycles to 70%); Tesla's 3AC sheet gives no throughput, so its wear cost would need an assumption. Prices are 2025-26 installed prices (the battery would be bought now, not in 2023); the federal Cheaper Home Batteries rebate (from July 2025) is a sensitivity for Part 8.

**Decision: Sigenergy SigenStor, 2 x 8 kWh modules (15.6 kWh usable, 5 kW).** It can AC-couple to the existing ~9 kWp array (a standard Powerwall 3 cannot), and its published throughput warranty makes the wear cost calculable (Task 5.3). Two modules cover the 90th-percentile shiftable energy (10.7 kWh/day) with room for ToU and wholesale arbitrage; one and three modules are compared in Part 8 (Finding 8).

Sources: Tesla Powerwall 3AC datasheet (AU, 2026); SolarQuotes Powerwall 3 review (AC-coupling limit); Sigenergy SigenStor and Sigen Battery datasheets; SigenStor factory warranty (throughput); Gridly / Solar Choice / Aussie Solar Tech price surveys. Full references in the report.

In [351]:
@dataclass(frozen=True)
class BatterySpec:
    """Everything the battery model needs, in one place. Energy in kWh, power in kW, fractions 0-1."""
    name: str
    usable_kWh: float            # usable energy at beginning of life
    p_charge_kW: float           # max continuous charge power (AC side)
    p_discharge_kW: float        # max continuous discharge power (AC side)
    rte: float                   # round-trip efficiency, AC to AC
    soc_min: float = 0.0         # operating window as a fraction of usable energy
    soc_max: float = 1.0
    soc_init: float = 0.5        # state of charge at the start of a simulation
    temp_opt_low: float = 0.0    # datasheet optimum temperature range (degC); full power inside it
    temp_opt_high: float = 30.0
    power_frac_outside: float = 1.0   # fraction of max power allowed outside the optimum range (assumption)
    warranty_cycles: float = np.nan   # full equivalent cycles to end of warranty
    eol_capacity: float = np.nan      # capacity fraction remaining at end of warranty
    capex_aud: float = np.nan         # installed cost (AUD), for degradation cost and payback

    @property
    def eta_ch(self) -> float:   # one-way efficiency: split the round trip evenly between charge and discharge
        return float(np.sqrt(self.rte))

    @property
    def eta_dis(self) -> float:
        return float(np.sqrt(self.rte))

    @property
    def e_min(self) -> float:
        return self.soc_min * self.usable_kWh

    @property
    def e_max(self) -> float:
        return self.soc_max * self.usable_kWh

In [352]:
"""Tests for BatterySpec"""
_b = BatterySpec("test", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.81, soc_min=0.1, soc_max=0.9)
assert np.isclose(_b.eta_ch, 0.9) and np.isclose(_b.eta_ch * _b.eta_dis, 0.81)   # one-way efficiencies multiply to the round trip
assert np.isclose(_b.e_min, 1.0) and np.isclose(_b.e_max, 9.0)
print("BatterySpec tests passed")

BatterySpec tests passed


In [353]:
# Task 5.1 - Battery spec. Assumptions marked (A) to confirm before final results.
SIGEN = BatterySpec(
    name="Sigenergy SigenStor 2 x 8 kWh", usable_kWh=15.6, p_charge_kW=5, p_discharge_kW=5,
    rte=0.90,                              # (A) no AC-to-AC figure published; in line with similar AC batteries
    power_frac_outside=0.5,                # (A) half power below 0 C
    warranty_cycles=47540 / 15.6, eol_capacity=0.70,   # throughput warranty 23.77 MWh per module, to 70%
    capex_aud=15600,                       # about $1,000 per usable kWh, before rebate
)
BATTERY = SIGEN
BATTERY

BatterySpec(name='Sigenergy SigenStor 2 x 8 kWh', usable_kWh=15.6, p_charge_kW=5, p_discharge_kW=5, rte=0.9, soc_min=0.0, soc_max=1.0, soc_init=0.5, temp_opt_low=0.0, temp_opt_high=30.0, power_frac_outside=0.5, warranty_cycles=3047.4358974358975, eol_capacity=0.7, capex_aud=15600)

## Task 5.2 - Operating limits: state of charge and temperature

### Background
**State of charge window.** The datasheet's usable energy is already the safe range set by the manufacturer. A tighter window (e.g. keeping 10-20% in reserve) can be chosen for backup or longevity, at the cost of less energy to shift.

**Temperature.** Thredbo ranges from about -10 to 26 °C (Finding 1.3), so the battery spends many winter hours below the datasheets' optimum range of 0-30 °C. Neither datasheet publishes a derating curve, so this is a stated assumption: inside the optimum range the battery runs at full power; outside it, power is limited to a fraction `power_frac_outside`. This reduces wear in cold conditions, as the brief asks. The air temperature from the site data is used as a conservative stand-in for the battery's own temperature (an indoor or garage install would run warmer).

`power_limit_factor` returns the fraction of rated power allowed in each interval.

In [354]:
def power_limit_factor(temp_C: pd.Series, spec: BatterySpec) -> pd.Series:
    """Fraction of rated charge/discharge power allowed at each temperature.

    1.0 inside [spec.temp_opt_low, spec.temp_opt_high]; spec.power_frac_outside outside it.
    """
    out = pd.Series(np.where((temp_C >= spec.temp_opt_low) & (temp_C <= spec.temp_opt_high), 1.0, spec.power_frac_outside), index=temp_C.index)
    return out

In [355]:
"""Tests for Task 5.2"""
_s = BatterySpec("t", 10, 5, 5, 0.9, temp_opt_low=0, temp_opt_high=30, power_frac_outside=0.5)
_f = power_limit_factor(pd.Series([-5.0, 0.0, 15.0, 30.0, 35.0]), _s)
assert _f.tolist() == [0.5, 1.0, 1.0, 1.0, 0.5]                     # the boundaries count as inside
assert isinstance(_f, pd.Series)
print("Task 5.2 tests passed")

Task 5.2 tests passed


In [356]:
# How often would the limit bite at this site? (Year 2, using my BATTERY settings)
_factor = power_limit_factor(dfc.loc[dfc.year == 2, "temp_C"], BATTERY)
print(f"Share of 2023 intervals outside the optimum range: {(_factor < 1).mean():.1%}")
(_factor < 1).groupby(_factor.index.month).mean().mul(100).round(0).rename("% of intervals derated, by month")

Share of 2023 intervals outside the optimum range: 8.9%


timestamp
1      0.0
2      0.0
3      0.0
4      1.0
5     19.0
6     27.0
7     26.0
8     17.0
9     12.0
10     3.0
11     0.0
12     0.0
Name: % of intervals derated, by month, dtype: float64

## Task 5.3 - Degradation cost

### Background
Every kWh cycled through the battery uses up part of its life. Putting a price on that wear stops the optimiser cycling the battery for tiny gains (the brief asks for this). A standard estimate spreads the installed cost over all the energy the battery will discharge before end of warranty:

**degradation cost ($/kWh discharged) = capex / (warranty cycles × usable kWh × average capacity over life)**

Capacity fades roughly linearly from 100% to the end-of-warranty value, so the average capacity is (1 + eol_capacity) / 2. The cost is applied to each kWh discharged in the optimiser (Part 7).

In [357]:
def degradation_cost_per_kwh(spec: BatterySpec) -> float:
    """Wear cost ($ per kWh discharged) = capex / (warranty_cycles x usable_kWh x average capacity over life)."""
    avg_capacity = (1.0 + spec.eol_capacity) / 2.0
    deg_cost = spec.capex_aud / (spec.warranty_cycles * spec.usable_kWh * avg_capacity)

    return deg_cost

In [358]:
"""Tests for Task 5.3"""
_s = BatterySpec("t", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.9,
                 warranty_cycles=6000, eol_capacity=0.6, capex_aud=12000)
assert np.isclose(degradation_cost_per_kwh(_s), 12000 / (6000 * 10 * 0.8))   # = $0.25/kWh
print("Task 5.3 tests passed")

Task 5.3 tests passed


**Values in `BATTERY`:** `warranty_cycles` = 23.77 MWh / 7.8 kWh per module, about 3,050 equivalent full cycles, and `eol_capacity` = 0.70, both from the SigenStor factory warranty (70% of usable energy at 10 years or the minimum throughput, whichever comes first); `capex_aud` = $15,600, about $1,000 per usable kWh installed before rebate (2025-26 price surveys, Task 5.1; the rebate is a sensitivity in Part 8). Then compare the degradation cost with the price spreads from Task 3.5: a shift is only worth doing if the spread beats the wear cost plus the round-trip loss.

In [359]:
print(f"{BATTERY.name}: degradation cost {degradation_cost_per_kwh(BATTERY):.3f} $/kWh discharged")

Sigenergy SigenStor 2 x 8 kWh: degradation cost 0.386 $/kWh discharged


## Task 5.4 - State-of-charge simulator

### Background
Given a battery power schedule (kW, **positive = discharging, negative = charging**, as in Part 0), the stored energy evolves as:
- charging at p kW for dt_h hours adds **eta_ch × p × dt_h** kWh (some energy is lost going in);
- discharging at p kW removes **p / eta_dis × dt_h** kWh (more is drawn from storage than is delivered).

`simulate_soc` runs this forward. It is used to check that any schedule (rule-based in Part 6, optimised in Parts 7 and 11) respects the battery's limits, independently of the optimiser that produced it.

In [360]:
def simulate_soc(p_batt_kW: pd.Series, spec: BatterySpec, dt_h: float = DT_H, e0_kWh: float = None) -> pd.Series:
    """Stored energy (kWh) at the END of each interval for a battery power schedule.

    p_batt_kW: positive = discharging, negative = charging. e0_kWh defaults to spec.soc_init x usable_kWh.
    Does not clip: if the schedule breaks a limit, the result shows it (checked by the caller).
    """
    e0 = spec.soc_init * spec.usable_kWh if e0_kWh is None else e0_kWh

    # Charging (p < 0) adds eta_ch * |p| * dt_h; discharging (p > 0) removes (p / eta_dis) * dt_h
    delta_e = pd.Series(
        np.where(
            p_batt_kW < 0,
            spec.eta_ch * (-p_batt_kW) * dt_h,
            -(p_batt_kW / spec.eta_dis) * dt_h,
        ),
        index=p_batt_kW.index,
    )

    return e0 + delta_e.cumsum()

In [361]:
"""Tests for Task 5.4"""
_s = BatterySpec("t", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.81)        # eta = 0.9 each way
_p = pd.Series([-2.0, -2.0, 0.0, 3.0])                                                   # charge, charge, idle, discharge
_e = simulate_soc(_p, _s, dt_h=1.0, e0_kWh=5.0)
assert np.allclose(_e, [5 + 1.8, 5 + 3.6, 5 + 3.6, 5 + 3.6 - 3 / 0.9])
# a full round trip loses exactly (1 - rte) of the energy charged from the grid
_e2 = simulate_soc(pd.Series([-1.0, 0.81]), _s, dt_h=1.0, e0_kWh=0.0)
assert np.isclose(_e2.iloc[-1], 0.0)                                                     # 1 kWh in -> 0.81 kWh out
print("Task 5.4 tests passed")

Task 5.4 tests passed


> **Finding 5.1-5.4 - Battery model.** TODO in my words: the chosen product and size (why it fits the 7.4 / 10.7 kWh/day evidence); efficiency and SOC window; the temperature assumption and how often it bites at Thredbo; the degradation cost ($/kWh) and what spread a shift needs to be worth it.

# Part 6 - Benchmarks: No Battery and Simple Self-Consumption

Before optimising, two benchmarks set the bar the optimiser has to beat, on all three tariffs (Year 2, 2023):
1. **No battery**: the household's bill with solar only.
2. **Simple self-consumption battery**: the default mode the brief describes. The battery charges only from surplus solar and discharges only to cover the household's own load, with no prices and no forecasts. This is how most home batteries run out of the box.

Both use the same billing function and the same metrics, so every later result is directly comparable.

## Task 6.1 - Billing

### Background
Grid power follows the Part 0 convention: **positive = import, negative = export**. Each interval's import is billed at that interval's import price and each export credited at its export price (both $/kWh, from `prices`, Task 3.4). Any fixed monthly fee is added on top. The VPP fee ($25/month) is only charged when VPP-SPREE actually optimises the battery, so it is **not** applied to the two benchmarks.

In [362]:
def annual_bill(grid_kW: pd.Series, imp_price: pd.Series, exp_price: pd.Series,
                dt_h: float = DT_H, fee_per_month: float = 0.0) -> float:
    """Total bill ($) for a grid power series (kW; + import, - export) over the period covered.

    bill = sum(import kWh x import price) - sum(export kWh x export price) + fee_per_month x number of months
    """
    imp_kwh = grid_kW.clip(lower=0.0) * dt_h
    exp_kwh = (-grid_kW).clip(lower=0.0) * dt_h

    energy_cost = (imp_kwh * imp_price).sum()
    export_credit = (exp_kwh * exp_price).sum()
    n_months = grid_kW.index.to_period("M").nunique()

    out = float(energy_cost - export_credit + fee_per_month * n_months)

    return out

In [363]:
"""Tests for Task 6.1"""
_i = pd.date_range("2023-01-31 23:00", periods=3, freq="h")          # spans January and February
_g = pd.Series([2.0, -1.0, 0.5], index=_i)                            # import 2 kWh, export 1 kWh, import 0.5 kWh
_imp = pd.Series([0.30, 0.30, 0.40], index=_i)
_exp = pd.Series([0.05, 0.05, 0.05], index=_i)
assert np.isclose(annual_bill(_g, _imp, _exp, dt_h=1.0), 2 * 0.30 - 1 * 0.05 + 0.5 * 0.40)      # = 0.75
assert np.isclose(annual_bill(_g, _imp, _exp, dt_h=1.0, fee_per_month=25), 0.75 + 2 * 25)      # 2 calendar months
_neg = pd.Series([-1.0, -1.0, -1.0], index=_i)
assert np.isclose(annual_bill(_neg, _imp, -_exp, dt_h=1.0), 3 * 0.05)                          # exporting at a negative price costs money
print("Task 6.1 tests passed")

Task 6.1 tests passed


In [364]:
def site_metrics(load: pd.Series, pv_used: pd.Series, grid_kW: pd.Series, batt_kW: pd.Series, dt_h: float = DT_H) -> dict:
    """Energy metrics (kWh unless stated) for one operating mode.

    pv_used: PV after any curtailment. batt_kW: + discharging, - charging.
    Self-consumption = share of PV used on site (not exported); self-sufficiency = share of load not imported.
    """
    imp = grid_kW.clip(lower=0).sum() * dt_h
    exp = (-grid_kW).clip(lower=0).sum() * dt_h
    pv = pv_used.sum() * dt_h
    discharged = batt_kW.clip(lower=0).sum() * dt_h
    return dict(import_kWh=imp, export_kWh=exp, pv_kWh=pv,
                self_consumption=1 - exp / pv if pv > 0 else np.nan,
                self_sufficiency=1 - imp / (load.sum() * dt_h),
                discharged_kWh=discharged)

## Task 6.2 - Benchmark 1: no battery

In [365]:
y2 = dfc.year == 2
load2, pv2, temp2 = dfc.loc[y2, "load_kW"], dfc.loc[y2, "pv_kW"], dfc.loc[y2, "temp_C"]
zero = pd.Series(0.0, index=load2.index)

grid_nobatt = load2 - pv2                                   # + import, - export
results = {}                                                # (mode, tariff) -> dict of results; filled through Parts 6-8
for t in TARIFFS:
    results[("No battery", t)] = dict(bill=annual_bill(grid_nobatt, prices[f"{t}_imp"], prices[f"{t}_exp"]),
                                      **site_metrics(load2, pv2, grid_nobatt, zero))
pd.DataFrame(results).T[["bill", "import_kWh", "export_kWh", "self_consumption", "self_sufficiency"]].round(2)

bill  import_kWh  export_kWh  self_consumption  self_sufficiency
No battery flat       821.87     3718.45     5873.23              0.54              0.65
           tou        605.60     3718.45     5873.23              0.54              0.65
           wholesale  648.51     3718.45     5873.23              0.54              0.65

## Task 6.3 - Benchmark 2: simple self-consumption battery

### Background
The rule, every 5-minute interval, with no knowledge of prices or the future:
- **Surplus solar** (PV > load): charge with the surplus, limited by the charge power limit (rated power x temperature factor, Task 5.2) and by the room left in the battery. Whatever cannot be stored is exported.
- **Deficit** (load > PV): discharge to cover it, limited by the discharge power limit and the energy left above the minimum. Whatever cannot be covered is imported.

Energy limits in power terms, for an interval of dt_h hours with stored energy e:
- most it can **charge**: (e_max - e) / (eta_ch x dt_h) kW
- most it can **discharge**: (e - e_min) x eta_dis / dt_h kW

`self_consumption_dispatch` steps through the year interval by interval (a plain loop, because each interval depends on the stored energy left by the previous one) and returns the battery power (+ discharging, - charging). The schedule is then checked independently with `simulate_soc` (Task 5.4).

In [366]:
def self_consumption_dispatch(load: pd.Series, pv: pd.Series, temp_C: pd.Series, spec: BatterySpec,
                              dt_h: float = DT_H, e0_kWh: float = None) -> pd.Series:
    """Rule-based battery power (kW; + discharging, - charging): store surplus solar, cover deficits, nothing else."""
    factor = power_limit_factor(temp_C, spec).to_numpy()
    net = (load - pv).to_numpy()                       # + deficit, - surplus
    e = spec.soc_init * spec.usable_kWh if e0_kWh is None else e0_kWh
    e_min = getattr(spec, "soc_min", 0.0) * spec.usable_kWh
    e_max = spec.usable_kWh
    p = np.zeros(len(net))

    for k in range(len(net)):
        if net[k] < 0:
            surplus_kW = -net[k]
            room_kWh = max(0.0, e_max - e)
            charge_limit_kW = spec.p_charge_kW * factor[k]
            max_energy_charge_kW = room_kWh / (spec.eta_ch * dt_h) if spec.eta_ch > 0 else 0.0
            charge_kW = min(surplus_kW, charge_limit_kW, max_energy_charge_kW)
            p[k] = -charge_kW
            e += spec.eta_ch * charge_kW * dt_h
        elif net[k] > 0:
            deficit_kW = net[k]
            avail_kWh = max(0.0, e - e_min)
            discharge_limit_kW = spec.p_discharge_kW * factor[k]
            max_energy_discharge_kW = avail_kWh * spec.eta_dis / dt_h
            discharge_kW = min(deficit_kW, discharge_limit_kW, max_energy_discharge_kW)
            p[k] = discharge_kW
            e -= (discharge_kW / spec.eta_dis) * dt_h
            
    return pd.Series(p, index=load.index)

In [367]:
"""Tests for Task 6.3"""
_s = BatterySpec("t", usable_kWh=10, p_charge_kW=3, p_discharge_kW=3, rte=0.81)       # eta 0.9 each way
_i = pd.date_range("2023-01-02", periods=6, freq="h")
_load = pd.Series([1, 1, 1, 5, 5, 5], index=_i, dtype=float)
_pv   = pd.Series([6, 6, 1, 0, 0, 0], index=_i, dtype=float)
_t    = pd.Series(20.0, index=_i)
_p = self_consumption_dispatch(_load, _pv, _t, _s, dt_h=1.0, e0_kWh=5.0)
# surplus 5 kW, capped at 3 kW; then only 1/(0.9) kW of room; idle; then deficit 5 capped at 3; then empty
assert np.allclose(_p.iloc[:3], [-3.0, -(10 - 5 - 2.7) / 0.9, 0.0])
assert np.isclose(_p.iloc[3], 3.0) and _p.iloc[4] > 0
_e = simulate_soc(_p, _s, dt_h=1.0, e0_kWh=5.0)
assert _e.min() >= -1e-9 and _e.max() <= 10 + 1e-9                                       # never outside the battery's limits
assert np.isclose(_e.iloc[-1], 0.0)                                                      # fully used by the end
# never charges from the grid or discharges into the grid
assert ((_p < 0) <= (_pv > _load)).all() and ((_p > 0) <= (_load > _pv)).all()
# cold: half power below 0 C
_cold = self_consumption_dispatch(_load, _pv, pd.Series(-5.0, index=_i), replace(_s, power_frac_outside=0.5), dt_h=1.0, e0_kWh=5.0)
assert np.isclose(_cold.iloc[0], -1.5)
print("Task 6.3 tests passed")

Task 6.3 tests passed


In [368]:
batt_sc = self_consumption_dispatch(load2, pv2, temp2, BATTERY)
e_sc = simulate_soc(batt_sc, BATTERY)
assert e_sc.min() >= BATTERY.e_min - 1e-6 and e_sc.max() <= BATTERY.e_max + 1e-6     # independent check of the schedule
grid_sc = load2 - pv2 - batt_sc
cycles = batt_sc.clip(lower=0).sum() * DT_H / BATTERY.usable_kWh
print(f"Self-consumption battery: {cycles:.0f} equivalent full cycles in 2023 ({cycles / 365:.2f} per day)")
for t in TARIFFS:
    results[("Self-consumption", t)] = dict(bill=annual_bill(grid_sc, prices[f"{t}_imp"], prices[f"{t}_exp"]),
                                            **site_metrics(load2, pv2, grid_sc, batt_sc))

Self-consumption battery: 175 equivalent full cycles in 2023 (0.48 per day)


## Task 6.4 - Benchmark results

In [369]:
def results_table(results: dict, baseline: str = "No battery") -> pd.DataFrame:
    """One row per (mode, tariff): bill, savings vs the baseline on the same tariff, energy metrics."""
    tab = pd.DataFrame(results).T
    tab.index.names = ["mode", "tariff"]
    base = tab.xs(baseline, level="mode")["bill"]
    tab["saving"] = base.reindex(tab.index.get_level_values("tariff")).to_numpy() - tab["bill"]
    tab["cycles"] = tab["discharged_kWh"] / BATTERY.usable_kWh
    cols = ["bill", "saving", "import_kWh", "export_kWh", "self_consumption", "self_sufficiency", "cycles"]
    return tab[cols]

bench = results_table(results)
bench.style.format({"bill": "${:,.0f}", "saving": "${:,.0f}", "import_kWh": "{:,.0f}", "export_kWh": "{:,.0f}",
                    "self_consumption": "{:.0%}", "self_sufficiency": "{:.0%}", "cycles": "{:.0f}"})

In [370]:
# Figure 6.1 - Annual bill by tariff: no battery vs simple self-consumption battery (2023)
names = {"flat": "Flat", "tou": "Time of use", "wholesale": "Wholesale pass-through"}
fig = go.Figure()
for mode, colour in [("No battery", "#7f7f7f"), ("Self-consumption", "#2ca02c")]:
    b = bench.xs(mode, level="mode")["bill"]
    fig.add_trace(go.Bar(x=[names[t] for t in b.index], y=b.values, name=mode, marker_color=colour,
                         text=[f"${v:,.0f}" for v in b.values], textposition="outside"))
fig.update_layout(title="Annual electricity bill, 2023: no battery vs simple self-consumption battery",
                  yaxis_title="Annual bill (AUD)", barmode="group", template="plotly_white", height=430,
                  legend=dict(orientation="h", yanchor="top", y=-0.12, x=0))
fig.show()

> **Finding 6 - Benchmarks.** TODO in my words: the annual bill on each tariff with no battery (which tariff is cheapest for this household, and why, linking to Figure 3.1); what the simple self-consumption battery saves on each tariff, its self-consumption and self-sufficiency, and how many cycles it does; why its saving differs between tariffs even though its behaviour is identical (it ignores prices); and the simple payback at this saving vs the ~$15,600 capex (a first look; Part 8 does it properly).

# Part 7 - Perfect-Foresight Optimiser

The optimiser schedules the battery to minimise what the household pays over the year on each tariff, assuming load, PV and prices are known in advance (perfect foresight, as the brief allows for the first stage). It is the upper bound on what any real controller can achieve; Parts 9-12 then measure how much is lost when the future has to be forecast.

## Task 7.1 - Formulation

A **linear program** (LP), solved with `scipy.optimize.linprog` (HiGHS). For every 5-minute interval t there are six decision variables (kW unless stated):

| Variable | Meaning | Bounds |
|---|---|---|
| c_t | battery charging (AC side) | 0 to p_charge x temperature factor |
| d_t | battery discharging (AC side) | 0 to p_discharge x temperature factor |
| i_t | grid import | >= 0 |
| x_t | grid export | >= 0 |
| u_t | PV curtailed | 0 to PV_t |
| e_t | stored energy at the end of the interval (kWh) | e_min to e_max |

**Constraints**
- Power balance: load_t = (PV_t - u_t) + d_t - c_t + i_t - x_t
- Energy: e_t = e_(t-1) + eta_ch x c_t x dt - d_t / eta_dis x dt

**Objective** (minimise, $):  sum over t of [ import price_t x i_t - export price_t x x_t + degradation cost x d_t ] x dt

Everything the brief asks for falls out of this: the battery charges and discharges only when the price difference beats the round-trip loss plus the wear cost; curtailment is chosen only when exporting costs money (negative export price), exactly the brief's rule; temperature limits and the SOC window are bounds.

**Two technical details.** (1) The year is solved one week at a time (2,016 intervals per LP), passing the stored energy from one week to the next; each week must end at least at the starting state of charge, so a week cannot "borrow" energy it never repays. Tested against monthly blocks: the annual bill differs by under 2% and the solve is about 3x faster (daily blocks are faster again but lose about 15% of the value, because they cannot plan across days). (2) In a handful of extreme negative-price intervals, the wholesale import price falls *below* the export price. Physically the meter nets import and export, so the LP uses max(import price, export price) as the import price there, which keeps it from importing and exporting at once; bills are always settled on the real prices.

**Decisions: all three switches on**, with reasons in the cell below and in Finding 7.
- `ALLOW_GRID_CHARGING`: may the battery charge from the grid (needed for ToU and wholesale arbitrage)?
- `ALLOW_BATTERY_EXPORT`: may the battery discharge into the grid when the export price is high (the core of a VPP)?
- `USE_DEGRADATION_COST`: include the wear cost in the objective? (Task 5.3 found ~$0.39/kWh, as large as the price spreads, so this matters a lot.)

In [371]:
ALLOW_GRID_CHARGING  = True    # VPP-SPREE's offer lets the battery charge from the grid; the LP only does it when cheap
                               # (low or negative midday prices) beats the wear cost, so allowing it can only help
                               
ALLOW_BATTERY_EXPORT = True    # wholesale value is earned by exporting into price spikes: 84% of the optimiser's saving
                               # comes from the top 1% of intervals (Task 13.2); without export it can only offset load

USE_DEGRADATION_COST = True    # without a wear cost the LP cycles for cents: ~380 cycles/yr on wholesale at $0/kWh
                               # (Task 13.1), using up the ~3,050-cycle warranty in about 8 years

In [372]:
def optimise_dispatch(load, pv, imp, exp, factor, spec: BatterySpec, dt_h: float = DT_H, e0: float = None,
                      e_end_min: float = None, deg_cost: float = 0.0,
                      allow_grid_charging: bool = True, allow_battery_export: bool = True,
                      terminal_value: float = 0.0) -> pd.DataFrame:
    """Cost-minimising battery schedule for one block of intervals (LP, perfect foresight).

    terminal_value ($/kWh, default 0 = no credit): value of energy left at the end of the block (Task 13.4).

    All inputs are aligned arrays/Series of equal length (kW, $/kWh, power-limit factor 0-1).
    Returns a DataFrame (kW, and kWh for energy) with columns:
    charge_kW, discharge_kW, import_kW, export_kW, curtail_kW, energy_kWh.
    """
    idx = load.index if hasattr(load, "index") else None
    L, P, pi, pe, f = (np.asarray(a, float) for a in (load, pv, imp, exp, factor))
    n = len(L)
    e0 = spec.soc_init * spec.usable_kWh if e0 is None else e0
    pi_lp = np.maximum(pi, pe)                       # keep the LP convex (see Task 7.1)
    C, D, I, X, U, E = (np.arange(n) + k * n for k in range(6))
    nv = 6 * n

    cost = np.zeros(nv)
    cost[I] = pi_lp * dt_h
    cost[X] = -pe * dt_h
    cost[D] = deg_cost * dt_h
    cost[E[-1]] -= terminal_value                    # credit for energy left at the end (0 = Part 7 behaviour)

    rows, cols, vals, rhs = [], [], [], []
    r = np.arange(n)
    # power balance: -c + d + i - x - u = load - pv
    for var, coef in ((C, -1.0), (D, 1.0), (I, 1.0), (X, -1.0), (U, -1.0)):
        rows.append(r); cols.append(var); vals.append(np.full(n, coef))
    rhs.append(L - P)
    # energy: e_t - e_(t-1) - eta_ch dt c_t + dt/eta_dis d_t = 0   (e_(-1) = e0 moves to the right-hand side)
    r2 = r + n
    rows += [r2, r2, r2]; cols += [E, C, D]
    vals += [np.ones(n), np.full(n, -spec.eta_ch * dt_h), np.full(n, dt_h / spec.eta_dis)]
    rows.append(r2[1:]); cols.append(E[:-1]); vals.append(-np.ones(n - 1))
    b_energy = np.zeros(n); b_energy[0] = e0
    rhs.append(b_energy)
    A = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(2 * n, nv))
    b = np.concatenate(rhs)

    surplus, deficit = np.clip(P - L, 0, None), np.clip(L - P, 0, None)
    c_max = spec.p_charge_kW * f if allow_grid_charging else np.minimum(spec.p_charge_kW * f, surplus)
    d_max = spec.p_discharge_kW * f if allow_battery_export else np.minimum(spec.p_discharge_kW * f, deficit)
    e_lo = np.full(n, spec.e_min)
    if e_end_min is not None:
        e_lo[-1] = max(spec.e_min, min(e_end_min, spec.e_max))
    bounds = np.column_stack([
        np.concatenate([np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n), e_lo]),
        np.concatenate([c_max, d_max, np.full(n, np.inf), np.full(n, np.inf), P, np.full(n, spec.e_max)]),
    ])
    res = linprog(cost, A_eq=A, b_eq=b, bounds=bounds, method="highs")
    if res.status != 0:
        raise RuntimeError(f"LP failed: {res.message}")
    x = res.x
    return pd.DataFrame({"charge_kW": x[C], "discharge_kW": x[D], "import_kW": x[I],
                         "export_kW": x[X], "curtail_kW": x[U], "energy_kWh": x[E]}, index=idx)


def optimise_year(load, pv, imp, exp, temp_C, spec: BatterySpec, dt_h: float = DT_H, deg_cost: float = 0.0,
                  block: str = "W-MON", **switches) -> pd.DataFrame:
    """Solve block by block (default: weeks ending Monday), carrying stored energy forward.

    Each block must end at least at spec.soc_init so no block borrows energy it does not repay.
    """
    factor = power_limit_factor(temp_C, spec)
    e = spec.soc_init * spec.usable_kWh
    parts = []
    for _, blk in load.groupby(pd.Grouper(freq=block)):
        ix = blk.index
        out = optimise_dispatch(load[ix], pv[ix], imp[ix], exp[ix], factor[ix], spec, dt_h, e0=e,
                                e_end_min=spec.soc_init * spec.usable_kWh, deg_cost=deg_cost, **switches)
        parts.append(out)
        e = out["energy_kWh"].iloc[-1]
    return pd.concat(parts)

In [373]:
"""Tests for Task 7.1 (small cases with known answers)"""
_s = BatterySpec("t", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.81)        # eta 0.9 each way
_i = pd.date_range("2023-01-02", periods=4, freq="h")
_one = pd.Series(1.0, index=_i)
_load = pd.Series([0, 0, 2, 2], index=_i, dtype=float)
_pv0 = pd.Series(0.0, index=_i)

# 1) cheap then expensive: charge early, discharge to cover the expensive load
_imp = pd.Series([0.10, 0.10, 0.50, 0.50], index=_i); _exp = pd.Series(0.0, index=_i)
_o = optimise_dispatch(_load, _pv0, _imp, _exp, _one, _s, dt_h=1.0, e0=0.0)
assert _o["charge_kW"].iloc[:2].sum() > 0 and np.allclose(_o["import_kW"].iloc[2:], 0, atol=1e-6)
# energy balance and stored-energy check against the independent simulator (Task 5.4)
_bal = _pv0 - _o.curtail_kW + _o.discharge_kW - _o.charge_kW + _o.import_kW - _o.export_kW
assert np.allclose(_bal, _load, atol=1e-6)
assert np.allclose(simulate_soc(_o.discharge_kW - _o.charge_kW, _s, dt_h=1.0, e0_kWh=0.0), _o.energy_kWh, atol=1e-6)

# 2) a spread smaller than the losses: not worth cycling (0.81 x 0.50 < 0.45)
_o2 = optimise_dispatch(_load, _pv0, pd.Series([0.45, 0.45, 0.50, 0.50], index=_i), _exp, _one, _s, dt_h=1.0, e0=0.0)
assert np.allclose(_o2["charge_kW"], 0, atol=1e-6)

# 3) a degradation cost larger than the spread also stops cycling
_o3 = optimise_dispatch(_load, _pv0, _imp, _exp, _one, _s, dt_h=1.0, e0=0.0, deg_cost=0.50)
assert np.allclose(_o3["charge_kW"], 0, atol=1e-6)

# 4) negative export price: curtail rather than export (the brief's rule), and only then
_pv = pd.Series([4, 4, 0, 0], index=_i, dtype=float)
_full = replace(_s, usable_kWh=0.001)                                                     # (almost) no battery
_o4 = optimise_dispatch(_load, _pv, _imp, pd.Series([-0.05, 0.05, 0, 0], index=_i), _one, _full, dt_h=1.0, e0=0.0)
assert np.isclose(_o4["curtail_kW"].iloc[0], 4, atol=0.01) and np.isclose(_o4["curtail_kW"].iloc[1], 0, atol=1e-6)

# 5) switches: no grid charging means charge only from surplus PV; no battery export means discharge only to cover load
_o5 = optimise_dispatch(_load, _pv0, _imp, _exp, _one, _s, dt_h=1.0, e0=0.0, allow_grid_charging=False)
assert np.allclose(_o5["charge_kW"], 0, atol=1e-6)
_o6 = optimise_dispatch(_load, _pv0, _imp, pd.Series([0, 0, 0, 0.9], index=_i), _one, _s, dt_h=1.0, e0=5.0,
                        allow_battery_export=False)
assert (_o6["discharge_kW"] <= _load + 1e-6).all()

# 6) blocks carry energy forward and end at or above the starting state of charge
_iy = pd.date_range("2023-01-30 20:00", periods=8, freq="h")                             # Monday night into Tuesday: a week boundary
_oy = optimise_year(pd.Series(1.0, index=_iy), pd.Series(0.0, index=_iy), pd.Series(0.3, index=_iy),
                    pd.Series(0.0, index=_iy), pd.Series(20.0, index=_iy), _s, dt_h=1.0)
assert _oy.loc["2023-01-30", "energy_kWh"].iloc[-1] >= 5 - 1e-6
print("Task 7.1 tests passed")

Task 7.1 tests passed


## Task 7.2 - Check every schedule independently

An optimiser can return a schedule that looks cheap because of a modelling slip. `check_schedule` verifies a schedule against the physics, without trusting the optimiser: energy balance at every interval, power within the (temperature-limited) ratings, stored energy within the SOC window when re-simulated with `simulate_soc`, and curtailment never more than the PV available.

In [374]:
def check_schedule(sched: pd.DataFrame, load, pv, temp_C, spec: BatterySpec, dt_h: float = DT_H, tol: float = 1e-4) -> dict:
    """Physical checks on an optimiser schedule. Returns the worst violation of each check (all should be ~0)."""
    # 1. Power balance error
    balance = (load - (pv - sched.curtail_kW + sched.discharge_kW - sched.charge_kW + sched.import_kW - sched.export_kW)).abs().max()

    # 2. Charge and discharge power limits (adjusted by temperature factor)
    f = power_limit_factor(temp_C, spec)
    charge_over = (sched.charge_kW - spec.p_charge_kW * f).clip(lower=0).max()
    discharge_over = (sched.discharge_kW - spec.p_discharge_kW * f).clip(lower=0).max()
    # 3.
    e_first = (
        sched.energy_kWh.iloc[0] - (spec.eta_ch * sched.charge_kW.iloc[0] - sched.discharge_kW.iloc[0] / spec.eta_dis) * dt_h
    )

    e = simulate_soc(sched.discharge_kW - sched.charge_kW, spec, dt_h=dt_h, e0_kWh=e_first,)

    energy_gap = (e - sched.energy_kWh).abs().max()
    energy_outside = max((spec.e_min - e).max(), (e - spec.e_max).max(), 0.0,)

    # 4. Curtailment above available PV
    curtail_over = (sched.curtail_kW - pv).clip(lower=0).max()

    return {
        "balance": float(balance),
        "charge_over": float(charge_over),
        "discharge_over": float(discharge_over),
        "energy_gap": float(energy_gap),
        "energy_outside": float(energy_outside),
        "curtail_over": float(curtail_over),
    }


In [375]:
"""Tests for Task 7.2"""
_s = BatterySpec("t", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.81)
_i = pd.date_range("2023-01-02", periods=4, freq="h")
_load = pd.Series([0, 0, 2, 2], index=_i, dtype=float); _pv = pd.Series(0.0, index=_i); _t = pd.Series(20.0, index=_i)
_o = optimise_dispatch(_load, _pv, pd.Series([0.1, 0.1, 0.5, 0.5], index=_i), pd.Series(0.0, index=_i),
                       pd.Series(1.0, index=_i), _s, dt_h=1.0, e0=5.0)
_ok = check_schedule(_o, _load, _pv, _t, _s, dt_h=1.0)
assert all(v < 1e-4 for v in _ok.values()), _ok
_bad = _o.copy(); _bad.loc[_i[0], "charge_kW"] += 1.0                     # break the energy balance and the energy track
assert max(check_schedule(_bad, _load, _pv, _t, _s, dt_h=1.0).values()) > 0.5
print("Task 7.2 tests passed")

Task 7.2 tests passed


## Task 7.3 - Optimise all three tariffs (2023)

In [376]:
DEG_COST = degradation_cost_per_kwh(BATTERY) if USE_DEGRADATION_COST else 0.0
switches = dict(allow_grid_charging=ALLOW_GRID_CHARGING, allow_battery_export=ALLOW_BATTERY_EXPORT)
opt = {}
for t in TARIFFS:
    opt[t] = optimise_year(load2, pv2, prices[f"{t}_imp"], prices[f"{t}_exp"], temp2, BATTERY, deg_cost=DEG_COST, **switches)
    worst = check_schedule(opt[t], load2, pv2, temp2, BATTERY)
    assert max(worst.values()) < 1e-3, (t, worst)
    grid = opt[t]["import_kW"] - opt[t]["export_kW"]
    fee = VPP_FEE_PER_MONTH if t == "wholesale" else 0.0          # VPP-SPREE optimises the battery: fee applies on wholesale
    results[("Optimised (perfect foresight)", t)] = dict(
        bill=annual_bill(grid, prices[f"{t}_imp"], prices[f"{t}_exp"], fee_per_month=fee),
        **site_metrics(load2, pv2 - opt[t]["curtail_kW"], grid, opt[t]["discharge_kW"] - opt[t]["charge_kW"]))
    print(f"{t:9s} solved and checked | curtailed {opt[t].curtail_kW.sum() * DT_H:,.0f} kWh")

def with_wear(tab: pd.DataFrame, deg_cost: float) -> pd.DataFrame:
    """Add the battery wear cost so modes that cycle more or less are compared fairly: total = bill + wear."""
    out = tab.copy()
    out["wear"] = out["cycles"] * BATTERY.usable_kWh * deg_cost
    out["bill_plus_wear"] = out["bill"] + out["wear"]
    base = out.xs("No battery", level="mode")["bill_plus_wear"]
    out["net_saving"] = base.reindex(out.index.get_level_values("tariff")).to_numpy() - out["bill_plus_wear"]
    return out

tab7 = with_wear(results_table(results), degradation_cost_per_kwh(BATTERY))
tab7.style.format({"bill": "${:,.0f}", "saving": "${:,.0f}", "wear": "${:,.0f}", "bill_plus_wear": "${:,.0f}",
                   "net_saving": "${:,.0f}", "import_kWh": "{:,.0f}", "export_kWh": "{:,.0f}",
                   "self_consumption": "{:.0%}", "self_sufficiency": "{:.0%}", "cycles": "{:.0f}"})

flat      solved and checked | curtailed 0 kWh
tou       solved and checked | curtailed 0 kWh
wholesale solved and checked | curtailed 793 kWh


In [377]:
# Figure 7.1 - What the optimised battery does on a summer and a winter day, per tariff
days = {"Summer day (15 Jan 2023)": "2023-01-15", "Winter day (15 Jul 2023)": "2023-07-15"}
fig = make_subplots(rows=3, cols=2, shared_xaxes=True, vertical_spacing=0.07, horizontal_spacing=0.06,
                    subplot_titles=[f"{names[t]}: {d.split(' (')[0]}" for t in TARIFFS for d in days],
                    specs=[[{"secondary_y": True}] * 2] * 3)
for r_, t in enumerate(TARIFFS, start=1):
    for c_, (label, day) in enumerate(days.items(), start=1):
        o, show = opt[t].loc[day], (r_ == 1 and c_ == 1)
        h = o.index.hour + o.index.minute / 60
        fig.add_trace(go.Scatter(x=h, y=load2.loc[day], name="Load", line=dict(color="#444444"), showlegend=show), r_, c_)
        fig.add_trace(go.Scatter(x=h, y=pv2.loc[day], name="PV", line=dict(color="#f2a900"), showlegend=show), r_, c_)
        fig.add_trace(go.Bar(x=h, y=o.discharge_kW - o.charge_kW, name="Battery (+ discharge, - charge)",
                             marker_color="#2ca02c", showlegend=show), r_, c_)
        fig.add_trace(go.Scatter(x=h, y=prices.loc[day, f"{t}_imp"] * 100, name="Import price (c/kWh, right axis)",
                                 line=dict(color="#d62728", dash="dot"), showlegend=show), r_, c_, secondary_y=True)
fig.update_layout(title="Optimised battery operation (perfect foresight), 2023", template="plotly_white", height=900,
                  barmode="relative", legend=dict(orientation="h", yanchor="top", y=-0.05, x=0))
fig.update_yaxes(title_text="kW", secondary_y=False)
fig.show()

> **Finding 7 - Optimiser.** TODO in my words: the three switch decisions and why; the bill, saving, wear and net saving on each tariff vs both benchmarks (Table above); what the battery does differently on each tariff (Figure 7.1: e.g. ToU charging off-peak overnight in winter, wholesale discharging into 4-7 pm prices, curtailment at negative prices); and the key insight that the wear cost (~$0.39/kWh) is as large as the price spreads, so the optimiser cycles only when it really pays.

# Part 8 - Sizing and Payback

Part 7 showed that the result turns on two things: how big the battery is, and how much each kWh of cycling costs in wear. Part 8 tests both, together with the federal battery rebate, and turns the savings into a payback period.

## Task 8.1 - Scenarios

| Dimension | Options | Why |
|---|---|---|
| Size | 1, 2 or 3 Sigenergy 8 kWh modules (7.8 / 15.6 / 23.4 kWh usable) | Task 4.2: shiftable energy median 7.4, 90th percentile 10.7 kWh/day |
| Wear basis | **warranty**: 23.77 MWh throughput per module to 70% (about 3,050 cycles); **cycle life**: datasheet up to 10,000 cycles | The warranty is the manufacturer's guaranteed minimum, so a conservative (high) wear cost; the datasheet cycle life is optimistic (low) |
| Rebate | none; federal Cheaper Home Batteries rebate | Lowers capex, so both the payback and the wear cost |

Power: each module is rated 4 kW, and the single-phase energy controller limits the system to 5 kW. Cost: about $1,000 per usable kWh installed before rebate (price surveys); the per-kWh price falls a little for larger systems, which is ignored (conservative for 3 modules). Rebate: about $250 per usable kWh on the first 14 kWh (rate from 1 May 2026; earlier it was up to $372/kWh), tapering above 14 kWh; the taper is ignored here.

In [378]:
# ---------- Sizing inputs (sources in Task 8.1) ----------
MODULE_KWH = 7.8                    # usable kWh per Sigen 8 kWh module
MODULE_KW = 4.0                     # max charge/discharge per module
CONTROLLER_KW = 5.0                 # single-phase energy controller limit (SigenStor SP 16: 5 kW)
COST_PER_KWH = 1000.0               # installed, before rebate, $/usable kWh
WARRANTY_KWH_PER_MODULE = 23_770    # minimum throughput to end of warranty (70%)
CYCLE_LIFE_DATASHEET = 10_000       # datasheet cycle life (optimistic)
REBATE_PER_KWH, REBATE_CAP_KWH = 250.0, 14.0   # federal rebate, first 14 kWh

In [379]:
def sigen_spec(n_modules: int, wear_basis: str = "warranty", rebate: bool = False) -> BatterySpec:
    """SigenStor with n modules, built from SIGEN (Task 5.1) and the sizing inputs above.

    usable = n x MODULE_KWH; power = min(n x MODULE_KW, CONTROLLER_KW);
    warranty_cycles = WARRANTY_KWH_PER_MODULE / MODULE_KWH ("warranty") or CYCLE_LIFE_DATASHEET ("cycle life");
    capex = COST_PER_KWH x usable, minus REBATE_PER_KWH x min(usable, REBATE_CAP_KWH) if rebate.
    Everything else (rte, temperature settings, eol_capacity) is copied from SIGEN.
    """
    usable = n_modules * MODULE_KWH
    power = min(n_modules * MODULE_KW, CONTROLLER_KW)

    if wear_basis == "warranty":
        cycles = WARRANTY_KWH_PER_MODULE / MODULE_KWH
    elif wear_basis == "cycle life":
        cycles = CYCLE_LIFE_DATASHEET
    else:
        raise ValueError(f"Unknown wear_basis: {wear_basis!r}")

    capex = COST_PER_KWH * usable
    if rebate:
        capex -= REBATE_PER_KWH * min(usable, REBATE_CAP_KWH)

    return replace(
        SIGEN,
        name=f"SigenStor ({n_modules} mod)",
        usable_kWh=usable,
        p_charge_kW=power,
        p_discharge_kW=power,
        warranty_cycles=cycles,
        capex_aud=capex,
    )


def simple_payback(capex: float, annual_saving: float) -> float:
    """Years to recover capex from annual savings; infinite if the battery saves nothing (or loses money)."""
    return capex / annual_saving if annual_saving > 0 else np.inf

In [380]:
"""Tests for Task 8.1"""
_one = sigen_spec(1)
assert np.isclose(_one.usable_kWh, 7.8) and np.isclose(_one.p_charge_kW, 4.0) and np.isclose(_one.capex_aud, 7800)
_three = sigen_spec(3, wear_basis="cycle life", rebate=True)
assert np.isclose(_three.usable_kWh, 23.4) and np.isclose(_three.p_discharge_kW, 5.0)           # controller limit
assert np.isclose(_three.warranty_cycles, 10_000)
assert np.isclose(_three.capex_aud, 23_400 - 250 * 14)                                          # rebate only on the first 14 kWh
assert np.isclose(sigen_spec(2).warranty_cycles, 23_770 / 7.8)
assert np.isclose(sigen_spec(2).rte, SIGEN.rte) and sigen_spec(2).power_frac_outside == SIGEN.power_frac_outside
assert simple_payback(10_000, 500) == 20 and simple_payback(10_000, 0) == np.inf and simple_payback(10_000, -50) == np.inf
print("Task 8.1 tests passed")

Task 8.1 tests passed


## Task 8.2 - Sweep

Each scenario runs the Part 7 optimiser on all three tariffs (with that scenario's wear cost in the objective) and the self-consumption rule. 12 battery scenarios x 3 tariffs is 36 year-long optimisations, which takes roughly 5-15 minutes, so results are cached to `data/results/sizing.csv` and re-used on later runs. Delete the file (or set `RERUN_SWEEP = True`) after changing any input.

In [381]:
RERUN_SWEEP = False
SWEEP_FILE = PRICE_DIR / "results" / "sizing.csv"
SCENARIOS = [(n, w, r) for n in (1, 2, 3) for w in ("warranty", "cycle life") for r in (False, True)]

def run_sweep() -> pd.DataFrame:
    rows = []
    nobatt = {t: results[("No battery", t)]["bill"] for t in TARIFFS}
    for n, wear, rebate in SCENARIOS:
        spec = sigen_spec(n, wear, rebate)
        deg = degradation_cost_per_kwh(spec)
        sc = self_consumption_dispatch(load2, pv2, temp2, spec)
        for t in TARIFFS:
            o = optimise_year(load2, pv2, prices[f"{t}_imp"], prices[f"{t}_exp"], temp2, spec, deg_cost=deg, **switches)
            assert max(check_schedule(o, load2, pv2, temp2, spec).values()) < 1e-3
            fee = VPP_FEE_PER_MONTH if t == "wholesale" else 0.0
            for mode, grid, dis in [("Self-consumption", load2 - pv2 - sc, sc.clip(lower=0)),
                                    ("Optimised", o.import_kW - o.export_kW, o.discharge_kW)]:
                bill = annual_bill(grid, prices[f"{t}_imp"], prices[f"{t}_exp"], fee_per_month=fee if mode == "Optimised" else 0.0)
                rows.append(dict(modules=n, usable_kWh=spec.usable_kWh, wear_basis=wear, rebate=rebate, tariff=t, mode=mode,
                                 capex=spec.capex_aud, deg_cost=deg, bill=bill, saving=nobatt[t] - bill,
                                 cycles=dis.sum() * DT_H / spec.usable_kWh))
        print(f"done: {n} module(s), {wear}, rebate={rebate}")
    return pd.DataFrame(rows)

if SWEEP_FILE.exists() and not RERUN_SWEEP:
    sweep = pd.read_csv(SWEEP_FILE)
    print(f"Loaded cached sweep ({len(sweep)} rows) from {SWEEP_FILE.name}")
else:
    sweep = run_sweep()
    SWEEP_FILE.parent.mkdir(parents=True, exist_ok=True)
    sweep.to_csv(SWEEP_FILE, index=False)
sweep["payback_years"] = [simple_payback(c, s) for c, s in zip(sweep.capex, sweep.saving)]
sweep["years_to_warranty_throughput"] = (WARRANTY_KWH_PER_MODULE / MODULE_KWH) / sweep.cycles.replace(0, np.nan)
sweep.pivot_table(index=["mode", "tariff"], columns=["wear_basis", "rebate", "modules"], values="payback_years").round(0)

Loaded cached sweep (72 rows) from sizing.csv


wear_basis                 cycle life                               warranty                               
rebate                          False             True                 False              True             
modules                             1     2     3     1     2     3        1     2      3     1     2     3
mode             tariff                                                                                    
Optimised        flat            14.0  24.0  34.0  10.0  18.0  29.0      inf   inf    inf   inf   inf   inf
                 tou             18.0  34.0  50.0  12.0  22.0  43.0      inf   inf    inf  20.0  40.0  66.0
                 wholesale       13.0  19.0  27.0   9.0  14.0  22.0     67.0  79.0  114.0  19.0  31.0  66.0
Self-consumption flat            14.0  23.0  33.0  10.0  18.0  28.0     14.0  23.0   33.0  10.0  18.0  28.0
                 tou             17.0  30.0  44.0  13.0  24.0  37.0     17.0  30.0   44.0  13.0  24.0  37.0
                 wholesale       16.0  28.0  41.0  12.0  22.0  34.0     16.0  28.0   41.0  12.0  22.0  34.0

In [382]:
# Figure 8.1 - Payback by size, tariff and assumptions (optimised battery)
opt_sw = sweep[sweep["mode"] == "Optimised"]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=("Wear cost from warranty (conservative)",
                                                                       "Wear cost from datasheet cycle life (optimistic)"))
colours = {"flat": "#7f7f7f", "tou": "#1f77b4", "wholesale": "#d62728"}
for c_, wear in enumerate(["warranty", "cycle life"], start=1):
    for t in TARIFFS:
        for rebate, dash in [(False, "solid"), (True, "dash")]:
            s = opt_sw[(opt_sw.wear_basis == wear) & (opt_sw.tariff == t) & (opt_sw.rebate == rebate)]
            y = s["payback_years"].replace(np.inf, np.nan).clip(upper=40)
            fig.add_trace(go.Scatter(x=s["usable_kWh"], y=y, mode="lines+markers",
                                     name=f"{names[t]}{' + rebate' if rebate else ''}", showlegend=(c_ == 1),
                                     line=dict(color=colours[t], dash=dash)), 1, c_)
    fig.add_hline(y=10, line_dash="dot", line_color="black", annotation_text="10-year warranty", row=1, col=c_)
fig.update_xaxes(title_text="Usable capacity (kWh)", tickvals=[7.8, 15.6, 23.4])
fig.update_yaxes(title_text="Simple payback (years; gaps = never pays back; capped at 40)", row=1, col=1)
fig.update_layout(title="Payback of the optimised battery, 2023 prices", template="plotly_white", height=460,
                  legend=dict(orientation="h", yanchor="top", y=-0.18, x=0))
fig.show()

> **Finding 8 - Sizing and payback.** TODO in my words: which size pays back fastest on each tariff, and under which assumptions; how much the wear basis (warranty vs cycle life) and the rebate change the answer; whether any scenario pays back within the 10-year warranty; my chosen size and why (it becomes `BATTERY` for Parts 9-13); and what this says about why VPP uptake is low (the brief's context).

# Part 9 - PV Forecasting

The rolling-horizon scheduler (Part 11) cannot see the future, so it needs a PV forecast. The brief asks for a model trained on Year 1 (measured PV vs weather), used with Year 2 weather to forecast Year 2 PV, and scored against the commercial forecast, which stands in for the real Year 2 generation.

## Task 9.1 - Model

### Background
PV output is close to proportional to irradiance, reduced as the panels heat up:

**PV ≈ size × G × PR × (1 + γ (T_cell − 25))**, with G the irradiance in kW/m² and γ the temperature coefficient (−0.004 /°C, Part 0).

Expanding it gives a model that is **linear in two features**, G and G × T:

**PV ≈ a × G + b × G × T**

so the two coefficients can be fitted from Year 1 data by least squares, with no need to know the array size, the performance ratio or the cell temperature exactly. The fitted coefficients are physically interpretable (a ≈ size × PR × (1 − 25γ) plus the cell heating; b/a ≈ γ), which makes the model easy to sanity-check. A pure physics model built from the Part 0/Task 4.3 values (size 8.9 kWp, PR 0.85, γ, and a standard cell-temperature estimate) is scored alongside it as a baseline.

Training uses only Year 1 rows with real measured PV and real (not gap-filled) weather (Part 2 flags).

In [383]:
def fit_pv_model(irr_Wm2: pd.Series, temp_C: pd.Series, pv_kW: pd.Series) -> np.ndarray:
    """Least-squares fit of pv ~ a x G + b x G x T (G = irr / 1000, in kW/m^2). Returns np.array([a, b])."""
    G = irr_Wm2 / 1000.0
    X = np.column_stack([G, G * temp_C])
    coef, _, _, _ = np.linalg.lstsq(X, pv_kW, rcond=None)
    return coef


def predict_pv(irr_Wm2: pd.Series, temp_C: pd.Series, coef: np.ndarray) -> pd.Series:
    """PV forecast (kW) = a x G + b x G x T, never negative. Same index as irr_Wm2."""
    G = irr_Wm2 / 1000.0
    pred = coef[0] * G + coef[1] * (G * temp_C)
    return pd.Series(pred, index=irr_Wm2.index).clip(lower=0.0)


def forecast_errors(forecast: pd.Series, actual: pd.Series, capacity_kW: float) -> dict:
    """Forecast error metrics (kW): MAE, RMSE, nRMSE (RMSE / capacity, as a fraction) and MBE (mean of forecast - actual)."""
    err = forecast - actual
    mae = float(err.abs().mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    nrmse = float(rmse / capacity_kW)
    mbe = float(err.mean())

    out = {
        "MAE": mae,
        "RMSE": rmse,
        "nRMSE": nrmse,
        "MBE": mbe,
    }

    return out

In [384]:
"""Tests for Task 9.1"""
_i = pd.date_range("2022-01-01 10:00", periods=50, freq="5min")
_irr = pd.Series(np.linspace(100, 1000, 50), index=_i)
_tmp = pd.Series(np.linspace(0, 30, 50), index=_i)
_pv = 9.0 * _irr / 1000 - 0.03 * (_irr / 1000) * _tmp                         # known a = 9.0, b = -0.03
assert np.allclose(fit_pv_model(_irr, _tmp, _pv), [9.0, -0.03])              # recovers the coefficients exactly
_p = predict_pv(pd.Series([0.0, 500.0, -10.0], index=_i[:3]), pd.Series([10.0, 10.0, 10.0], index=_i[:3]), np.array([9.0, -0.03]))
assert np.allclose(_p, [0.0, 4.5 - 0.15, 0.0]) and _p.index.equals(_i[:3])    # never negative
_e = forecast_errors(pd.Series([1.0, 3.0]), pd.Series([2.0, 2.0]), capacity_kW=10)
assert np.isclose(_e["MAE"], 1.0) and np.isclose(_e["RMSE"], 1.0) and np.isclose(_e["nRMSE"], 0.1) and np.isclose(_e["MBE"], 0.0)
print("Task 9.1 tests passed")

Task 9.1 tests passed


In [385]:
# Physics baseline (no fitting): size x G x PR x (1 + gamma (T_cell - 25)), T_cell from the standard NOCT estimate
NOCT_C = 45.0   # (A) typical nominal operating cell temperature for crystalline panels

def physics_pv(irr_Wm2: pd.Series, temp_C: pd.Series) -> pd.Series:
    G = irr_Wm2 / 1000
    t_cell = temp_C + G * (NOCT_C - 20) / 0.8
    return (PV_KWP_EST * G * PV_PR * (1 + PV_GAMMA * (t_cell - 25))).clip(lower=0)

# Training rows: Year 1, measured PV present, weather not gap-filled
train = (dfc.year == 1) & ~dfc.pv_meas_missing & ~dfc.irr_filled & ~dfc.temp_filled
PV_COEF = fit_pv_model(dfc.loc[train, "irr_Wm2"], dfc.loc[train, "temp_C"], dfc.loc[train, "pv_meas_kW"])
print(f"Fitted: PV = {PV_COEF[0]:.3f} x G + {PV_COEF[1]:.4f} x G x T   (b/a = {PV_COEF[1] / PV_COEF[0]:+.4f} per degC; "
      f"datasheet gamma {PV_GAMMA:+.4f})  | trained on {train.sum():,} intervals")

# Year 2 forecast from Year 2 weather; this is the PV forecast the scheduler uses in Part 11
PV_FORECAST = predict_pv(dfc.loc[y2, "irr_Wm2"], dfc.loc[y2, "temp_C"], PV_COEF)

Fitted: PV = 7.848 x G + -0.0310 x G x T   (b/a = -0.0040 per degC; datasheet gamma -0.0040)  | trained on 104,936 intervals


## Task 9.2 - How good is the forecast?

In [386]:
# Score against the commercial forecast (= Year 2 'actual' PV), daylight only, excluding gap-filled weather
score_rows = y2 & (dfc.irr_Wm2 > 0) & ~dfc.irr_filled & ~dfc.temp_filled
actual = dfc.loc[score_rows, "pv_comm_kW"]
cands = {"Fitted model (Task 9.1)": PV_FORECAST[score_rows], "Physics baseline": physics_pv(dfc.loc[score_rows, "irr_Wm2"], dfc.loc[score_rows, "temp_C"])}
rows = {}
for name, f in cands.items():
    rows[(name, "5-min")] = forecast_errors(f, actual, PV_KWP_EST)
    rows[(name, "hourly")] = forecast_errors(f.resample("h").mean().dropna(), actual.resample("h").mean().dropna(), PV_KWP_EST)
# in-sample check on Year 1 (fitted model vs measured PV)
f1 = predict_pv(dfc.loc[train, "irr_Wm2"], dfc.loc[train, "temp_C"], PV_COEF)
rows[("Fitted model, Year 1 in-sample (vs measured)", "5-min")] = forecast_errors(f1[dfc.loc[train, "irr_Wm2"] > 0], dfc.loc[train & (dfc.irr_Wm2 > 0), "pv_meas_kW"], PV_KWP_EST)
pv_scores = pd.DataFrame(rows).T
pv_scores.style.format({"MAE": "{:.3f} kW", "RMSE": "{:.3f} kW", "nRMSE": "{:.1%}", "MBE": "{:+.3f} kW"})

In [387]:
# Figure 9.1 - Forecast vs commercial PV on three contrasting days, and error by month
fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.09,
                    subplot_titles=("Three days in 2023 (kW)", "Monthly nRMSE, daylight, 5-min"))
win = slice("2023-01-14", "2023-01-16")
fig.add_trace(go.Scatter(x=dfc.loc[win].index, y=dfc.loc[win, "pv_comm_kW"], name="Commercial forecast (actual)",
                         line=dict(color="black", width=1.5)), 1, 1)
fig.add_trace(go.Scatter(x=dfc.loc[win].index, y=PV_FORECAST.loc[win], name="My fitted model",
                         line=dict(color="#1f77b4", width=2)), 1, 1)
err = (PV_FORECAST[score_rows] - actual)
monthly = (np.sqrt((err ** 2).groupby(err.index.month).mean()) / PV_KWP_EST)
fig.add_trace(go.Bar(x=pd.to_datetime(monthly.index.astype(str), format="%m").strftime("%b"), y=monthly * 100,
                     marker_color="#1f77b4", showlegend=False), 1, 2)
fig.update_yaxes(title_text="kW", row=1, col=1); fig.update_yaxes(title_text="nRMSE (% of array size)", row=1, col=2)
fig.update_layout(title="PV forecast from Year 2 weather vs the commercial forecast", template="plotly_white", height=430,
                  legend=dict(orientation="h", yanchor="top", y=-0.15, x=0))
fig.show()

> **Finding 9 - PV forecast.** TODO in my words: the fitted coefficients and whether they make physical sense (b/a vs the datasheet γ, and what a implies for the array size); the error table: my model vs the commercial forecast (5-minute and hourly) and vs the physics baseline; and the key comparison: my model is within about 1% nRMSE of the commercial forecast in Year 2, but about 5% against real measured PV in Year 1. What does that say about the commercial forecast (smooth, weather-driven, no sub-hourly cloud dips), and so about how much PV uncertainty Part 12 can actually measure? Limitation: the model uses actual Year 2 weather, so it captures model error but not weather-forecast error, which would grow with lead time.

# Part 10 - Price Forecasting

The scheduler also has to forecast the wholesale price. The brief allows either AEMO pre-dispatch forecasts or an own model; an own model is used here because it can be built and tested entirely from the data already in hand, and it gives a transparent baseline against which a VPP's real forecasting can be judged.

## Task 10.1 - Two simple forecasts, with no look-ahead

### Background
The rule that matters most: a forecast issued at time τ may only use prices **known by τ**. With AEMO's interval-end timestamps (Task 3.1), the price stamped τ - 5 min is the last one known when the battery decides what to do in the interval that ends at τ + 5 min. Any forecast that peeks at later prices would make the forecast look better than any real controller could achieve.

Two seasonal forecasts, both built on the daily pattern seen in Figure 3.1:
- **Persistence (same time yesterday):** each target interval gets the price at the same time of day on the most recent day that is already known. For a 48-hour horizon, the first 24 hours use yesterday, the next 24 use the day before.
- **Profile median (last 7 days):** each target gets the median of the prices at the same time of day on the 7 most recent known days. The median resists one-off spikes, which persistence copies straight into tomorrow.

For a target t and issue time τ, the most recent known day at the same time of day is **k = ceil((t − τ + 5 min) / 1 day)** days back.

In [388]:
def days_back(targets: pd.DatetimeIndex, issue_time: pd.Timestamp) -> np.ndarray:
    """For each target, how many whole days back the same time of day was last KNOWN at issue_time (>= 1)."""
    lag = (targets - issue_time + pd.Timedelta(minutes=INTERVAL_MIN)) / pd.Timedelta(days=1)
    return np.maximum(np.ceil(lag).astype(int), 1)


def persistence_forecast(spot: pd.Series, issue_time: pd.Timestamp, targets: pd.DatetimeIndex) -> pd.Series:
    """Spot price forecast for `targets` = price at the same time of day on the most recent known day."""
    k = days_back(targets, issue_time)
    src_times = targets - pd.to_timedelta(k, unit = "D")
    values = spot.reindex(src_times).to_numpy()
    return pd.Series(values, index=targets)


def profile_median_forecast(spot: pd.Series, issue_time: pd.Timestamp, targets: pd.DatetimeIndex, days: int = 7) -> pd.Series:
    """Spot price forecast = median of the prices at the same time of day on the `days` most recent known days."""
    k = days_back(targets, issue_time)
    cols = []
    for j in range(days):
        src_times = targets - pd.to_timedelta(k + j, unit="D")
        cols.append(spot.reindex(src_times).to_numpy())

    hist = pd.DataFrame(np.column_stack(cols), index=targets)
    return hist.median(axis=1, skipna=True)

In [389]:
"""Tests for Task 10.1"""
_i = pd.date_range("2023-01-01", "2023-01-20 23:55", freq="5min")
_spot = pd.Series(np.arange(len(_i), dtype=float), index=_i)                 # every price is unique, so look-ups are checkable
_tau = pd.Timestamp("2023-01-10 12:00")
_tg = pd.DatetimeIndex([_tau, _tau + pd.Timedelta("23h55min"), _tau + pd.Timedelta("24h"), _tau + pd.Timedelta("47h")])
assert days_back(_tg, _tau).tolist() == [1, 1, 2, 2]
_f = persistence_forecast(_spot, _tau, _tg)
assert _f.tolist() == [_spot[t - pd.Timedelta(days=k)] for t, k in zip(_tg, [1, 1, 2, 2])] and _f.index.equals(_tg)
# no look-ahead: changing every price from the issue time onwards must not change the forecast
_future = _spot.copy(); _future[_future.index >= _tau] = 1e6
assert persistence_forecast(_future, _tau, _tg).equals(_f)
_m = profile_median_forecast(_spot, _tau, _tg, days=3)
assert np.isclose(_m.iloc[0], _spot[_tg[0] - pd.Timedelta(days=2)])             # median of days 1, 2, 3 back = day 2
assert profile_median_forecast(_future, _tau, _tg, days=3).equals(_m)
print("Task 10.1 tests passed")

Task 10.1 tests passed


## Task 10.2 - How good are they?

Each forecast is scored on 2023 for every hourly issue time, separately for targets 0-24 h and 24-48 h ahead, with a "perfect knowledge of the daily average shape" benchmark for context: the 2023 average price at that time of day, which a real forecaster could not have known in advance.

In [390]:
spot_y2 = dfc.loc[y2, "rrp_kWh"]
spot_all = dfc["rrp_kWh"]                                  # Year 1 prices give the history the first days of 2023 need
issues = pd.date_range("2023-01-01", "2023-12-31 00:00", freq="7h")   # every 7 h: covers all hours of the day evenly
def score_price(method):
    out = {"0-24 h": [], "24-48 h": []}
    for tau in issues:
        tg = spot_y2.index[(spot_y2.index >= tau) & (spot_y2.index < tau + pd.Timedelta(hours=48))]
        err = (method(spot_all, tau, tg) - spot_y2[tg]).to_numpy()
        lead = ((tg - tau) / pd.Timedelta(hours=1)).to_numpy()
        out["0-24 h"].append(err[lead < 24]); out["24-48 h"].append(err[lead >= 24])
    return {k: dict(MAE=np.mean(np.abs(np.concatenate(v))) * 1000, median_abs_error=np.median(np.abs(np.concatenate(v))) * 1000)
            for k, v in out.items()}

tod_mean = spot_y2.groupby([spot_y2.index.hour, spot_y2.index.minute]).transform("mean")
price_scores = pd.concat({
    "Persistence (yesterday)": pd.DataFrame(score_price(persistence_forecast)).T,
    "Profile median (7 days)": pd.DataFrame(score_price(profile_median_forecast)).T,
    "2023 average daily shape (hindsight, for context)": pd.DataFrame(
        {"all leads": dict(MAE=float((tod_mean - spot_y2).abs().mean() * 1000),
                           median_abs_error=float((tod_mean - spot_y2).abs().median() * 1000))}).T,
})
print("Errors in $/MWh (spot price; 2023)")
price_scores.round(1)

Errors in $/MWh (spot price; 2023)


MAE  median_abs_error
Persistence (yesterday)                           0-24 h     50.8              23.2
                                                  24-48 h    57.1              28.0
Profile median (7 days)                           0-24 h     40.9              21.9
                                                  24-48 h    42.0              23.0
2023 average daily shape (hindsight, for context) all leads  51.1              29.6

In [391]:
PRICE_METHOD = profile_median_forecast    # lower MAE in Task 10.2 (~$41-42/MWh vs $51-57 for persistence); used by Part 11.
                                          # Task 13.4 shows a mean forecast serves the scheduler better (expected, not typical, price)

In [392]:
# Figure 10.1 - Forecast vs actual spot price over one week (forecast issued daily at midnight, 24 h ahead)
wk = spot_y2.loc["2023-06-05":"2023-06-11"]
fc_days = []
for d in pd.date_range("2023-06-05", "2023-06-11", freq="D"):
    tg = wk.index[(wk.index >= d) & (wk.index < d + pd.Timedelta(days=1))]
    fc_days.append(pd.DataFrame({"persistence": persistence_forecast(spot_all, d, tg),
                                 "median": profile_median_forecast(spot_all, d, tg)}))
fc_wk = pd.concat(fc_days)
fig = go.Figure()
fig.add_trace(go.Scatter(x=wk.index, y=wk * 1000, name="Actual", line=dict(color="black", width=1.5)))
fig.add_trace(go.Scatter(x=fc_wk.index, y=fc_wk["persistence"] * 1000, name="Persistence (yesterday)", line=dict(color="#ff7f0e", width=1.2)))
fig.add_trace(go.Scatter(x=fc_wk.index, y=fc_wk["median"] * 1000, name="Profile median (7 days)", line=dict(color="#1f77b4", width=2)))
fig.update_layout(title="Spot price forecasts vs actual, one week in June 2023 (day-ahead)", yaxis_title="Spot price ($/MWh)",
                  template="plotly_white", height=420, legend=dict(orientation="h", yanchor="top", y=-0.15, x=0))
fig.show()

> **Finding 10 - Price forecast.** TODO in my words: the two methods and the no-look-ahead rule; their errors (0-24 h vs 24-48 h, and vs the hindsight daily shape); which one I chose and why; what they get right (the daily shape, Figure 10.1) and what they cannot do (predict spikes and negative-price days, which is where most of the wholesale battery value is: Part 12 measures the cost of that).

# Part 11 - Rolling-Horizon Scheduler

This is the controller a real VPP would run, as the brief specifies:
- **Plan (every hour):** at the start of each hour, solve the Part 7 LP over the next **48 hours at hourly resolution**, using forecasts: PV from Part 9, prices from Part 10 (flat and ToU prices are known in advance, so only PV is uncertain there), and perfect load (the brief allows this). The battery starts from its actual stored energy.
- **Execute (every 5 minutes):** for the 12 intervals of that hour, the battery follows the plan's first-hour power setpoint, **without knowing that interval's price or PV**. If the plan expects a negative export price, the inverter is told to curtail to zero export (the brief's rule). Only physical limits (power, temperature, state of charge) can override the setpoint; the grid absorbs whatever the actual load and PV leave over.
- **Settle:** the bill is calculated on the **actual** 5-minute prices, PV and load.

Then the clock moves on one hour and the plan is redone with the battery's actual state of charge. This is the 8,760 x 3 = 26,280 small LPs, so the run is cached like the Part 8 sweep.

In [393]:
def plan_to_hourly(s: pd.Series) -> pd.Series:
    """Hourly mean of a 5-minute series, labelled by the hour it starts."""
    return s.resample("h").mean()


def execute_hour(p_set: float, L, PV, F, e: float, spec: BatterySpec, zero_export: bool,
                 allow_grid_charging: bool = True, allow_battery_export: bool = True, follow_net: bool = False):
    """Execute one hour of 5-minute intervals on ACTUAL load and PV, starting from stored energy e (kWh).

    p_set: planned battery power (kW, + discharge). follow_net=True ignores it and runs self-consumption
    (store the actual surplus, cover the actual deficit). Physical limits always apply.
    Returns arrays batt_kW, curtail_kW, grid_kW, energy_kWh for the hour, and the stored energy at its end.
    """
    n = len(L)
    batt, curt, grid, energy = (np.zeros(n) for _ in range(4))
    for k in range(n):
        p = (L[k] - PV[k]) if follow_net else p_set
        if p < 0:                                               # charging: power, temperature and room limits
            p = -min(-p, spec.p_charge_kW * F[k], max(spec.e_max - e, 0) / (spec.eta_ch * DT_H))
            if not allow_grid_charging:
                p = -min(-p, max(PV[k] - L[k], 0))
        elif p > 0:                                             # discharging: power, temperature and energy limits
            p = min(p, spec.p_discharge_kW * F[k], max(e - spec.e_min, 0) * spec.eta_dis / DT_H)
            if not allow_battery_export:
                p = min(p, max(L[k] - PV[k], 0))
        g = L[k] - PV[k] - p                                    # + import, - export (before curtailment)
        u = min(-g, PV[k]) if (zero_export and g < 0) else 0.0  # curtail the surplus that would be exported
        g += u
        e += (spec.eta_ch * -p if p < 0 else -p / spec.eta_dis) * DT_H
        batt[k], curt[k], grid[k], energy[k] = p, u, g, e
    return batt, curt, grid, energy, e


def rolling_horizon(load, pv_actual, pv_fc, imp_actual, exp_actual, imp_fc_fn, exp_fc_fn, temp_C, spec: BatterySpec,
                    deg_cost: float = 0.0, horizon_h: int = 48, allow_grid_charging: bool = True,
                    allow_battery_export: bool = True, terminal_value_fn=None, fallback_fn=None) -> pd.DataFrame:
    """Hourly 48 h planning on forecasts, 5-minute execution on actuals.

    imp_fc_fn / exp_fc_fn: functions (issue_time, horizon_index_5min) -> forecast import / export price Series ($/kWh).
    Optional hooks (Task 13.4; None = the Part 11 controller):
      terminal_value_fn(imp_h, exp_h) -> $/kWh credited for energy left at the horizon end, replacing the
                                         'end at least at soc_init' rule;
      fallback_fn(plan, net_h, imp_h, exp_h, e0, tv) -> True to run self-consumption for this hour instead of the plan.
    Returns 5-minute columns: batt_kW (+ discharge), curtail_kW, grid_kW (+ import), energy_kWh.
    """
    idx = load.index
    factor = power_limit_factor(temp_C, spec)
    load_h, pvfc_h, fac_h = plan_to_hourly(load), plan_to_hourly(pv_fc), factor.resample("h").min()
    hours = load_h.index
    L, PV, F = load.to_numpy(), pv_actual.to_numpy(), factor.to_numpy()
    hour_of = np.searchsorted(hours.to_numpy(), idx.floor("h").to_numpy())
    batt, curt, grid, energy = (np.zeros(len(idx)) for _ in range(4))
    e = spec.soc_init * spec.usable_kWh
    starts = np.searchsorted(hour_of, np.arange(len(hours)))
    for h, tau in enumerate(hours):
        H = min(horizon_h, len(hours) - h)
        win5 = idx[(idx >= tau) & (idx < tau + pd.Timedelta(hours=H))]
        imp_h = plan_to_hourly(imp_fc_fn(tau, win5)).to_numpy()[:H]
        exp_h = plan_to_hourly(exp_fc_fn(tau, win5)).to_numpy()[:H]
        tv = terminal_value_fn(imp_h, exp_h) if terminal_value_fn is not None else 0.0
        end_min = spec.soc_init * spec.usable_kWh if (H == horizon_h and terminal_value_fn is None) else None
        plan = optimise_dispatch(load_h.iloc[h:h + H].to_numpy(), pvfc_h.iloc[h:h + H].to_numpy(), imp_h, exp_h,
                                 fac_h.iloc[h:h + H].to_numpy(), spec, dt_h=1.0, e0=e, e_end_min=end_min,
                                 deg_cost=deg_cost, allow_grid_charging=allow_grid_charging,
                                 allow_battery_export=allow_battery_export, terminal_value=tv)
        p_set = plan["discharge_kW"].iloc[0] - plan["charge_kW"].iloc[0]
        net_h = load_h.iloc[h:h + H].to_numpy() - pvfc_h.iloc[h:h + H].to_numpy()
        follow = fallback_fn is not None and bool(fallback_fn(plan, net_h, imp_h, exp_h, e, tv))
        zero_export = exp_h[0] < 0                                  # forecast negative export price -> curtail to zero export
        k0, k1 = starts[h], (starts[h + 1] if h + 1 < len(starts) else len(idx))
        batt[k0:k1], curt[k0:k1], grid[k0:k1], energy[k0:k1], e = execute_hour(
            p_set, L[k0:k1], PV[k0:k1], F[k0:k1], e, spec, zero_export, allow_grid_charging, allow_battery_export, follow)
    return pd.DataFrame({"batt_kW": batt, "curtail_kW": curt, "grid_kW": grid, "energy_kWh": energy}, index=idx)

In [394]:
"""Tests for Task 11 (small synthetic week; checks the controller's rules, not its economics)"""
_i = pd.date_range("2023-01-02", periods=3 * 288, freq="5min")
_h = np.asarray(_i.hour + _i.minute / 60)
_load = pd.Series(0.5, index=_i)
_pv = pd.Series(np.clip(4 * np.sin(np.pi * (_h - 6) / 12), 0, None), index=_i)
_imp = pd.Series(np.where((_h >= 17) & (_h < 21), 0.60, 0.20), index=_i)
_exp = pd.Series(np.where((_h >= 11) & (_h < 13), -0.05, 0.05), index=_i)             # negative export at midday
_s = BatterySpec("t", usable_kWh=5, p_charge_kW=3, p_discharge_kW=3, rte=0.81)
_t = pd.Series(20.0, index=_i)
_rh = rolling_horizon(_load, _pv, _pv, _imp, _exp, lambda tau, w: _imp[w], lambda tau, w: _exp[w], _t, _s)
# energy within limits and consistent with the battery power (independent re-simulation)
assert _rh.energy_kWh.min() >= -1e-9 and _rh.energy_kWh.max() <= 5 + 1e-9
assert np.allclose(simulate_soc(_rh.batt_kW, _s, e0_kWh=2.5), _rh.energy_kWh)
# power balance on actuals
assert np.allclose(_load - (_pv - _rh.curtail_kW) - _rh.batt_kW, _rh.grid_kW)
# zero export whenever the (here perfectly) forecast export price is negative
neg = _exp < 0
assert (_rh.grid_kW[neg] >= -1e-9).all() and _rh.curtail_kW[~neg].abs().max() < 1e-12
# with perfect information it should beat doing nothing
assert annual_bill(_rh.grid_kW, _imp, _exp) < annual_bill(_load - _pv, _imp, _exp)
print("Task 11 tests passed")

Task 11 tests passed


## Task 11.1 - Run the scheduler on 2023 (forecast PV and forecast prices)

Flat and ToU prices are known in advance, so their "forecast" is the tariff itself; only the wholesale price is forecast (Part 10, `PRICE_METHOD`), and turned into an import and export price with the same formula as Task 3.3. Results are cached in `data/results/rolling_<scenario>.parquet`-style CSVs; set `RERUN_ROLLING = True` after changing anything upstream.

In [395]:
RERUN_ROLLING = False
ROLL_DIR = PRICE_DIR / "results"

def tariff_forecasters(t: str, price_method=None):
    """(import, export) forecast functions for tariff t. price_method=None means perfect price foresight."""
    if t != "wholesale" or price_method is None:
        return (lambda tau, w: prices.loc[w, f"{t}_imp"]), (lambda tau, w: prices.loc[w, f"{t}_exp"])
    def fc(tau, w):
        return wholesale_retail_price(price_method(spot_all, tau, w))
    return (lambda tau, w: fc(tau, w)["import_kWh"]), (lambda tau, w: fc(tau, w)["export_kWh"])

def run_rolling(t: str, pv_fc: pd.Series, price_method, tag: str) -> pd.DataFrame:
    """Run (or load from cache) one rolling-horizon year for tariff t."""
    f = ROLL_DIR / f"rolling_{tag}_{t}.csv"
    if f.exists() and not RERUN_ROLLING:
        return pd.read_csv(f, index_col=0, parse_dates=True)
    imp_fn, exp_fn = tariff_forecasters(t, price_method)
    out = rolling_horizon(load2, pv2, pv_fc, prices[f"{t}_imp"], prices[f"{t}_exp"], imp_fn, exp_fn, temp2, BATTERY,
                          deg_cost=DEG_COST, **switches)
    f.parent.mkdir(parents=True, exist_ok=True); out.to_csv(f)
    return out

def settle(t: str, run: pd.DataFrame, label: str):
    fee = VPP_FEE_PER_MONTH if t == "wholesale" else 0.0
    results[(label, t)] = dict(bill=annual_bill(run.grid_kW, prices[f"{t}_imp"], prices[f"{t}_exp"], fee_per_month=fee),
                               **site_metrics(load2, pv2 - run.curtail_kW, run.grid_kW, run.batt_kW))

roll = {}
for t in TARIFFS:
    roll[t] = run_rolling(t, PV_FORECAST, PRICE_METHOD, "forecast")
    settle(t, roll[t], "Rolling horizon (forecasts)")
    print(f"{t:9s} done")
tab11 = with_wear(results_table(results), degradation_cost_per_kwh(BATTERY))
tab11.style.format({"bill": "${:,.0f}", "saving": "${:,.0f}", "wear": "${:,.0f}", "bill_plus_wear": "${:,.0f}",
                    "net_saving": "${:,.0f}", "import_kWh": "{:,.0f}", "export_kWh": "{:,.0f}",
                    "self_consumption": "{:.0%}", "self_sufficiency": "{:.0%}", "cycles": "{:.0f}"})

flat      done
tou       done
wholesale done


> **Finding 11 - Rolling horizon.** TODO in my words: how the scheduler works (plan hourly 48 h ahead on forecasts, execute every 5 minutes without knowing the current price or PV, settle on actuals); its bill and net saving on each tariff compared with the perfect-foresight optimiser (Part 7) and the benchmarks (Part 6); and a first look at how much value is lost to forecasting (Part 12 splits that loss into PV and price).

# Part 12 - Perfect Foresight vs Forecasts: Is PV or Price the Bigger Driver?

The brief asks how much value is lost to uncertainty, and whether PV or price has the larger influence. Part 12 answers it by re-running the Part 11 scheduler with the forecasts swapped for perfect information one at a time (an ablation study). Everything else stays identical: the same 48-hour hourly planning, the same 5-minute execution, the same settlement on actuals.

| Run | PV used for planning | Price used for planning | Tariffs |
|---|---|---|---|
| A | perfect (actual) | perfect (actual) | all three |
| B | forecast (Part 9) | perfect | wholesale |
| C | perfect | forecast (Part 10) | wholesale |
| D | forecast | forecast (= Part 11) | all three |

On flat and ToU the price is known in advance, so B = D and C = A: only runs A and D are needed.

**Value lost** (in $ of battery saving per year): to PV = A − B; to price = A − C; the rest of A − D is the **interaction** (errors in both at once, which can add to or offset each other). Run A is also compared with the Part 7 optimiser: both have perfect information, so the gap between them is the cost of the 48-hour horizon and hourly planning, not of forecasting.

In [396]:
def battery_saving(nobatt_bill: float, run: pd.DataFrame, imp_price: pd.Series, exp_price: pd.Series) -> float:
    """Bill saving ($) of a scheduler run vs no battery, BEFORE any VPP fee.

    run: a Part 11-style result with a grid_kW column (+ import, - export).
    """
    savings = float(nobatt_bill - annual_bill(run["grid_kW"], imp_price, exp_price, fee_per_month=0.0))
    return savings


def attribute_loss(save_A: float, save_B: float, save_C: float, save_D: float) -> dict:
    """Split the value lost to forecasting (A - D) into PV (A - B), price (A - C) and interaction (the remainder)."""

    total = save_A - save_D
    pv = save_A - save_B
    price = save_A - save_C
    interaction = total - (pv + price)

    att_loss = {
        "total": float(total),
        "pv": float(pv),
        "price": float(price),
        "interaction": float(interaction),
    }

    return att_loss

In [397]:
"""Tests for Task 12"""
_i = pd.date_range("2023-01-01", periods=2, freq="5min")
_run = pd.DataFrame({"grid_kW": [12.0, -12.0]}, index=_i)                      # 1 kWh imported, then 1 kWh exported
assert np.isclose(battery_saving(1.0, _run, pd.Series(0.30, index=_i), pd.Series(0.05, index=_i)), 1.0 - (0.30 - 0.05))
_a = attribute_loss(500, 480, 100, 90)
assert _a == dict(total=410, pv=20, price=400, interaction=-10)          # -10: the two errors partly offset each other
assert attribute_loss(300, 300, 300, 300) == dict(total=0, pv=0, price=0, interaction=0)
print("Task 12 tests passed")

Task 12 tests passed


In [398]:
# Task 12.1 - Run the scenarios. Fill the dict `runs` with one Part 11-style result per (scenario, tariff).

runs = {}
for t in TARIFFS:
    # Scenario D is already stored in roll[t] from Part 11
    runs[("D", t)] = roll[t]

    # Scenario A: perfect PV + perfect prices (needed for all tariffs)
    runs[("A", t)] = run_rolling(t, pv2, None, "perfect")

    if t == "wholesale":
        # Scenario B: forecast PV + perfect prices
        runs[("B", t)] = run_rolling(t, PV_FORECAST, None, "pvfc")
        # Scenario C: perfect PV + forecast prices
        runs[("C", t)] = run_rolling(t, pv2, PRICE_METHOD, "pricefc")
    else:
        # On flat and ToU tariffs, prices are known ahead of time:
        # B has the same PV forecast and price knowledge as D, and C matches A
        runs[("B", t)] = runs[("D", t)]
        runs[("C", t)] = runs[("A", t)]

assert set(runs) == {(s, t) for s in "ABCD" for t in TARIFFS}, "runs should hold every (scenario, tariff) pair"
print("All scenarios ready:", sorted(runs))

All scenarios ready: [('A', 'flat'), ('A', 'tou'), ('A', 'wholesale'), ('B', 'flat'), ('B', 'tou'), ('B', 'wholesale'), ('C', 'flat'), ('C', 'tou'), ('C', 'wholesale'), ('D', 'flat'), ('D', 'tou'), ('D', 'wholesale')]


In [399]:
# Task 12.2 - Results table. Build two DataFrames, both with one row per tariff (index = TARIFFS):
abl = pd.DataFrame({
    t: {
        "Part 7 optimiser": (
            results[("No battery", t)]["bill"]
            - results[("Optimised (perfect foresight)", t)]["bill"]
            + (VPP_FEE_PER_MONTH * 12 if t == "wholesale" else 0.0)
        ),
        **{
            s: battery_saving(
                results[("No battery", t)]["bill"],
                runs[(s, t)],
                prices[f"{t}_imp"],
                prices[f"{t}_exp"],
            )
            for s in "ABCD"
        },
    }
    for t in TARIFFS
}).T

attr = pd.DataFrame({
    t: attribute_loss(abl.loc[t, "A"], abl.loc[t, "B"], abl.loc[t, "C"], abl.loc[t, "D"],)
    for t in TARIFFS}).T

assert list(abl.columns) == ["Part 7 optimiser", "A", "B", "C", "D"] and list(attr.columns) == ["total", "pv", "price", "interaction"]
print("Battery saving vs no battery, $/yr, before the VPP fee")
pd.concat({"saving": abl, "value lost to forecasting": attr}, axis=1).round(0)

Battery saving vs no battery, $/yr, before the VPP fee


saving                           value lost to forecasting                        
          Part 7 optimiser      A      B     C     D                     total   pv  price interaction
flat                   0.0    0.0    0.0   0.0   0.0                       0.0  0.0    0.0         0.0
tou                    0.0    1.0    1.0   1.0   1.0                      -0.0 -0.0    0.0         0.0
wholesale            498.0  421.0  422.0  28.0  28.0                     393.0 -1.0  393.0         1.0

In [400]:
# Figure 12.1 - Where the wholesale battery's value goes
w = abl.loc["wholesale"]; a = attr.loc["wholesale"]
steps = [("Perfect foresight<br>(Part 7, weekly LP)", w["Part 7 optimiser"], "absolute"),
         ("48 h hourly planning<br>(still perfect)", w["A"] - w["Part 7 optimiser"], "relative"),
         ("PV forecast error", -a["pv"], "relative"),
         ("Price forecast error", -a["price"], "relative"),
         ("Interaction", -a["interaction"], "relative"),
         ("Rolling horizon<br>with forecasts (Part 11)", w["D"], "total")]
fig = go.Figure(go.Waterfall(x=[s[0] for s in steps], y=[s[1] for s in steps], measure=[s[2] for s in steps],
                             text=[f"${v:,.0f}" for v in [s[1] for s in steps]], textposition="outside",
                             decreasing=dict(marker_color="#b92121"), increasing=dict(marker_color="#2ca02c"),
                             totals=dict(marker_color="#1f77b4")))
fig.add_hline(y=VPP_FEE_PER_MONTH * 12, line_dash="dot", annotation_text="VPP fee ($300/yr)")
fig.update_layout(title="Wholesale battery saving (before the VPP fee), 2023: what perfect foresight is worth and where forecasting loses it",
                  yaxis_title="Saving vs no battery ($/yr)", template="plotly_white", height=460, showlegend=False)
fig.show()

> **Finding 12 - PV vs price.** TODO in my words: the saving with perfect information (A, and vs the Part 7 optimiser: the cost of 48 h hourly planning); how much is lost to PV forecast error vs price forecast error on wholesale, and to PV on flat/ToU; which matters more and **why** (wholesale value comes from rare spikes and negative-price days that a daily-pattern forecast cannot see, while the PV forecast is within ~1% of the 'actual' PV, Finding 9); whether the forecast-driven saving covers the $300/yr VPP fee; and the caveat that PV uncertainty is understated because the Year 2 'actual' PV is itself a smooth forecast.

# Part 13 - Sensitivity and Robustness

Parts 7-12 rest on single assumptions. This Part tests the ones that decide the answer:
- **13.1 Wear cost.** The single input that decides whether the optimiser cycles at all on flat and ToU.
- **13.2 Where the value comes from.** How concentrated the wholesale saving is, and a confidence band on the annual figure.
- **13.3 2022 regime test.** The only year with measured PV, so the only year where PV forecast error is real.
- **13.4 A better scheduler.** Whether the rolling-horizon loss in Parts 11-12 can be recovered, run as a new mode beside the original so no earlier result changes.

New results are cached under a key built from the inputs (battery, switches, network windows), so a changed assumption can never reuse a stale file.

In [401]:
def cache_key(*parts) -> str:
    """Short hash of everything a cached result depends on. Change any input -> new file name."""
    return hashlib.md5(json.dumps([repr(p) for p in parts]).encode()).hexdigest()[:8]

def cached(name: str, key: str, compute, rerun: bool = False) -> pd.DataFrame:
    """Load data/results/{name}_{key}.csv if it exists, else compute() and save it."""
    path = ROLL_DIR / f"{name}_{key}.csv"
    if path.exists() and not rerun:
        return pd.read_csv(path)
    out = compute()
    out.to_csv(path, index=False)
    return out

## Task 13.1 - Wear-cost sensitivity

Re-run the Part 7 optimiser (perfect foresight, 2023) across a grid of wear costs, for each tariff. Two reference points are marked:
- **Warranty basis** (Part 5): capex spread over the warranted throughput.
- **Cycle-life basis**: capex spread over the datasheet cycle life (Part 8 `wear_basis="cycle life"`).

In [402]:
WEAR_REF = {"warranty": DEG_COST,
            "cycle life": degradation_cost_per_kwh(sigen_spec(2, wear_basis="cycle life"))}
WEAR_GRID = sorted({0.0, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, *WEAR_REF.values()})   # $/kWh discharged

def wear_sensitivity(load, pv, temp, prices, spec: BatterySpec, wear_grid=WEAR_GRID) -> pd.DataFrame:
    """Perfect-foresight optimiser at each wear cost. Bills are before the VPP fee."""
    rows = []
    for t in TARIFFS:
        imp, exp = prices[f"{t}_imp"], prices[f"{t}_exp"]
        nob = annual_bill(load - pv, imp, exp)
        for w in wear_grid:
            o = optimise_year(load, pv, imp, exp, temp, spec, deg_cost=w, **switches)
            bill = annual_bill(o.import_kW - o.export_kW, imp, exp)
            cyc = o.discharge_kW.sum() * DT_H / spec.usable_kWh
            rows.append(dict(tariff=t, wear=w, bill=bill, saving=nob - bill, cycles=cyc,
                             wear_paid=cyc * spec.usable_kWh * w))
    return pd.DataFrame(rows)

RERUN_SENS = False
sens = cached("wear_sens", cache_key(BATTERY, switches, NETWORK_WINDOWS, WEAR_GRID),
              lambda: wear_sensitivity(load2, pv2, temp2, prices, BATTERY), rerun=RERUN_SENS)

# Checks: more wear cost -> never more cycling; the wear = DEG_COST row reproduces Part 7
for t, g in sens.groupby("tariff"):
    assert (g.sort_values("wear").cycles.diff().dropna() <= 0.5).all(), f"{t}: cycles rose with wear cost"
_p7 = sens[np.isclose(sens.wear, DEG_COST)].set_index("tariff")["bill"]
for t in TARIFFS:
    _fee = VPP_FEE_PER_MONTH * 12 if t == "wholesale" else 0.0
    assert np.isclose(_p7[t] + _fee, results[("Optimised (perfect foresight)", t)]["bill"], atol=1.0), t
sens.pivot(index="wear", columns="tariff", values=["saving", "cycles"]).round(1)

saving                  cycles                 
tariff     flat    tou wholesale   flat    tou wholesale
wear                                                    
0.000000  656.6  583.7    1279.6  172.2  196.4     382.7
0.020000  656.6  583.7    1273.1  172.2  196.4     340.2
0.050000  656.6  583.7    1246.7  172.2  196.4     290.8
0.080000  656.6  583.7    1204.4  172.2  196.4     248.9
0.100000  656.6  454.7    1171.9  172.2  106.3     225.6
0.117647  656.6  454.7    1135.1  172.2  106.3     203.8
0.150000  656.6  454.7    1053.0  172.2  106.3     164.5
0.200000  656.6  308.6     946.1  172.2   58.1     125.3
0.300000    0.0  301.2     670.0    0.0   56.1      55.3
0.386053    0.0    0.0     498.3    0.0    0.0      22.5

In [403]:
# Figure 13.1 - Battery saving and cycling vs assumed wear cost (perfect foresight, 2023)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("Bill saving vs no battery ($/yr, before VPP fee)", "Full cycles per year"))
colours = {"flat": "#1f77b4", "tou": "#ff7f0e", "wholesale": "#2ca02c"}
for t in TARIFFS:
    g = sens[sens.tariff == t].sort_values("wear")
    for c, y in [(1, "saving"), (2, "cycles")]:
        fig.add_trace(go.Scatter(x=g.wear, y=g[y], name=names[t], mode="lines+markers", line_color=colours[t],
                                 legendgroup=t, showlegend=c == 1), 1, c)
for label, w in WEAR_REF.items():
    for c in (1, 2):
        fig.add_vline(x=w, line_dash="dot", line_color="grey", row=1, col=c)
        fig.add_annotation(x=w, y=0.97, xref=f"x{c if c > 1 else ''}", yref=f"y{c if c > 1 else ''} domain",
                           text=f"{label}<br>${w:.3f}/kWh", showarrow=False, font_size=11, bgcolor="white",
                           xanchor="right" if label == "warranty" else "left", xshift=-4 if label == "warranty" else 4)
fig.update_xaxes(title_text="Wear cost ($ per kWh discharged)", range=[-0.01, 0.40])
for c, y in [(1, "saving"), (2, "cycles")]:                       # headroom so the labels sit above the lines
    fig.update_yaxes(range=[-0.05 * sens[y].max(), 1.3 * sens[y].max()], row=1, col=c)
fig.update_layout(height=450, template="plotly_white", margin=dict(t=70),
                  legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"))
fig.show()

> **Finding 13.1 - Wear cost.** TODO in my words: below what wear cost does the optimiser start cycling on each tariff (read off the figure); how far the warranty basis sits from that; what the cycle-life basis would change; why this one assumption explains the zero Part 7 savings on flat and ToU.

## Task 13.2 - Where does the wholesale value come from?

Two questions the single annual number hides:
- **Concentration.** Is the saving spread across the year, or earned on a handful of price-spike days? If it is concentrated, a forecast that misses a few spikes loses most of the value (the Part 12 result).
- **Uncertainty.** How much could the annual saving differ in another year like 2023? Resampling whole weeks (bootstrap) gives a band. Weeks are treated as independent, which ignores season; the band is indicative, not a forecast.

Compared on wholesale: the Part 7 optimiser (perfect foresight, weekly LP), scenario A (rolling horizon, perfect information) and scenario D (rolling horizon, forecasts). All before the VPP fee.

In [404]:
def interval_gain(grid_with: pd.Series, grid_without: pd.Series, imp: pd.Series, exp: pd.Series) -> pd.Series:
    """Saving earned in each interval ($): cost without the battery minus cost with it."""
    cost = lambda g: (g.clip(lower=0) * imp - (-g).clip(lower=0) * exp) * DT_H
    return cost(grid_without) - cost(grid_with)

def bootstrap_annual_saving(weekly_saving: pd.Series, n: int = 5000, seed: int = RNG_SEED) -> np.ndarray:
    """Annual saving from weeks resampled with replacement (same number of weeks as the data)."""
    rng = np.random.default_rng(seed)
    w = weekly_saving.to_numpy()
    return rng.choice(w, size=(n, len(w))).sum(axis=1)

imp_w, exp_w = prices["wholesale_imp"], prices["wholesale_exp"]
grid_nob = load2 - pv2
gain = {
    "Part 7 optimiser": interval_gain(opt["wholesale"].import_kW - opt["wholesale"].export_kW, grid_nob, imp_w, exp_w),
    "A: rolling, perfect info": interval_gain(runs[("A", "wholesale")].grid_kW, grid_nob, imp_w, exp_w),
    "D: rolling, forecasts": interval_gain(runs[("D", "wholesale")].grid_kW, grid_nob, imp_w, exp_w),
}
# Reconciliation: interval gains must add up to the Part 12 savings
assert np.isclose(gain["Part 7 optimiser"].sum(), abl.loc["wholesale", "Part 7 optimiser"], atol=1.0)
assert np.isclose(gain["A: rolling, perfect info"].sum(), abl.loc["wholesale", "A"], atol=1.0)
assert np.isclose(gain["D: rolling, forecasts"].sum(), abl.loc["wholesale", "D"], atol=1.0)

conc = {}
for k, g in gain.items():
    daily = g.resample("D").sum().sort_values(ascending=False)
    top1pct = g.sort_values(ascending=False).iloc[: int(len(g) * 0.01)].sum()
    boot = bootstrap_annual_saving(g.resample("W-MON").sum())
    conc[k] = {"annual saving ($)": g.sum(),
               "top 1% of intervals ($)": top1pct,
               "best 10 days ($)": daily.iloc[:10].sum(),
               "all other days ($)": daily.iloc[10:].sum(),
               "days gaining / losing": f"{(daily > 0).sum()} / {(daily < 0).sum()}",
               "worst day ($)": daily.min(),
               "bootstrap 5% ($)": np.percentile(boot, 5),
               "bootstrap 95% ($)": np.percentile(boot, 95),
               "P(saving > VPP fee)": (boot > VPP_FEE_PER_MONTH * 12).mean()}
conc = pd.DataFrame(conc).T
conc

,annual saving ($),top 1% of intervals ($),best 10 days ($),all other days ($),days gaining / losing,worst day ($),bootstrap 5% ($),bootstrap 95% ($),P(saving > VPP fee)
Part 7 optimiser,498.329584,420.292017,249.543574,248.786011,321 / 15,-0.187954,327.934388,710.373447,0.976
"A: rolling, perfect info",420.799043,378.882377,249.326202,171.472841,294 / 71,-1.125808,254.400858,628.083519,0.866
"D: rolling, forecasts",28.160086,38.497189,16.989917,11.170169,284 / 81,-1.353158,9.603807,49.802678,0.0


In [405]:
# Figure 13.2 - Cumulative wholesale saving, days ranked from best to worst (2023, before the VPP fee)
fig = go.Figure()
for (k, g), colour in zip(gain.items(), ["#7f7f7f", "#2ca02c", "#d62728"]):
    daily = g.resample("D").sum().sort_values(ascending=False)
    fig.add_trace(go.Scatter(x=np.arange(1, len(daily) + 1), y=daily.cumsum().values, name=k, line_color=colour))
fig.add_hline(y=VPP_FEE_PER_MONTH * 12, line_dash="dash", line_color="black",
              annotation_text=f"VPP fee ${VPP_FEE_PER_MONTH * 12:.0f}/yr", annotation_position="bottom right")
fig.update_layout(xaxis_title="Number of days (best first)", yaxis_title="Cumulative saving ($)",
                  height=420, template="plotly_white", legend=dict(orientation="h", y=-0.2))
fig.show()

> **Finding 13.2 - Concentration and uncertainty.** TODO in my words: what share of the saving the best 10 days / top 1% of intervals earn; how scenario D earns on spike days but gives it back on the rest; the bootstrap band and the chance the saving covers the $300 fee; what that means for a VPP that cannot see spikes coming.

## Task 13.3 - 2022 regime test

Everything so far is 2023. 2022 is a different world, and it is the only year where the site's PV is **measured**:
- **Prices.** The 2022 energy crisis (NSW mean spot about $183/MWh vs about $96 in 2023), including the June 2022 market suspension and administered price cap.
- **PV.** Measured PV has real cloud transients, so PV forecast error is real here; in 2023 the 'actual' PV is the smooth commercial forecast (Finding 9). To keep the PV forecast honest, the model is **cross-fitted**: trained on three of four month-groups, predicting the fourth.
- **Inputs.** The 184 missing measured-PV intervals are filled from the Part 9 model so the scheduler has complete inputs. Jan-Jun 2022 uses indicative 2021-22 network rates (Part 0). December 2021 prices give the price forecast its first week of history.

This is a regime test, not a forecast: it asks whether the 2023 conclusions (ranking of tariffs and modes; price, not PV, drives the forecasting loss) survive a very different year. It does not say what the battery will earn in future.

In [406]:
y1 = dfc.year == 1
load1, temp1 = dfc.loc[y1, "load_kW"], dfc.loc[y1, "temp_C"]
pv1 = dfc.loc[y1, "pv_meas_kW"].copy()
pv1_gap = pv1.isna()
pv1[pv1_gap] = predict_pv(dfc.loc[y1, "irr_Wm2"], temp1, PV_COEF)[pv1_gap]        # scheduler needs complete inputs

spot_hist = pd.concat([load_spot_prices("NSW1", "2021-12", "2021-12", PRICE_DIR) / 1000, dfc["rrp_kWh"]])
spot_hist = spot_hist[~spot_hist.index.duplicated(keep="last")].sort_index()     # 1 Jan 2022 00:00 is in both files

w1 = wholesale_retail_price(dfc.loc[y1, "rrp_kWh"])
prices1 = pd.DataFrame({
    "flat_imp": dfc.loc[y1, "flat_imp"], "flat_exp": dfc.loc[y1, "flat_exp"],
    "tou_imp": dfc.loc[y1, "tou_imp"],   "tou_exp": dfc.loc[y1, "tou_exp"],
    "wholesale_imp": w1["import_kWh"],   "wholesale_exp": w1["export_kWh"],
})
YEARS = {2022: dict(load=load1, pv=pv1, temp=temp1, prices=prices1, spot=spot_hist),
         2023: dict(load=load2, pv=pv2, temp=temp2, prices=prices, spot=spot_all)}

# Contract checks: complete inputs, history reaches back far enough for the 7-day price forecast
assert pv1.notna().all() and prices1.notna().all().all() and len(prices1) == y1.sum()
assert spot_hist.index.is_monotonic_increasing and spot_hist.index.min() <= pd.Timestamp("2021-12-24")
print(f"2022: {pv1_gap.sum()} measured-PV intervals ({pv1_gap.mean():.2%}) filled from the PV model")
print(f"Mean spot: 2022 ${dfc.loc[y1, 'rrp_kWh'].mean() * 1000:,.0f}/MWh, 2023 ${spot_y2.mean() * 1000:,.0f}/MWh")

2022: 184 measured-PV intervals (0.18%) filled from the PV model
Mean spot: 2022 $183/MWh, 2023 $96/MWh


In [407]:
def crossfit_pv(df: pd.DataFrame, n_folds: int = 4) -> pd.Series:
    """Out-of-sample PV forecast for Year 1: train on other months, predict the held-out ones.
    Avoids the in-sample optimism of fitting and scoring on the same year."""
    y1 = df[(df.year == 1) & ~df.pv_meas_missing & ~df.irr_filled & ~df.temp_filled]
    folds = (y1.index.month - 1) % n_folds
    out = pd.Series(np.nan, index=df.index[df.year == 1])
    for k in range(n_folds):
        tr, te = y1[folds != k], df[(df.year == 1) & (((df.index.month - 1) % n_folds) == k)]
        coef = fit_pv_model(tr.irr_Wm2, tr.temp_C, tr.pv_meas_kW)
        out.loc[te.index] = predict_pv(te.irr_Wm2, te.temp_C, coef)
    return out

PV_FC1 = crossfit_pv(dfc)
assert PV_FC1.notna().all() and PV_FC1.index.equals(load1.index)

# How good is the PV forecast against REAL measured PV, compared with 2023's smooth 'actual'?
ix1 = dfc.index[y1 & ~dfc.pv_meas_missing & ~dfc.irr_filled & ~dfc.temp_filled & (dfc.irr_Wm2 > 0)]
act1 = dfc.loc[ix1, "pv_meas_kW"]
pv_scores = pd.DataFrame({
    "2022: in-sample model vs measured":     forecast_errors(predict_pv(dfc.loc[ix1, "irr_Wm2"], dfc.loc[ix1, "temp_C"], PV_COEF), act1, PV_KWP_EST),
    "2022: cross-fitted model vs measured":  forecast_errors(PV_FC1[ix1], act1, PV_KWP_EST),
    "2023: model vs commercial 'actual'":    forecast_errors(PV_FORECAST[score_rows], dfc.loc[score_rows, "pv_comm_kW"], PV_KWP_EST),
}).T
pv_scores.round(3)

,MAE,RMSE,nRMSE,MBE
2022: in-sample model vs measured,0.244,0.476,0.053,-0.001
2022: cross-fitted model vs measured,0.244,0.476,0.053,-0.001
2023: model vs commercial 'actual',0.072,0.093,0.010,-0.072


In [408]:
RERUN_2022 = False

def forecasters(inp: dict, t: str, price_method=None):
    """(import, export) forecast functions for one year's inputs. price_method=None: perfect price foresight."""
    pr = inp["prices"]
    if t != "wholesale" or price_method is None:
        return (lambda tau, w: pr.loc[w, f"{t}_imp"]), (lambda tau, w: pr.loc[w, f"{t}_exp"])
    fc = lambda tau, w: wholesale_retail_price(price_method(inp["spot"], tau, w))
    return (lambda tau, w: fc(tau, w)["import_kWh"]), (lambda tau, w: fc(tau, w)["export_kWh"])

def rolling_year(year: int, t: str, pv_fc: pd.Series, price_method, tag: str, rerun: bool = False,
                 key_extra=None, **hooks) -> pd.DataFrame:
    """Part 11 scheduler on any year's inputs, cached under a hash of everything it depends on.
    Hooks (Task 13.4) are keyed by function name: set rerun=True after editing a hook's code."""
    inp = YEARS[year]
    key = cache_key(year, t, tag, BATTERY, switches, DEG_COST, NETWORK_WINDOWS, NETWORK_RATES,
                    getattr(price_method, "__name__", None), sorted((k, v.__name__) for k, v in hooks.items()),
                    round(float(pv_fc.sum()), 1), key_extra)
    path = ROLL_DIR / f"rolling_{year}_{tag}_{t}_{key}.csv"
    if path.exists() and not rerun:
        return pd.read_csv(path, index_col=0, parse_dates=True)
    imp_fn, exp_fn = forecasters(inp, t, price_method)
    out = rolling_horizon(inp["load"], inp["pv"], pv_fc, inp["prices"][f"{t}_imp"], inp["prices"][f"{t}_exp"],
                          imp_fn, exp_fn, inp["temp"], BATTERY, deg_cost=DEG_COST, **switches, **hooks)
    out.to_csv(path)
    return out

# Check: the generic runner (and the refactored Part 11 controller) reproduces Part 11 exactly
_d23 = rolling_year(2023, "wholesale", PV_FORECAST, PRICE_METHOD, "forecast", rerun=RERUN_2022)
assert np.allclose(_d23.grid_kW.to_numpy(), roll["wholesale"].grid_kW.to_numpy(), atol=1e-6), "does not reproduce Part 11"
print("Generic runner reproduces the Part 11 wholesale run")

Generic runner reproduces the Part 11 wholesale run


In [409]:
def year_scenarios(year: int, pv_fc: pd.Series) -> pd.DataFrame:
    """Benchmarks, Part 7 optimiser and Part 12 scenarios A-D for one year. Savings vs no battery, before the VPP fee."""
    inp = YEARS[year]
    L, P, T, pr = inp["load"], inp["pv"], inp["temp"], inp["prices"]
    grid_sc = L - P - self_consumption_dispatch(L, P, T, BATTERY)
    rows = {}
    for t in TARIFFS:
        imp, exp = pr[f"{t}_imp"], pr[f"{t}_exp"]
        nob = annual_bill(L - P, imp, exp)
        o = optimise_year(L, P, imp, exp, T, BATTERY, deg_cost=DEG_COST, **switches)
        A = rolling_year(year, t, P, None, "perfect", rerun=RERUN_2022)
        if t == "wholesale":
            B = rolling_year(year, t, pv_fc, None, "pvfc", rerun=RERUN_2022)
            C = rolling_year(year, t, P, PRICE_METHOD, "pricefc", rerun=RERUN_2022)
            D = rolling_year(year, t, pv_fc, PRICE_METHOD, "forecast", rerun=RERUN_2022)
        else:                                                    # prices known: B = D and C = A (as in Part 12)
            D = rolling_year(year, t, pv_fc, None, "forecast", rerun=RERUN_2022)
            B, C = D, A
        s = {k: battery_saving(nob, r, imp, exp) for k, r in zip("ABCD", (A, B, C, D))}
        rows[t] = {"no-battery bill": nob,
                   "self-consumption": nob - annual_bill(grid_sc, imp, exp),
                   "Part 7 optimiser": nob - annual_bill(o.import_kW - o.export_kW, imp, exp),
                   **s,
                   **{f"lost: {k}": v for k, v in attribute_loss(s["A"], s["B"], s["C"], s["D"]).items()},
                   "cycles (D)": D.batt_kW.clip(lower=0).sum() * DT_H / BATTERY.usable_kWh}
    return pd.DataFrame(rows).T

r22 = year_scenarios(2022, PV_FC1)

# 2023 from the earlier Parts (no re-run)
r23 = pd.DataFrame({t: {"no-battery bill": results[("No battery", t)]["bill"],
                        "self-consumption": results[("No battery", t)]["bill"] - results[("Self-consumption", t)]["bill"],
                        "Part 7 optimiser": abl.loc[t, "Part 7 optimiser"],
                        **abl.loc[t, ["A", "B", "C", "D"]].to_dict(),
                        **{f"lost: {k}": attr.loc[t, k] for k in attr.columns},
                        "cycles (D)": runs[("D", t)].batt_kW.clip(lower=0).sum() * DT_H / BATTERY.usable_kWh}
                    for t in TARIFFS}).T
regime = pd.concat({"2022": r22, "2023": r23}).swaplevel().sort_index(level=0, sort_remaining=False)
regime.round(0)

no-battery bill  self-consumption  Part 7 optimiser      A      B     C     D  lost: total  lost: pv  lost: price  lost: interaction  cycles (D)
flat      2022           1084.0             636.0               0.0    0.0    0.0   0.0   0.0          0.0       0.0          0.0                0.0         0.0
          2023            822.0             666.0               0.0    0.0    0.0   0.0   0.0          0.0       0.0          0.0                0.0         0.0
tou       2022            831.0             505.0               0.0    1.0    1.0   1.0   1.0         -0.0      -0.0          0.0                0.0         0.0
          2023            606.0             513.0               0.0    1.0    1.0   1.0   1.0         -0.0      -0.0          0.0                0.0         0.0
wholesale 2022           1134.0             553.0             554.0  438.0  438.0  69.0  70.0        368.0       0.0        369.0               -1.0        14.0
          2023            649.0             552.0             498.0  421.0  422.0  28.0  28.0        393.0      -1.0        393.0                1.0         6.0

In [410]:
# Figure 13.3 - Wholesale battery saving by scenario: 2022 vs 2023 (before the VPP fee)
cols = ["self-consumption", "Part 7 optimiser", "A", "B", "C", "D"]
labels = ["Self-consumption", "Part 7 optimiser", "A: perfect info", "B: PV forecast", "C: price forecast", "D: both forecasts"]
fig = go.Figure()
for yr, colour in [("2022", "#9467bd"), ("2023", "#2ca02c")]:
    v = regime.loc[("wholesale", yr), cols]
    fig.add_trace(go.Bar(x=labels, y=v.values, name=yr, marker_color=colour,
                         text=[f"${x:,.0f}" for x in v.values], textposition="outside"))
fig.add_hline(y=VPP_FEE_PER_MONTH * 12, line_dash="dash", line_color="black",
              annotation_text=f"VPP fee ${VPP_FEE_PER_MONTH * 12:.0f}/yr", annotation_position="top right")
fig.update_layout(barmode="group", yaxis_title="Saving vs no battery ($/yr)", height=450, template="plotly_white",
                  legend=dict(orientation="h", y=-0.2))
fig.show()

> **Finding 13.3 - 2022 regime test.** TODO in my words: how much worse the PV forecast is against real measured PV (and cross-fitted vs in-sample); whether the PV share of the forecasting loss is still small with real PV error; whether the ranking of tariffs and modes holds; how much more a battery earns in a crisis year and why that is not a forecast of future value.

## Task 13.4 - Can the scheduler recover the lost value?

Part 12 showed that almost all the wholesale value lost to forecasting comes from the price forecast. Four changes are tested on 2023 wholesale (forecast PV and prices, as scenario D), each alone and together, and run beside the Part 11 controller so no earlier result changes:
1. **Mean price forecast.** The LP minimises expected cost, so it needs the expected price. A median ignores spikes by construction; the mean of the same 7 days does not.
2. **Terminal value.** Replace the rule "end each 48 h plan at least at 50%" with a value per stored kWh, so the plan neither hoards nor dumps energy at the horizon.
3. **Fallback to self-consumption.** If the plan expects to gain less than a margin over the horizon, run the battery as self-consumption for that hour (how most commercial VPPs operate between events).
4. **Rule-based benchmark.** Self-consumption plus full-power export in the evening peak, with no forecasts at all: the LP should beat something smarter than "no battery".

Results are shown in cash (what the bill says) and net of wear (warranty basis), because change 3 trades wear for cash.

In [411]:
def profile_mean_forecast(spot: pd.Series, issue_time: pd.Timestamp, targets: pd.DatetimeIndex, days: int = 7) -> pd.Series:
    """Spot price forecast = MEAN of the prices at the same time of day on the `days` most recent known days.
    """
    k = days_back(targets, issue_time)

    cols = []
    for j in range(days):
        src_times = targets - pd.to_timedelta(k + j, unit = "D")
        cols.append(spot.reindex(src_times).to_numpy())

    hist = pd.DataFrame(np.column_stack(cols), index = targets)
    return hist.mean(axis = 1, skipna = True)


def terminal_value(imp_h: np.ndarray, exp_h: np.ndarray) -> float:
    """$ per kWh credited for energy left in the battery at the end of a 48 h plan."""
    best_price = np.maximum(imp_h, exp_h)
    val = float(np.median(best_price)) * BATTERY.eta_dis - DEG_COST
    return max(0.0, val)


def plan_gain(plan: pd.DataFrame, net_h: np.ndarray, imp_h: np.ndarray, exp_h: np.ndarray, e0: float, tv: float) -> float:
    """Expected $ gain of a 48 h plan over no battery, on the forecasts (hourly steps, so kW x 1 h = kWh).
    no battery: imports max(net, 0) at imp_h, exports max(-net, 0) at exp_h.
    plan:       imports plan.import_kW at max(imp_h, exp_h) (the LP's convexity fix, Task 7.1), exports plan.export_kW
                at exp_h, pays DEG_COST per kWh discharged, and is credited tv per kWh added to storage (end minus e0).
    gain = cost(no battery) - cost(plan)."""

    no_batt_cost = float((np.maximum(net_h, 0.0) * imp_h - np.maximum(-net_h, 0.0) * exp_h).sum())

    plan_imp_cost = float((plan["import_kW"].to_numpy() * np.maximum(imp_h, exp_h)).sum())
    plan_exp_rev = float((plan["export_kW"].to_numpy() * exp_h).sum())
    wear_cost = float(plan["discharge_kW"].to_numpy().sum()) * DEG_COST
    e_end = float(plan["energy_kWh"].iloc[-1])
    storage_credit = tv * (e_end - e0)

    plan_cost = plan_imp_cost - plan_exp_rev + wear_cost - storage_credit
    return float(no_batt_cost - plan_cost)




def evening_export_rule(load: pd.Series, pv: pd.Series, temp_C: pd.Series, spec: BatterySpec,
                        start_h: int = 17, end_h: int = 20, reserve: float = 0.2) -> pd.Series:
    """Rule-based wholesale benchmark (no forecasts). Battery kW (+ discharge).
    Self-consumption all day (store surplus, cover deficit); from start_h to end_h, discharge at full (temperature-limited)
    power, exporting what the house does not use, but only while stored energy is above reserve x usable."""

    factor = power_limit_factor(temp_C, spec).to_numpy()
    net = (load - pv).to_numpy()
    hours = load.index.hour
    e = spec.soc_init * spec.usable_kWh
    e_min, e_max = spec.e_min, spec.e_max
    e_reserve = reserve * spec.usable_kWh
    p = np.zeros(len(net))

    for k in range(len(net)):
        in_window = start_h <= hours[k] < end_h
        if in_window:
            # evening peak: full power, but never below the reserve (at the reserve: stay idle)
            avail_above_reserve = max(0.0, e - e_reserve)
            max_energy_dis_kW = avail_above_reserve * spec.eta_dis / DT_H
            dis_kW = min(spec.p_discharge_kW * factor[k], max_energy_dis_kW)
            p[k] = dis_kW
            e -= (dis_kW / spec.eta_dis) * DT_H
        elif net[k] < 0:
            surplus_kW = -net[k]
            room_kWh = max(0.0, e_max - e)
            charge_limit_kW = spec.p_charge_kW * factor[k]
            max_energy_ch_kW = room_kWh / (spec.eta_ch * DT_H)
            charge_kW = min(surplus_kW, charge_limit_kW, max_energy_ch_kW)
            p[k] = -charge_kW
            e += spec.eta_ch * charge_kW * DT_H
        elif net[k] > 0:
            deficit_kW = net[k]
            avail_kWh = max(0.0, e - e_min)
            discharge_limit_kW = spec.p_discharge_kW * factor[k]
            max_energy_dis_kW = avail_kWh * spec.eta_dis / DT_H
            discharge_kW = min(deficit_kW, discharge_limit_kW, max_energy_dis_kW)
            p[k] = discharge_kW
            e -= (discharge_kW / spec.eta_dis) * DT_H

    return pd.Series(p, index=load.index)


FALLBACK_MARGIN = 0.10   # provisional; chosen from the margin sweep after the tests

def fallback(plan, net_h, imp_h, exp_h, e0, tv) -> bool:
    """Hook for rolling_horizon: True -> run self-consumption this hour instead of the plan."""
    return plan_gain(plan, net_h, imp_h, exp_h, e0, tv) < FALLBACK_MARGIN

In [412]:
"""Tests for Task 13.4 (hand-calculated)"""
# 1) mean forecast: price = day number; issued at day 9 00:00 for day 9 12:00 -> mean of days 8..2 = 5
_i = pd.date_range("2023-01-01", "2023-01-10 23:55", freq="5min")
_sp = pd.Series((_i - _i[0]).days.astype(float), index=_i)
_tau, _tg = pd.Timestamp("2023-01-10 00:00"), pd.DatetimeIndex(["2023-01-10 12:00"])
assert np.isclose(profile_mean_forecast(_sp, _tau, _tg).iloc[0], 5.0)
_sp2 = _sp.copy(); _sp2[_sp2.index.day == 8] += 70                          # one spike day: mean moves, median does not
assert profile_mean_forecast(_sp2, _tau, _tg).iloc[0] > profile_median_forecast(_sp2, _tau, _tg).iloc[0]
_fut = _sp.copy(); _fut[_fut.index >= _tau] = 999                            # no look-ahead
assert profile_mean_forecast(_fut, _tau, _tg).equals(profile_mean_forecast(_sp, _tau, _tg))

# 2) terminal value: never negative, never above the best price x eta_dis, rises with prices, zero if prices < wear
_imp, _exp = np.full(48, 0.30), np.full(48, 0.05)
_tv = terminal_value(_imp, _exp)
assert 0 <= _tv <= 0.30 * BATTERY.eta_dis + 1e-12
assert terminal_value(2 * _imp, 2 * _exp) >= _tv
assert terminal_value(np.full(48, DEG_COST / 10), np.zeros(48)) == 0

# 3) plan gain: an idle plan gains nothing; a hand case
_net = np.array([1.0, 1.0]); _pi = np.array([0.10, 0.50]); _pe = np.array([0.0, 0.0])
_idle = pd.DataFrame({"charge_kW": 0.0, "discharge_kW": 0.0, "import_kW": _net, "export_kW": 0.0, "energy_kWh": 5.0}, index=[0, 1])
assert np.isclose(plan_gain(_idle, _net, _pi, _pe, e0=5.0, tv=0.2), 0.0)
_shift = pd.DataFrame({"charge_kW": [1.0, 0.0], "discharge_kW": [0.0, 1.0], "import_kW": [2.0, 0.0],
                       "export_kW": 0.0, "energy_kWh": [6.0, 5.0]}, index=[0, 1])
assert np.isclose(plan_gain(_shift, _net, _pi, _pe, e0=5.0, tv=0.0), 0.6 - 0.2 - DEG_COST)          # 0.4 less wear
_keep = _shift.assign(discharge_kW=0.0, import_kW=[2.0, 1.0], energy_kWh=[6.0, 6.0])               # charge and hold
assert np.isclose(plan_gain(_keep, _net, _pi, _pe, e0=5.0, tv=0.3), 0.6 - 0.7 + 0.3)

# 4) evening rule: one synthetic day
_j = pd.date_range("2023-01-02", periods=288, freq="5min"); _hh = np.asarray(_j.hour + _j.minute / 60)
_ld = pd.Series(0.5, index=_j); _pvv = pd.Series(np.clip(4 * np.sin(np.pi * (_hh - 6) / 12), 0, None), index=_j)
_sb = BatterySpec("t", usable_kWh=5, p_charge_kW=3, p_discharge_kW=3, rte=0.81, soc_init=0.5)
_b = evening_export_rule(_ld, _pvv, pd.Series(20.0, index=_j), _sb)
_e = simulate_soc(_b, _sb)
assert _e.min() >= _sb.e_min - 1e-9 and _e.max() <= _sb.e_max + 1e-9
assert ((-_b).clip(lower=0) <= (_pvv - _ld).clip(lower=0) + 1e-9).all()          # charges only from surplus PV
win = (_hh >= 17) & (_hh < 20)
assert np.isclose(_b[win].iloc[0], 3.0)                                            # full power at the start of the window
assert (_e[win] >= 0.2 * 5 - 1e-6).all()                                           # never below the reserve in the window
assert (_b[~win] <= (_ld - _pvv).clip(lower=0)[~win] + 1e-9).all()                       # outside the window: no export
print("Task 13.4 tests passed")

Task 13.4 tests passed


### Choosing the fallback margin

The margin is how much the 48 h plan must expect to gain over no battery (on the forecasts) before it is allowed to override self-consumption for the hour. Too low and the battery follows noisy forecasts; too high and it is always self-consumption. The sweep below runs change 3 at several margins (2023 wholesale, forecasts) to see whether the choice matters.

In [413]:
RERUN_V2 = False      # set True after changing any Task 13.4 function (cache is keyed by function names); used below too
MARGIN_GRID = [0.0, 0.10, 0.50, 2.00, 5.00]   # $ per 48 h plan

def make_fallback(m: float):
    """Fallback hook with margin m; named by its margin so each margin gets its own cache file."""
    def hook(plan, net_h, imp_h, exp_h, e0, tv):
        return plan_gain(plan, net_h, imp_h, exp_h, e0, tv) < m
    hook.__name__ = f"fallback_{m:g}"
    return hook

fee_w = VPP_FEE_PER_MONTH * 12
rows = {}
for m in MARGIN_GRID:
    r = rolling_year(2023, "wholesale", PV_FORECAST, PRICE_METHOD, f"v2_fb{m:g}", rerun=RERUN_V2,
                     key_extra=m, fallback_fn=make_fallback(m))
    s = results[("No battery", "wholesale")]["bill"] - annual_bill(r.grid_kW, prices["wholesale_imp"], prices["wholesale_exp"])
    cyc = r.batt_kW.clip(lower=0).sum() * DT_H / BATTERY.usable_kWh
    rows[m] = {"saving before fee": s, "cash saving (after fee)": s - fee_w, "cycles": cyc,
               "net of wear": s - fee_w - cyc * BATTERY.usable_kWh * DEG_COST}
    print(f"margin ${m:g}: done")
margin_sweep = pd.DataFrame(rows).T
margin_sweep.index.name = "fallback margin ($ per 48 h)"
margin_sweep.round(0)

margin $0: done
margin $0.1: done
margin $0.5: done
margin $2: done
margin $5: done


,saving before fee,cash saving (after fee),cycles,net of wear
fallback margin ($ per 48 h),,,,
0.0,146.0,-154.0,45.0,-426.0
0.1,441.0,141.0,143.0,-718.0
0.5,523.0,223.0,167.0,-780.0
2.0,547.0,247.0,175.0,-806.0
5.0,547.0,247.0,175.0,-806.0


In [414]:
FALLBACK_MARGIN = 0.10   # Illustrative. The sweep shows the margin trades cash for wear: higher margins move the
                         # battery towards self-consumption (more cash, more cycles). Net of wear, no margin beats
                         # the Part 11 controller without fallback (-$306), so the fallback only pays if wear is cheap

In [415]:
V2 = {                # label: (price method, hooks)
    "D: Part 11 controller":    (PRICE_METHOD, {}),
    "1: mean price forecast":   (profile_mean_forecast, {}),
    "2: terminal value":        (PRICE_METHOD, dict(terminal_value_fn=terminal_value)),
    "3: fallback":              (PRICE_METHOD, dict(fallback_fn=fallback)),
    "1+2+3":                    (profile_mean_forecast, dict(terminal_value_fn=terminal_value, fallback_fn=fallback)),
}
v2 = {}
for k, (pm, hooks) in V2.items():
    tag = "forecast" if not hooks and pm is PRICE_METHOD else "v2_" + k.split(":")[0].replace("+", "")
    v2[k] = rolling_year(2023, "wholesale", PV_FORECAST, pm, tag, rerun=RERUN_V2 and tag != "forecast",
                         key_extra=FALLBACK_MARGIN if "fallback_fn" in hooks else None, **hooks)
    print(f"{k:28s} done")

imp_w, exp_w = prices["wholesale_imp"], prices["wholesale_exp"]
nob_w = results[("No battery", "wholesale")]["bill"]
batt_rule = evening_export_rule(load2, pv2, temp2, BATTERY)
strategies = {**{k: (r.grid_kW, r.batt_kW, True) for k, r in v2.items()},
              "4: evening export rule":         (load2 - pv2 - batt_rule, batt_rule, True),
              "Self-consumption (household)":   (grid_sc, batt_sc, False),
              "A: perfect information":         (runs[("A", "wholesale")].grid_kW, runs[("A", "wholesale")].batt_kW, True)}
rows = {}
for k, (g, b, vpp) in strategies.items():
    saving = nob_w - annual_bill(g, imp_w, exp_w)
    fee = VPP_FEE_PER_MONTH * 12 if vpp else 0.0
    cyc = b.clip(lower=0).sum() * DT_H / BATTERY.usable_kWh
    rows[k] = {"saving before fee": saving, "VPP fee": fee, "cash saving": saving - fee, "cycles": cyc,
               "wear (warranty basis)": cyc * BATTERY.usable_kWh * DEG_COST,
               "net of wear": saving - fee - cyc * BATTERY.usable_kWh * DEG_COST}
v2tab = pd.DataFrame(rows).T
assert np.isclose(v2tab.loc["D: Part 11 controller", "saving before fee"], abl.loc["wholesale", "D"], atol=1.0)
v2tab.round(0)

D: Part 11 controller        done
1: mean price forecast       done
2: terminal value            done
3: fallback                  done
1+2+3                        done


,saving before fee,VPP fee,cash saving,cycles,wear (warranty basis),net of wear
D: Part 11 controller,28.0,300.0,-272.0,6.0,35.0,-306.0
1: mean price forecast,171.0,300.0,-129.0,26.0,157.0,-285.0
2: terminal value,28.0,300.0,-272.0,5.0,33.0,-305.0
3: fallback,441.0,300.0,141.0,143.0,859.0,-718.0
1+2+3,507.0,300.0,207.0,134.0,810.0,-603.0
4: evening export rule,795.0,300.0,495.0,252.0,1517.0,-1022.0
Self-consumption (household),552.0,0.0,552.0,175.0,1053.0,-501.0
A: perfect information,421.0,300.0,121.0,22.0,130.0,-10.0


In [416]:
# Figure 13.4 - Wholesale strategies on 2023: cash saving and saving net of wear (after the VPP fee where it applies)
order = v2tab.index.tolist()
fig = go.Figure()
for col, colour in [("cash saving", "#1f77b4"), ("net of wear", "#d62728")]:
    fig.add_trace(go.Bar(x=order, y=v2tab[col], name=col.capitalize(), marker_color=colour,
                         text=[f"${x:,.0f}" for x in v2tab[col]], textposition="outside"))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(barmode="group", yaxis_title="$/yr vs no battery", height=480, template="plotly_white",
                  legend=dict(orientation="h", y=-0.35), xaxis_tickangle=-25)
fig.show()

> **Finding 13.4 - Improving the scheduler.** TODO in my words: which change recovers most of the lost value and why (expected vs median price); whether any version beats the evening rule and self-consumption; the cash vs net-of-wear trade-off of the fallback; what a VPP would need (better spike forecasts, pre-dispatch) to make the fee worthwhile.

In [417]:
load_spot_prices("NSW1", "2021-12", "2021-12", PRICE_DIR).describe()

count    8928.000000
mean       67.960395
std        40.430759
min     -1000.000000
25%        55.010000
50%        65.030000
75%        78.070000
max      1175.710000
Name: rrp_MWh, dtype: float64

# Part 14 - Techno-Economics

Everything so far is an annual bill. A purchase decision needs the whole life of the battery: discounted cash flows, NPV, IRR and payback, then the break-even numbers an executive would ask for.

**Cash basis.** Payback uses the cash bill saving, including the VPP fee where VPP-SPREE runs the battery. Wear is already priced inside the optimiser as a shadow cost, so subtracting it again would double-count; the "net of wear" column is shown separately as an *is cycling worth it* view.

**Capacity fade.** Savings fall linearly to `eol_capacity` (70%) by the end of the horizon. At this site's low cycle counts ageing is calendar-driven, so this is a pessimistic simplification.

## Task 14.1 - Discounted cash flow, NPV, IRR and payback

In [ ]:
DISCOUNT_RATE = 0.07   # real, pre-tax: Oxford Economics Australia (Dec 2024), Discount Rates for Energy
                       # Infrastructure, for AEMO's 2026 ISP; a utility-investor rate, so conservative for a household

HORIZON_YEARS = 10   # battery warranty: 70% capacity at 10 years or 23.77 MWh/module, whichever first (Sigenergy
                     # warranty); PV is a sunk cost, so the battery's life sets the horizon; 15 years in Task 14.3

SAVING_ESCALATION = 0.0   # AEMC Residential Electricity Price Trends 2025: NSW residential prices per unit
                          # -0.6%/yr real over 2026-2035, so flat real prices (slightly generous to the battery)

def cashflows(capex: float, annual_saving: float, spec: BatterySpec, years: int = HORIZON_YEARS,
              rate: float = DISCOUNT_RATE, escalation: float = SAVING_ESCALATION) -> pd.DataFrame:
    """Year-by-year cash flow. Saving scaled by remaining capacity (linear fade to spec.eol_capacity at `years`)."""
    t = np.arange(1, years + 1)
    fade = 1 - (1 - spec.eol_capacity) * t / years
    cf = annual_saving * (1 + escalation) ** t * fade
    disc = cf / (1 + rate) ** t
    return pd.DataFrame({"year": t, "cashflow": cf, "discounted": disc, "cum_discounted": -capex + np.cumsum(disc)})

def discounted_payback(cf: pd.DataFrame) -> float:
    """First (interpolated) year the cumulative discounted cash flow turns positive; inf if never."""
    cum = cf["cum_discounted"].to_numpy()
    if cum[-1] < 0:
        return np.inf
    k = int(np.argmax(cum >= 0))
    before = cum[k - 1] if k > 0 else cum[0] - cf["discounted"].iloc[0]   # = -capex at year 0
    return float(k + (-before / cf["discounted"].iloc[k]))

def npv_irr(capex: float, annual_saving: float, spec: BatterySpec, **kw) -> dict:
    cf = cashflows(capex, annual_saving, spec, **kw)
    f = lambda r: -capex + (cf["cashflow"].to_numpy() / (1 + r) ** cf["year"].to_numpy()).sum()
    try:
        irr = brentq(f, -0.99, 1.0)
    except ValueError:
        irr = np.nan                       # no sign change: savings never come close to capex
    return dict(npv=float(cf["cum_discounted"].iloc[-1]), irr=irr, disc_payback=discounted_payback(cf))

In [419]:
"""Tests for Task 14.1 (hand-calculated)"""
_nofade = replace(BATTERY, eol_capacity=1.0)
_cf = cashflows(1000, 250, _nofade, years=10, rate=0.0)
assert np.isclose(_cf.cum_discounted.iloc[-1], 1500)                       # 10 x 250 - 1000
assert np.isclose(discounted_payback(_cf), 4.0)                            # exactly 4 years
assert np.isclose(discounted_payback(cashflows(1000, 400, _nofade, years=10, rate=0.0)), 2.5)
assert discounted_payback(cashflows(1000, 50, _nofade, years=10, rate=0.0)) == np.inf
_r = npv_irr(1000, 250, _nofade, years=10, rate=0.0)
assert np.isclose(npv_irr(1000, 250, _nofade, years=10, rate=_r["irr"])["npv"], 0, atol=1e-6)   # NPV = 0 at the IRR
assert npv_irr(1000, 50, _nofade, years=10)["irr"] < 0                     # never repays: negative IRR
assert np.isclose(cashflows(1000, 100, BATTERY, years=10, rate=0).cashflow.iloc[-1], 100 * BATTERY.eol_capacity)
print("Task 14.1 tests passed")

Task 14.1 tests passed


## Task 14.2 - Master results table

One row per operating mode and tariff (2023, Sigenergy 15.6 kWh, capex before rebate). `saving` is the cash saving after the VPP fee; `saving_before_fee` adds the fee back; `net_saving` subtracts wear at the warranty basis.

In [420]:
MODES = ["No battery", "Self-consumption", "Optimised (perfect foresight)", "Rolling horizon (forecasts)"]
CAPEX = {"no rebate": BATTERY.capex_aud, "with rebate": sigen_spec(2, rebate=True).capex_aud}

def master_table(results: dict, spec: BatterySpec = BATTERY, capex: float = None) -> pd.DataFrame:
    capex = spec.capex_aud if capex is None else capex
    tab = with_wear(results_table(results), degradation_cost_per_kwh(spec))
    tab["vpp_fee"] = [VPP_FEE_PER_MONTH * 12 if (m.startswith(("Optimised", "Rolling")) and t == "wholesale") else 0.0
                      for m, t in tab.index]
    tab["saving_before_fee"] = tab["saving"] + tab["vpp_fee"]
    econ = [npv_irr(capex, s, spec) if s > 0 else dict(npv=-capex, irr=np.nan, disc_payback=np.inf)
            for s in tab["saving"]]
    tab = tab.join(pd.DataFrame(econ, index=tab.index))
    tab["simple_payback"] = [simple_payback(capex, s) for s in tab["saving"]]
    tab.loc["No battery", ["npv", "irr", "disc_payback", "simple_payback"]] = np.nan   # no investment to evaluate
    return tab.loc[[m for m in MODES if m in tab.index.get_level_values("mode")]]

master = master_table(results)

# Reconciliation: these catch stale caches and stale variables
assert np.isclose(master.loc[("No battery", "flat"), "bill"], results[("No battery", "flat")]["bill"])
assert np.isclose(master.loc[("Rolling horizon (forecasts)", "wholesale"), "saving_before_fee"],
                  abl.loc["wholesale", "D"], atol=1.0), "Part 14 disagrees with Part 12"
assert np.isclose(master.loc[("Optimised (perfect foresight)", "wholesale"), "saving_before_fee"],
                  abl.loc["wholesale", "Part 7 optimiser"], atol=1.0), "Part 14 disagrees with Part 7"
assert master["self_consumption"].between(0, 1).all()

cols = ["bill", "vpp_fee", "saving", "saving_before_fee", "net_saving", "self_consumption", "self_sufficiency",
        "cycles", "npv", "irr", "disc_payback", "simple_payback"]
master[cols].style.format({**{c: "${:,.0f}" for c in ["bill", "vpp_fee", "saving", "saving_before_fee", "net_saving", "npv"]},
                           "self_consumption": "{:.0%}", "self_sufficiency": "{:.0%}", "cycles": "{:.0f}",
                           "irr": "{:.1%}", "disc_payback": "{:.1f}", "simple_payback": "{:.1f}"}, na_rep="-")

In [421]:
# Figure 14.1 - Cumulative discounted cash flow for each battery mode on its cheapest tariff
best = master.drop("No battery", level="mode")["bill"].groupby(level="mode").idxmin()   # cheapest tariff per mode
fig = go.Figure()
for mode, (m, t) in best.items():
    for basis, cx in CAPEX.items():
        cf = cashflows(cx, master.loc[(m, t), "saving"], BATTERY)
        fig.add_trace(go.Scatter(x=np.r_[0, cf.year], y=np.r_[-cx, cf.cum_discounted], mode="lines+markers",
                                 name=f"{m} - {names[t]} ({basis})", line_dash="solid" if basis == "no rebate" else "dot"))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(title=f"Cumulative discounted cash flow ({DISCOUNT_RATE:.0%} discount rate, {HORIZON_YEARS} years)",
                  xaxis_title="Year", yaxis_title="AUD", height=450, template="plotly_white",
                  legend=dict(orientation="h", y=-0.25))
fig.show()

## Task 14.3 - Sensitivity: what moves the NPV most?

Vary one input at a time for one case and record the NPV. Economic inputs (capex, rebate, discount rate, horizon, escalation, VPP fee) need no re-optimisation; battery and network inputs (efficiency, cold derating, wear basis, network windows) re-run the dispatch. Rolling-horizon cases are not re-run here (about 10 minutes per run).

In [ ]:
TORNADO_CASES = [("Self-consumption", "tou"), ("Optimised (perfect foresight)", "wholesale")]   # the household's cheapest own-battery
                # option and VPP-SPREE's best case; rolling-horizon cases are not re-run (about 10 min per bar)

def case_saving(case, spec: BatterySpec = BATTERY, windows: str = NETWORK_WINDOWS) -> float:
    """Re-run one case's dispatch with a changed battery or network windows. Cash saving $/yr, BEFORE any VPP fee."""
    mode, t = case
    imp, exp = prices[f"{t}_imp"], prices[f"{t}_exp"]
    if t == "wholesale" and windows != NETWORK_WINDOWS:
        w = wholesale_retail_price(dfc.loc[y2, "rrp_kWh"], windows)
        imp, exp = w["import_kWh"], w["export_kWh"]
    if mode == "Self-consumption":
        grid = load2 - pv2 - self_consumption_dispatch(load2, pv2, temp2, spec)
    elif mode.startswith("Optimised"):
        wear = degradation_cost_per_kwh(spec) if USE_DEGRADATION_COST else 0.0
        o = optimise_year(load2, pv2, imp, exp, temp2, spec, deg_cost=wear, **switches)
        grid = o.import_kW - o.export_kW
    else:
        raise ValueError(f"{mode}: not re-run in the tornado")
    return float(annual_bill(load2 - pv2, imp, exp) - annual_bill(grid, imp, exp))

def tornado(case) -> pd.DataFrame:
    mode, t = case
    vpp = mode.startswith(("Optimised", "Rolling")) and t == "wholesale"
    fee = VPP_FEE_PER_MONTH * 12 if vpp else 0.0
    s0 = case_saving(case)
    assert np.isclose(s0 - fee, master.loc[case, "saving"], atol=1.0), "re-run does not reproduce the master table"
    npv = lambda saving=s0 - fee, capex=BATTERY.capex_aud, **kw: npv_irr(capex, saving, BATTERY, **kw)["npv"]
    rerun = lambda **kw: case_saving(case, **kw) - fee
    rows = {
        "Capex -25% / +25%":            (npv(capex=0.75 * BATTERY.capex_aud), npv(capex=1.25 * BATTERY.capex_aud)),
        "Rebate on / off":              (npv(capex=CAPEX["with rebate"]), npv()),
        "Discount rate 3% / 9.5%":      (npv(rate=0.03), npv(rate=0.095)),
        "Horizon 15 / 10 years":        (npv(years=15), npv()),
        "Escalation +2% / 0% per year": (npv(escalation=0.02), npv()),
        "Round trip 0.95 / 0.85":       (npv(rerun(spec=replace(BATTERY, rte=0.95))), npv(rerun(spec=replace(BATTERY, rte=0.85)))),
        "Cold derating off / on":       (npv(rerun(spec=replace(BATTERY, power_frac_outside=1.0))), npv()),
    }
    if mode.startswith("Optimised"):
        cl = sigen_spec(2, wear_basis="cycle life")
        rows["Wear basis cycle life / warranty"] = (npv(rerun(spec=cl)), npv())
    if t == "wholesale":
        other = "essential" if NETWORK_WINDOWS == "assignment" else "assignment"
        rows[f"Network windows {other} / {NETWORK_WINDOWS}"] = (npv(rerun(windows=other)), npv())
    if vpp:
        rows["VPP fee $0 / $40 per month"] = (npv(s0), npv(s0 - 40 * 12))
    out = pd.DataFrame(rows, index=["npv_a", "npv_b"]).T
    out["base"] = npv()
    out["swing"] = (out.npv_a - out.npv_b).abs()
    return out.sort_values("swing")

torn = {case: tornado(case) for case in TORNADO_CASES}
pd.concat({f"{m} - {t}": d for (m, t), d in torn.items()}).round(0)

npv_a    npv_b     base   swing
Self-consumption - tou                    Cold derating off / on                 -12387.0 -12391.0 -12391.0     4.0
                                          Round trip 0.95 / 0.85                 -12286.0 -12504.0 -12391.0   218.0
                                          Escalation +2% / 0% per year           -12070.0 -12391.0 -12391.0   321.0
                                          Discount rate 4% / 8%                  -12083.0 -12659.0 -12391.0   576.0
                                          Horizon 15 / 10 years                  -11304.0 -12391.0 -12391.0  1086.0
                                          Rebate on / off                         -8891.0 -12391.0 -12391.0  3500.0
                                          Capex -25% / +25%                       -8491.0 -16291.0 -12391.0  7800.0
Optimised (perfect foresight) - wholesale Network windows essential / assignment -14406.0 -14360.0 -14360.0    46.0
                                          Round trip 0.95 / 0.85                 -14337.0 -14384.0 -14360.0    48.0
                                          Cold derating off / on                 -14278.0 -14360.0 -14360.0    83.0
                                          Escalation +2% / 0% per year           -14236.0 -14360.0 -14360.0   124.0
                                          Discount rate 4% / 8%                  -14241.0 -14464.0 -14360.0   222.0
                                          Horizon 15 / 10 years                  -13941.0 -14360.0 -14360.0   420.0
                                          VPP fee $0 / $40 per month             -12485.0 -15485.0 -14360.0  3001.0
                                          Rebate on / off                        -10860.0 -14360.0 -14360.0  3500.0
                                          Wear basis cycle life / warranty       -10380.0 -14360.0 -14360.0  3980.0
                                          Capex -25% / +25%                      -10460.0 -18260.0 -14360.0  7800.0

In [423]:
# Figure 14.2 - Tornado: change in NPV from the base case, one input at a time
fig = make_subplots(rows=1, cols=len(torn), horizontal_spacing=0.3,
                    subplot_titles=[f"{m}<br>{names[t]} (base NPV ${d.base.iloc[0]:,.0f})" for (m, t), d in torn.items()])
for c, d in enumerate(torn.values(), start=1):
    fig.add_trace(go.Bar(y=d.index, x=d.npv_a - d.base, orientation="h", name="first value", marker_color="#2ca02c",
                         showlegend=c == 1), 1, c)
    fig.add_trace(go.Bar(y=d.index, x=d.npv_b - d.base, orientation="h", name="second value", marker_color="#d62728",
                         showlegend=c == 1), 1, c)
fig.update_layout(barmode="overlay", height=450, template="plotly_white", legend=dict(orientation="h", y=-0.15))
fig.update_xaxes(title_text="Change in NPV ($)")
fig.show()

## Task 14.4 - Break-even numbers

- **(a) Maximum capex** for the battery to pay back (discounted) within the horizon: the present value of the savings.
- **(b) Maximum VPP fee** at which handing the battery to VPP-SPREE on wholesale still beats the household's alternative, against two alternatives: running the same battery itself (self-consumption) on the cheaper of flat and ToU, or having no battery on the cheaper fixed tariff. Negative means no fee works.
- **(c) Wear cost at which cycling starts**: read from Task 13.1 (the largest grid value with more than one cycle a year).

In [424]:
def max_capex(annual_saving: float, spec: BatterySpec = BATTERY, **kw) -> float:
    """Capex at which NPV = 0 over the horizon (present value of the savings)."""
    return float(cashflows(0.0, annual_saving, spec, **kw)["discounted"].sum()) if annual_saving > 0 else 0.0

alt = master.loc["Self-consumption"].drop("wholesale")["bill"]          # household runs its own battery, flat or ToU
alt_tariff, alt_bill = alt.idxmin(), alt.min()
nob_best = master.loc["No battery"].drop("wholesale")["bill"].min()     # or no battery at all, on the cheaper fixed tariff
before_fee = lambda m: master.loc[(m, "wholesale"), "bill"] - master.loc[(m, "wholesale"), "vpp_fee"]
breakeven = pd.DataFrame({
    m: {"saving ($/yr)": master.loc[(m, t), "saving"],
        "(a) max capex ($)": max_capex(master.loc[(m, t), "saving"]),
        "(b) max VPP fee vs own battery ($/month)": (alt_bill - before_fee(m)) / 12 if m != "Self-consumption" else np.nan,
        "(b) max VPP fee vs no battery ($/month)": (nob_best - before_fee(m)) / 12 if m != "Self-consumption" else np.nan}
    for m, (_, t) in best.items()}).T
first_cycling = (sens[sens.cycles > 1].groupby("tariff").wear.max().reindex(TARIFFS))
print(f"Household alternative for (b): self-consumption on {alt_tariff}, bill ${alt_bill:,.0f}/yr")
print("(c) highest wear cost with > 1 cycle/yr:", {t: (None if pd.isna(w) else round(w, 3)) for t, w in first_cycling.items()})
breakeven.round(0)

Household alternative for (b): self-consumption on tou, bill $92/yr
(c) highest wear cost with > 1 cycle/yr: {'flat': 0.2, 'tou': 0.3, 'wholesale': 0.386}


,saving ($/yr),(a) max capex ($),(b) max VPP fee vs own battery ($/month),(b) max VPP fee vs no battery ($/month)
Optimised (perfect foresight),198.0,1240.0,-5.0,38.0
Rolling horizon (forecasts),1.0,8.0,-44.0,-1.0
Self-consumption,513.0,3209.0,NaN,NaN


## Task 14.5 - Tariff choice vs battery value

Switching tariff costs nothing; a battery costs $15,600. Compare the two levers on the same footing.

In [425]:
nob = master.loc["No battery", "bill"]
lever = pd.DataFrame({
    "no-battery bill": nob,
    "tariff switch saving vs flat (no capex)": nob["flat"] - nob,
    **{f"battery saving: {m}": master.loc[m, "saving"] for m in MODES[1:]},
    **{f"battery net of wear: {m}": master.loc[m, "net_saving"] for m in MODES[1:]},
}).T
lever.style.format("${:,.0f}")

tariff,flat,tou,wholesale
no-battery bill,$822,$606,$649
tariff switch saving vs flat (no capex),$0,$216,$173
battery saving: Self-consumption,$666,$513,$552
battery saving: Optimised (perfect foresight),$0,$0,$198
battery saving: Rolling horizon (forecasts),$0,$1,$-272
battery net of wear: Self-consumption,$-387,$-540,$-501
battery net of wear: Optimised (perfect foresight),$0,$0,$63
battery net of wear: Rolling horizon (forecasts),$0,$-0,$-306


## Task 14.6 - Headline numbers

The numbers the executive summary quotes, in one place, pulled from the tables above (nothing new is calculated except the with-rebate NPV). 2023, Sigenergy 15.6 kWh, capex $15,600 before rebate, discount rate and horizon from Task 14.1.

In [426]:
HEADLINE_CASES = {   # the cheapest status quo, the best do-it-yourself option, and VPP-SPREE's offer as it would
                     # actually run (forecasts) next to its upper bound (perfect foresight)
    "No battery (ToU)":                       ("No battery", "tou"),
    "Own battery, self-consumption (ToU)":    ("Self-consumption", "tou"),
    "VPP-SPREE, forecasts (wholesale)":       ("Rolling horizon (forecasts)", "wholesale"),
    "VPP-SPREE, perfect foresight (wholesale)": ("Optimised (perfect foresight)", "wholesale"),
}
nob_best = master.loc["No battery", "bill"].min()

def headline(case) -> dict:
    r = master.loc[case]
    batt = case[0] != "No battery"
    vpp = r.vpp_fee > 0
    reb = (npv_irr(CAPEX["with rebate"], r.saving, BATTERY)["npv"] if r.saving > 0 else -CAPEX["with rebate"]) if batt else np.nan
    return {
        "Annual bill ($)":                          r.bill,
        "Saving vs cheapest no-battery bill ($/yr)": nob_best - r.bill,
        "Saving vs no battery, same tariff ($/yr)": r.saving if batt else np.nan,
        "VPP fee in that saving ($/yr)":            r.vpp_fee if vpp else np.nan,
        "Saving net of wear ($/yr)":                r.net_saving if batt else np.nan,
        "Self-consumption":                         r.self_consumption,
        "Self-sufficiency":                         r.self_sufficiency,
        "Cycles per year":                          r.cycles if batt else np.nan,
        "NPV, no rebate ($)":                       r.npv,
        "NPV, with rebate ($)":                     reb,
        "Discounted payback (years)":               r.disc_payback,
        "Battery value covers the VPP fee?":        ("yes" if r.saving_before_fee >= r.vpp_fee else "no") if vpp else "-",
    }

head = pd.DataFrame({k: headline(c) for k, c in HEADLINE_CASES.items()})
usd = lambda v: f"-${-v:,.0f}" if v < 0 else f"${v:,.0f}"        # -$315, not $-315
money = [i for i in head.index if "$" in i]
fmt = lambda i, v: ("-" if (isinstance(v, float) and np.isnan(v)) else
                    ("never" if isinstance(v, float) and np.isinf(v) else
                     (usd(v) if i in money else (f"{v:.0%}" if i.startswith("Self-") else
                      (f"{v:.1f}" if isinstance(v, float) else v)))))
head.apply(lambda col: [fmt(i, v) for i, v in col.items()])

,No battery (ToU),"Own battery, self-consumption (ToU)","VPP-SPREE, forecasts (wholesale)","VPP-SPREE, perfect foresight (wholesale)"
Annual bill ($),$606,$92,$920,$450
Saving vs cheapest no-battery bill ($/yr),$0,$513,-$315,$155
"Saving vs no battery, same tariff ($/yr)",-,$513,-$272,$198
VPP fee in that saving ($/yr),-,-,$300,$300
Saving net of wear ($/yr),-,-$540,-$306,$63
Self-consumption,54%,78%,58%,60%
Self-sufficiency,65%,91%,66%,67%
Cycles per year,-,174.9,5.7,22.5
"NPV, no rebate ($)",-,"-$12,391","-$15,600","-$14,360"
"NPV, with rebate ($)",-,"-$8,891","-$12,100","-$10,860"


In [427]:
# Key results in numbers (each traced to the Task it comes from)
wl, w22 = attr.loc["wholesale"], regime.loc[("wholesale", "2022")]
key = pd.Series({
    "Cheapest tariff without a battery (Task 6)":                f"{master.loc['No battery', 'bill'].idxmin()}: ${nob_best:,.0f}/yr",
    "Switching flat -> ToU, no capex (Task 14.5)":               f"${lever.loc['tariff switch saving vs flat (no capex)', 'tou']:,.0f}/yr",
    "Wholesale value lost to forecasting, 2023 (Task 12)":       f"{usd(wl['total'])}/yr: price {usd(wl['price'])}, PV {usd(wl['pv'])}",
    "Same with measured PV, 2022 (Task 13.3)":                   f"{usd(w22['lost: total'])}/yr: price {usd(w22['lost: price'])}, PV {usd(w22['lost: pv'])}",
    "Optimiser saving earned in top 1% of intervals (Task 13.2)": f"{conc.loc['Part 7 optimiser', 'top 1% of intervals ($)'] / conc.loc['Part 7 optimiser', 'annual saving ($)']:.0%}",
    "Mean instead of median price forecast (Task 13.4)":         f"${v2tab.loc['D: Part 11 controller', 'saving before fee']:,.0f} -> ${v2tab.loc['1: mean price forecast', 'saving before fee']:,.0f}/yr before fee",
    "Highest wear cost with cycling, by tariff (Task 13.1)":     ", ".join(f"{t} {'none' if pd.isna(w) else f'${w:.2f}'}" for t, w in first_cycling.items()) + f" (warranty basis ${DEG_COST:.3f}/kWh)",
    "Max capex for 10-year payback, self-consumption (Task 14.4)": f"${breakeven.loc['Self-consumption', '(a) max capex ($)']:,.0f} (vs ${BATTERY.capex_aud:,.0f})",
    "Max VPP fee vs no battery, perfect foresight (Task 14.4)":  f"${breakeven.loc['Optimised (perfect foresight)', '(b) max VPP fee vs no battery ($/month)']:,.0f}/month (vs ${VPP_FEE_PER_MONTH:.0f})",
}, name="value")
key.to_frame()

,value
Cheapest tariff without a battery (Task 6),tou: $606/yr
"Switching flat -> ToU, no capex (Task 14.5)",$216/yr
"Wholesale value lost to forecasting, 2023 (Task 12)","$393/yr: price $393, PV -$1"
"Same with measured PV, 2022 (Task 13.3)","$368/yr: price $369, PV $0"
Optimiser saving earned in top 1% of intervals (Task 13.2),84%
Mean instead of median price forecast (Task 13.4),$28 -> $171/yr before fee
"Highest wear cost with cycling, by tariff (Task 13.1)","flat $0.20, tou $0.30, wholesale $0.39 (warran..."
"Max capex for 10-year payback, self-consumption (Task 14.4)","$3,209 (vs $15,600)"
"Max VPP fee vs no battery, perfect foresight (Task 14.4)",$38/month (vs $25)


> **Finding 14 - Techno-economics.** TODO in my words: NPV / IRR / discounted payback of the best mode, with and without the rebate; the break-even capex and VPP fee and what they mean for VPP-SPREE's offer; tariff switching vs buying a battery; the discount-rate and horizon choices and why.

# Appendix B - Reproducibility

**Code:** TODO: GitHub link. **Data:** the site file (Part 0 `DATA_PATH`) and AEMO NSW1 price files, downloaded once into `data/` by `load_spot_prices` (Part 3).

**Run time.** Restart & Run All takes a few minutes because the slow results load from `data/results/`. Each slow step has a flag; set it to `True` to regenerate its cache from scratch (approximate full re-run times on a laptop):

| Flag | Part | Regenerates | Full re-run |
|---|---|---|---|
| `RERUN_SWEEP` | 8 | battery size sweep | a few minutes |
| `RERUN_ROLLING` | 11-12 | rolling-horizon runs for 2023 (forecast, perfect, PV-only, price-only) | ~15 min |
| `RERUN_SENS` | 13.1 | wear-cost sweep | ~6 min |
| `RERUN_2022` | 13.3 | 2022 rolling-horizon runs | ~20 min |
| `RERUN_V2` | 13.4 | scheduler v2 runs | ~10 min |

Results from Part 13 on are cached under a hash of their inputs (battery, switches, network windows and rates, method names), so a changed assumption writes a new file instead of reusing a stale one. Earlier caches (Parts 8, 11-12) are keyed by name only: re-run them with their flag after changing an assumption they depend on. Some caches were computed in the AI assistant's workspace from the same code (see `AI_LOG.md`).

In [428]:
env = pd.Series({
    "Run date":      datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "Python":        sys.version.split()[0],
    "OS":            platform.platform(),
    "numpy":         np.__version__,
    "pandas":        pd.__version__,
    "scipy":         scipy.__version__,
    "plotly":        plotly.__version__,
    "LP solver":     "scipy.optimize.linprog, method='highs'",
    "Random seed":   RNG_SEED,
    "Rerun flags":   ", ".join(f"{k}={globals()[k]}" for k in
                               ["RERUN_SWEEP", "RERUN_ROLLING", "RERUN_SENS", "RERUN_2022", "RERUN_V2"]),
}, name="value")

cache = pd.DataFrame([(f.name, f.stat().st_size / 1e6,
                       datetime.datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M"))
                      for f in sorted(ROLL_DIR.glob("*.csv"))], columns=["cached result", "MB", "written"])
print(f"{len(cache)} cached result files, {cache.MB.sum():.0f} MB in {ROLL_DIR}")
env.to_frame()

28 cached result files, 137 MB in /Users/bilalp/Documents/SOLA4237/CER_Assignments/Assignment_2/SOLA/data/results


,value
Run date,2026-11-21 14:53
Python,3.11.0
OS,macOS-26.5.2-arm64-arm-64bit
numpy,2.4.1
pandas,2.3.3
scipy,1.17.0
plotly,6.6.0
LP solver,"scipy.optimize.linprog, method='highs'"
Random seed,42
Rerun flags,"RERUN_SWEEP=False, RERUN_ROLLING=False, RERUN_..."
